# Dynamic ORB System
### Opening-range breakout that switches its opening-range window with time-series and random-forest selectors

**Goal**: each session at 09:35 each asset picks ONE opening-range window (5 / 10 / 15 / 30 / 60 min), or stays flat. The choice is made by walk-forward **time-series selectors** (which window has worked best so far) and **random forests**, all refit every January on past data only:
- a stacked `RandomForestRegressor` forecast of each window's value μ̂_W;
- a multiclass `RandomForestClassifier` estimate of the probability that each window, or no trade, is best.

**Pipeline**
- **Cells 1–4**: data, the conservative ORB fill engine, and decision-time features.
- **Cells 5–7**: the dynamic layer, the Stage A mechanics screen, and the forests.
- **Cell 8**: bootstraps 2008–2015 to choose the best variation **D\*** and a static benchmark **S\***, then writes a pre-registration hash.
- **Cell 9**: tests D\* against S\* once, on the 2016+ holdout.
- **Cell 10**: robustness.
- **Cell 11**: every asset vs buy & hold, with one chart per asset.
- **Cell 12**: compares with volatility over **2022-03-07 → 2026-10-15**, using VXX as the reference.
- **Cell 13**: conclusions.

**Data**: every `*_clean_1min.csv` in `/Users/andrew/main/trading/data/etfs/` is auto-discovered. These are 1-minute bars from pitrading until 2022-03-07, then IEX. Every discovered ETF is traded except the sparse ones (Cell 1 prints the list and flags new symbols for review). **VXX is a volatility reference only**: it is never traded, never a feature, and never used in any selection.

## Cell 1 — Libraries & config

In [1]:
# ── standard library ─────────────────────────────────────────────────────────
import gc
import hashlib
import inspect
import itertools
import json
import math
import re
import time
import warnings
from concurrent.futures import ThreadPoolExecutor
from dataclasses import asdict, dataclass, replace
from pathlib import Path
from typing import Dict, List, Optional, Tuple

# ── numerical / data ──────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
from numba import njit
from scipy import stats

# ── machine learning / statistics ─────────────────────────────────────────────
import sklearn
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
import statsmodels.api as sm

# ── visualisation ─────────────────────────────────────────────────────────────
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import display

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", message="All-NaN slice encountered")   # sessions with an empty window
warnings.filterwarnings("ignore", message="Mean of empty slice")
warnings.filterwarnings("ignore", message="invalid value encountered")
warnings.filterwarnings("ignore", message="Degrees of freedom <= 0")
pd.options.mode.chained_assignment = None
pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)

# ── run mode ──────────────────────────────────────────────────────────────────
# QUICK = True → smoke test (reduced grid, K = 2 specs, B = 200 resamples). The
# conclusions cell refuses to run in QUICK mode.
QUICK = False

# ── project paths ─────────────────────────────────────────────────────────────
DATA_DIR  = Path("/Users/andrew/main/trading/data/etfs")
CACHE_DIR = Path("data_cache"); CACHE_DIR.mkdir(exist_ok=True)
PREREG_PATH = CACHE_DIR / "preregistration.json"          # frozen selection: one file per traded universe
                                                          # (Cell 8 adds a universe tag; written once, then asserted)

# ── universe: auto-discovered from DATA_DIR ───────────────────────────────────
# Any file matching *_clean_1min.csv dropped into DATA_DIR is picked up
# automatically — no code change needed when adding new ETFs.
ETF_SYMBOLS: List[str] = sorted(
    p.stem.replace("_clean_1min", "")
    for p in DATA_DIR.glob("*_clean_1min.csv")
)
assert ETF_SYMBOLS, f"No *_clean_1min.csv files found in {DATA_DIR}"

VOL_SYMBOL = "VXX"              # volatility reference ONLY — never traded, never loaded before Cell 12
SPARSE_ETFS = {"CPER"}          # too few bars to trade (≈ 29 bars/session); not loaded
TRADE_SYMBOLS: List[str] = [s for s in ETF_SYMBOLS if s != VOL_SYMBOL and s not in SPARSE_ETFS]
BENCHMARK = "SPY"
EQUITY_ETFS = {"SPY", "QQQ", "IWM", "DIA"}
ASSET_CLASS_KNOWN = {"SPY": "equity", "QQQ": "equity", "IWM": "equity", "DIA": "equity",
                     "TLT": "rates", "IEF": "rates", "SHY": "rates",
                     "GLD": "metals", "SLV": "metals", "USO": "energy"}
ASSET_CLASS = {s: ASSET_CLASS_KNOWN.get(s, "other") for s in TRADE_SYMBOLS}   # new CSVs → "other"
assert BENCHMARK in TRADE_SYMBOLS and VOL_SYMBOL not in TRADE_SYMBOLS
NEW_SYMBOLS = [s for s in TRADE_SYMBOLS if s not in ASSET_CLASS_KNOWN]  # auto-discovered, not yet reviewed

# ── corporate actions ─────────────────────────────────────────────────────────
# The CSV prices are split- AND dividend-adjusted, but on two bases: the pitrading segment (before
# IEX_SEAM) is adjusted to ≈ 2022-03, the IEX segment to the end of the file. Cell 2 rescales each
# pre-seam segment by one factor onto the IEX basis, so every series is a standard back-adjusted
# (total-return) series — without it, dividend payers show a fake 3–14 % drop on 2022-03-07.
# The engine converts each session back to (approximately) traded dollars with a per-date factor
# k_t so the $0.01 tick, slippage and $/share commission apply at the real share price. Splits are
# exact; dividends cannot be undone from this data (no unadjusted source), so the main run leaves
# them in — that OVERSTATES costs in early years (conservative) — and Cell 10 re-runs with an
# estimated dividend factor as a sensitivity (calibrated to the measured seam factor where there
# is one — see price_scale in Cell 3).
SPLITS = {                      # (first session trading on the new basis, new shares per old share)
    "IWM": [("2005-06-09", 2.0)],
    "SLV": [("2008-07-24", 10.0)],
    "USO": [("2020-04-29", 1 / 8)],     # 1-for-8 reverse split (verified: pre-split prices sit on a 0.08 grid)
}
DIV_YIELD_EST = {"SPY": 0.019, "QQQ": 0.008, "IWM": 0.013, "DIA": 0.021, "TLT": 0.030,
                 "IEF": 0.022, "SHY": 0.017, "GLD": 0.0, "SLV": 0.0, "USO": 0.0}   # sensitivity only

# ── session-timing constants ──────────────────────────────────────────────────
RTH_OPEN_MIN = 9 * 60 + 30      # column j of a session panel = bar opening at 09:30 + j minutes
N_BARS = 390                    # 09:30 … 15:59 inclusive

# ── data-source seam ──────────────────────────────────────────────────────────
IEX_SEAM = pd.Timestamp("2022-03-07")   # pitrading → IEX; vol drops to ~2%
SEAM_TOL = 0.01                 # repair when the last pre-seam close → first post-seam open gap is < −1 %
                                # (a dividend-basis break can only push the IEX segment down)

# ── execution & cost model (shared by every backtest) ────────────────────────
TICK = 0.01
COMMISSION = 0.0035             # $/share/side (IBKR Pro tiered entry level — verify current)
SLIPPAGE_TICKS = 1              # adverse ticks on every stop/market fill
MAX_COST_R = 0.15               # tradability rule: median round-trip cost ≤ 0.15 R over the prior 250 sessions

# ── portfolio research sizing (selection + bootstrap) ─────────────────────────
RISK = 0.0025                   # 0.25 % of equity risked per trade
POS_CAP = 1.0                   # ≤ 1× equity notional per trade
GROSS_CAP = 4.0                 # ≤ 4× equity open notional at any minute (causal, first-come allocation)

# ── per-asset account test (Cell 11) ─────────────────────────────────────────
ACCOUNT_EQUITY = 100_000.0      # above the $25k pattern-day-trader minimum throughout
ACCOUNT_RISK = dict(risk=0.01, pos_cap=4.0, gross_lev=4.0, max_trades=1,
                    daily_loss=0.0, dd_throttle=0.0, eq_filter=False)

# ── OR window candidates (minutes) — the timeframes the dynamic system switches between ──
OR_WINDOWS: List[int] = [5, 10, 15, 30, 60]
DECISION_MIN = min(OR_WINDOWS)  # the window is chosen at 09:35, before any candidate OR except 5-min completes

# ── research protocol ────────────────────────────────────────────────────────
RF_FIRST_YEAR = 2006            # first walk-forward RF year (its scores calibrate the 2008 thresholds)
WF_FIRST_YEAR = 2008            # first evaluated year
IS_END = pd.Timestamp("2015-12-31")     # every choice (spec, selector, threshold) uses 2008–2015 only
OOS_START = pd.Timestamp("2016-01-01")  # 2016+ = holdout, scored once against the pre-registration
VOL_START, VOL_END = pd.Timestamp("2022-03-07"), pd.Timestamp("2026-10-15")   # VXX comparison window
N_BOOT = 200 if QUICK else 2000
TOP_K = 2 if QUICK else 4

# ── random state ─────────────────────────────────────────────────────────────
SEED = 42
N_JOBS = 6                      # forests use threads; this machine has 6 cores

print(f"numpy {np.__version__}  |  pandas {pd.__version__}  |  scikit-learn {sklearn.__version__}")
print(f"DATA_DIR  : {DATA_DIR}")
print(f"Found     : {', '.join(ETF_SYMBOLS)}  ({len(ETF_SYMBOLS)} files)")
print(f"Traded    : {', '.join(TRADE_SYMBOLS)}  ({len(TRADE_SYMBOLS)} symbols)")
print(f"Excluded  : {', '.join(sorted(SPARSE_ETFS))} (too sparse) · {VOL_SYMBOL} (volatility reference only, Cell 12)")
if NEW_SYMBOLS:
    print(f"\n⚠ New CSVs traded as class 'other': {', '.join(NEW_SYMBOLS)} — before reading results, check them for "
          f"splits (add to SPLITS), set DIV_YIELD_EST and ASSET_CLASS_KNOWN, and inspect their row of the seam "
          f"table in Cell 2. They change the universe, so Cell 8 writes a new pre-registration.")
if QUICK:
    print("\n" + "!" * 78 + "\n  QUICK MODE — smoke test only: reduced grid, K = 2, B = 200. Do not read results.\n" + "!" * 78)

numpy 2.3.5  |  pandas 2.3.3  |  scikit-learn 1.7.2
DATA_DIR  : /Users/andrew/main/trading/data/etfs
Found     : CPER, DIA, GLD, IEF, IWM, QQQ, SHY, SLV, SPY, TLT, USO, VXX  (12 files)
Traded    : DIA, GLD, IEF, IWM, QQQ, SHY, SLV, SPY, TLT, USO  (10 symbols)
Excluded  : CPER (too sparse) · VXX (volatility reference only, Cell 12)


## Cell 2 — ETF data import

In [2]:
def load_etf(symbol: str) -> pd.DataFrame:
    """
    Load one ETF from a *_clean_1min.csv file and return a clean DataFrame
    indexed by a DatetimeIndex, RTH bars only (09:30–15:59).

    Known quirks handled here
    ─────────────────────────
    • pitrading → IEX seam on 2022-03-07: volume drops to ~2%; relative volume
      must therefore be computed *within* source regime (never across the seam).
      The two segments are also dividend-adjusted to different dates — see repair_seam.
    • ~10 % of sessions lack the exact 09:30 bar; bars are labelled by bar-open.
    • Prices are split- and dividend-adjusted (see SPLITS in Cell 1).
    • Duplicate rows (rare, pitrading artefact): keep first.
    """
    path = DATA_DIR / f"{symbol}_clean_1min.csv"
    df = pd.read_csv(path, engine="pyarrow")
    df["datetime"] = pd.to_datetime(df["datetime"], format="%Y-%m-%d %H:%M:%S")
    df = df.set_index("datetime")

    # drop duplicates (keep first occurrence)
    df = df[~df.index.duplicated(keep="first")]
    df.sort_index(inplace=True)

    # RTH filter: 09:30 ≤ bar < 16:00
    t = df.index.time
    rth_mask = (t >= pd.Timestamp("09:30").time()) & (t < pd.Timestamp("16:00").time())
    df = df[rth_mask].copy()

    # ensure canonical column types
    for col in ("Open", "High", "Low", "Close"):
        df[col] = df[col].astype("float32")
    df["Volume"] = df["Volume"].astype("int64")
    if "source" not in df.columns:
        df["source"] = "unknown"
    df["source"] = df["source"].astype("category")
    # 'source' is already a category; add explicit seam flag for downstream use
    df["post_seam"] = df.index >= IEX_SEAM

    return df


def repair_seam(df: pd.DataFrame) -> Tuple[pd.DataFrame, dict]:
    """
    Put the pitrading segment on the IEX segment's dividend basis.

    pitrading prices are dividend-adjusted to ≈ the seam, IEX prices to the end of the file, so a
    dividend payer's series drops by roughly its yield × years since the seam at the first IEX bar
    (TLT ≈ −14 %, SPY ≈ −6 %) — a loss nobody suffered. Multiplying every pre-seam price by
    (first post-seam open / last pre-seam close) gives one back-adjusted series. Only gaps below
    −SEAM_TOL are repaired: a basis break can only push the IEX segment down, and non-payers
    (GLD, SLV, USO) show their genuine overnight moves (< 0.6 %). The repair also removes the real
    overnight move of that one night — one overnight return out of ≈ 5,000.
    """
    post = df["post_seam"].to_numpy()
    info = {"last pre-seam close": np.nan, "first post-seam open": np.nan, "gap %": np.nan,
            "factor": 1.0, "repaired": False}
    if post.any() and (~post).any():
        cp, oq = float(df["Close"].to_numpy()[~post][-1]), float(df["Open"].to_numpy()[post][0])
        info.update({"last pre-seam close": cp, "first post-seam open": oq, "gap %": 100 * (oq / cp - 1)})
        if oq / cp - 1 < -SEAM_TOL:
            for col in ("Open", "High", "Low", "Close"):
                v = df[col].to_numpy(np.float64)
                v[~post] *= oq / cp
                df[col] = v.astype("float32")
            info.update({"factor": oq / cp, "repaired": True})
    return df, info


# ── load the traded ETFs (VXX is NOT loaded here — Cell 12 reads it on its own) ──
t0 = time.time()
raw_1m: Dict[str, pd.DataFrame] = {}
SEAM: Dict[str, dict] = {}

for sym in TRADE_SYMBOLS:
    raw_1m[sym], SEAM[sym] = repair_seam(load_etf(sym))

elapsed = time.time() - t0
assert VOL_SYMBOL not in raw_1m
SEAM_TABLE = pd.DataFrame(SEAM).T.rename_axis("Symbol")
SEAM_FACTOR = SEAM_TABLE["factor"].astype(float).to_dict()

# Data signature (file sizes + modification times + the loader's code and seam factors) — part of
# every cache key, so editing or adding a CSV, or changing the repair, invalidates cached
# simulations and model predictions automatically.
try:
    _loader_src = inspect.getsource(load_etf) + inspect.getsource(repair_seam)
except (OSError, TypeError):
    _loader_src = "loader-v2"
DATA_SIG = hashlib.sha1(repr([(s, (DATA_DIR / f"{s}_clean_1min.csv").stat().st_size,
                               (DATA_DIR / f"{s}_clean_1min.csv").stat().st_mtime_ns, round(SEAM_FACTOR[s], 10))
                              for s in TRADE_SYMBOLS] + [SEAM_TOL, _loader_src]).encode()).hexdigest()[:12]

# ── summary table ─────────────────────────────────────────────────────────────
rows = []
for sym, df in raw_1m.items():
    pre  = df[~df["post_seam"]]
    post = df[df["post_seam"]]
    rows.append({
        "Symbol"          : sym,
        "Class"           : ASSET_CLASS[sym],
        "First date"      : df.index.normalize().min().date(),
        "Last date"       : df.index.normalize().max().date(),
        "Total bars"      : len(df),
        "Pre-seam bars"   : len(pre),
        "Post-seam bars"  : len(post),
        "Pre vol/bar (k)" : round(pre["Volume"].mean() / 1_000, 1) if len(pre) else 0,
        "Post vol/bar (k)": round(post["Volume"].mean() / 1_000, 1) if len(post) else 0,
        "Splits undone"   : ", ".join(f"{d} ×{r:g}" for d, r in SPLITS.get(sym, [])) or "—",
    })

summary = pd.DataFrame(rows).set_index("Symbol")
print(f"Loaded {len(raw_1m)} traded ETFs in {elapsed:.1f}s  ·  data signature {DATA_SIG}\n")
print(summary.to_string())
print(f"\nIEX seam: {IEX_SEAM.date()}  |  post-seam vol is ~2 % of pitrading — "
      "do NOT compute relative volume across the seam.")
print(f"Dividend-basis repair at the seam (pre-seam prices × factor when the gap < −{SEAM_TOL:.0%}):")
print(SEAM_TABLE.assign(**{"gap %": SEAM_TABLE["gap %"].astype(float).round(2),
                           "factor": SEAM_TABLE["factor"].astype(float).round(5)}).to_string())
_gap = SEAM_TABLE["gap %"].astype(float)
if (_gap.abs() > 30).any():
    print(f"⚠ seam gap beyond ±30 % ({', '.join(_gap.index[_gap.abs() > 30])}) looks like an unrecorded split, "
          "not dividends — add it to SPLITS and re-check.")
if (_gap > 100 * SEAM_TOL).any():
    print(f"⚠ seam gap above +{SEAM_TOL:.0%} ({', '.join(_gap.index[_gap > 100 * SEAM_TOL])}): a dividend-basis break "
          "can only push the IEX segment down, so this is a real move or a data problem — left unrepaired; check it.")
print(f"{VOL_SYMBOL}: not loaded here — volatility reference only (Cell 12), never traded or used as a feature.")

Loaded 10 traded ETFs in 8.2s  ·  data signature 721256828fed

         Class  First date   Last date  Total bars  Pre-seam bars  Post-seam bars  Pre vol/bar (k)  Post vol/bar (k)      Splits undone
Symbol                                                                                                                                 
DIA     equity  2002-12-30  2026-09-21     2067170        1807905          259265             18.3               0.3                  —
GLD     metals  2004-11-18  2026-09-21     1924936        1616316          308620             21.4               0.6                  —
IEF      rates  2002-12-30  2026-09-21     1560831        1313213          247618              5.7               1.0                  —
IWM     equity  2002-12-30  2026-09-21     2217669        1792731          424938             86.5               1.9      2005-06-09 ×2
QQQ     equity  2002-12-30  2026-09-21     2245015        1817215          427800            154.5               1.7     

## Cell 3 — ORB execution engine

In [3]:
# ════════════════════════════════════════════════════════════════════════════
#  ORB execution engine — the Static ORB System's conservative fill model
#  (stop orders fill at max(level, bar open); bar-close entries fill at the close;
#  the stop wins same-bar ties; targets need a 1-tick trade-through; gaps fill at
#  the open; 1 tick slippage; $0.0035/share/side), with three corrections:
#   1. Session eligibility is EX-ANTE. The Static engine dropped sessions using the
#      whole day's bar coverage and required a bar after the OR — information not
#      known at the open, and the dropped sessions were mostly losers. Now a session
#      is tradable iff (a) its first bar prints by 09:32, (b) ≥ 4 of the 09:30–09:34
#      bars exist, (c) the trailing 20-session median full-day coverage (through
#      yesterday) is ≥ 80 %. A thin day after that simply produces fewer/no fills.
#   2. Prices are converted back to traded dollars (splits undone) before the tick,
#      slippage and $/share commission are applied.
#   3. The time exit fills at the first print at or after the exit minute (the last
#      earlier print only if none exists — flagged as a stale exit).
#  Only the traded universe gets panels — VXX never enters the engine.
# ════════════════════════════════════════════════════════════════════════════
LOOKBACK = 14
MIN_OR_COVERAGE = 0.80                         # share of OR minutes that must have a bar (known at OR end)
MIN_SESSION_COVERAGE = 0.80                    # trailing median full-day coverage (known at 09:30)
COVERAGE_LOOKBACK = 20
MAX_OPEN_DELAY = 2                             # first bar must open by 09:32
MIN_FIRST5 = 4                                 # ≥ 4 of the five 09:30–09:34 bars
MIN_PRICE = 5.0
MIN_STOP_TICKS = 3
TOL = TICK / 4                                 # float comparison tolerance
ENGINE_VERSION = "dyn-3"


def _first_true(mask: np.ndarray) -> np.ndarray:
    """Index of the first True per row, -1 if none."""
    i = mask.argmax(axis=1)
    return np.where(mask[np.arange(len(mask)), i], i, -1)


def _next_true(mask: np.ndarray) -> np.ndarray:
    """nxt[r, j] = first column ≥ j where mask is True (-1 if none)."""
    big = mask.shape[1] + 1
    idx = np.where(mask, np.arange(mask.shape[1])[None, :], big)
    nxt = np.minimum.accumulate(idx[:, ::-1], axis=1)[:, ::-1]
    return np.where(nxt >= big, -1, nxt)


def price_scale(sym: str, dates: pd.DatetimeIndex, mode: str = "split") -> np.ndarray:
    """k_t = traded price / CSV (adjusted) price.  'split': exact split factors;
    'split+div': also undo the dividend adjustment (sensitivity). For a symbol repaired at the seam
    (Cell 2 factor f < 1), 1/f is the measured total adjustment from the seam to the end of the file:
    it is spread over the IEX segment at a constant yield, and before the seam the estimated yield
    (DIV_YIELD_EST) compounds back from the seam on top of 1/f. Other symbols use DIV_YIELD_EST
    from the end of the file."""
    k = np.ones(len(dates))
    for d, ratio in SPLITS.get(sym, []):
        k[dates < pd.Timestamp(d)] *= ratio
    if mode == "split+div":
        f, y = float(SEAM_FACTOR.get(sym, 1.0)), DIV_YIELD_EST.get(sym, 0.0)
        t_end = (dates.max() - dates).days.to_numpy() / 365.25
        span = (dates.max() - IEX_SEAM).days / 365.25
        if f < 1.0 and span > 0:
            t_pre = (IEX_SEAM - dates).days.to_numpy() / 365.25
            k *= np.where(t_pre > 0, np.exp(y * t_pre) / f, np.exp(-np.log(f) / span * t_end))
        else:
            k *= np.exp(y * t_end)
    return k


# ── 1. Session panels: rows = sessions, columns = minutes after 09:30 (NaN = missing bar) ──
def build_panel(df: pd.DataFrame) -> dict:
    minute = df.index.hour * 60 + df.index.minute - RTH_OPEN_MIN
    df = df.loc[(minute >= 0) & (minute < N_BARS)]
    px = df[["Open", "High", "Low", "Close"]]
    df = df.loc[(px > 0).all(axis=1) & (df["High"] >= df["Low"]) & np.isfinite(px).all(axis=1)]
    col = (df.index.hour * 60 + df.index.minute - RTH_OPEN_MIN).to_numpy()
    codes, dates = pd.factorize(df.index.normalize(), sort=True)

    def mat(values: np.ndarray) -> np.ndarray:
        m = np.full((len(dates), N_BARS), np.nan, dtype=np.float32)
        m[codes, col] = values
        return m

    O, H, L, C, V = (mat(df[k].to_numpy(np.float64)) for k in ("Open", "High", "Low", "Close", "Volume"))
    source = df["source"].astype(str).groupby(codes).first().to_numpy()
    return panel_from_arrays(pd.DatetimeIndex(dates, name="date"), O, H, L, C, V, source)


def panel_from_arrays(dates: pd.DatetimeIndex, O, H, L, C, V, source) -> dict:
    """Everything derived from the bar matrices (also used by the look-ahead perturbation tests)."""
    present = ~np.isnan(C)
    has_bar = present.any(axis=1)
    first = np.where(has_bar, present.argmax(axis=1), -1)
    last = np.where(has_bar, N_BARS - 1 - present[:, ::-1].argmax(axis=1), -1)
    rows = np.arange(len(dates))
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", RuntimeWarning)          # all-NaN rows (no bars) → NaN
        daily = pd.DataFrame({
            "open": np.where(has_bar, O[rows, np.maximum(first, 0)], np.nan),
            "high": np.nanmax(H, axis=1), "low": np.nanmin(L, axis=1),
            "close": np.where(has_bar, C[rows, np.maximum(last, 0)], np.nan),
            "volume": np.nansum(V, axis=1),
        }, index=dates).astype(float).round(6)
    daily["source"] = source
    # ex-ante eligibility (see header); full-day coverage enters only through YESTERDAY's trailing median
    cov_full = pd.Series(present.sum(axis=1) / N_BARS, index=dates)
    cov_trail = cov_full.rolling(COVERAGE_LOOKBACK, min_periods=10).median().shift(1)
    elig = (has_bar & (first <= MAX_OPEN_DELAY) & (present[:, :DECISION_MIN].sum(axis=1) >= MIN_FIRST5)
            & (cov_trail >= MIN_SESSION_COVERAGE).to_numpy())
    # the Static engine's rule — kept ONLY to report what the look-ahead did (never used to trade)
    lookahead_ok = has_bar & (first <= MAX_OPEN_DELAY) & (present.sum(axis=1) / np.maximum(last + 1, 1) >= 0.80)
    return dict(dates=dates, O=O, H=H, L=L, C=C, V=V, first=first, last=last, daily=daily,
                elig=elig, lookahead_ok=lookahead_ok, cov_full=cov_full.to_numpy())


# ── 2. Ex-ante daily features (all shifted: known before 09:30) — in CSV (adjusted) price units ──
def daily_features(d: pd.DataFrame) -> pd.DataFrame:
    pc = d["close"].shift(1)
    tr = pd.concat([d["high"] - d["low"], (d["high"] - pc).abs(), (d["low"] - pc).abs()], axis=1).max(axis=1)
    atr14 = tr.ewm(alpha=1 / 14, adjust=False, min_periods=14).mean()
    atr50 = tr.ewm(alpha=1 / 50, adjust=False, min_periods=50).mean()
    rng = d["high"] - d["low"]
    f = pd.DataFrame(index=d.index)
    f["atr"] = atr14.shift(1)
    f["avg_vol"] = d["volume"].rolling(LOOKBACK, min_periods=LOOKBACK).mean().shift(1)
    f["gap_atr"] = (d["open"] - pc) / f["atr"]
    f["trend"] = np.sign(d["close"] - d["close"].rolling(20, min_periods=20).mean()).shift(1)
    f["atr_ratio"] = (atr14 / atr50).shift(1)
    f["nr7"] = (rng <= rng.rolling(7, min_periods=7).min()).shift(1, fill_value=False).astype(bool)
    f["open"] = d["open"]
    return f


# ── 3. Opening-range cache per window (known at 09:30 + n) ──
def or_cache(p: dict, n: int) -> dict:
    O, H, L, C, V, d = p["O"], p["H"], p["L"], p["C"], p["V"], p["daily"]
    rows = np.arange(len(d))
    present = ~np.isnan(C[:, :n])
    last_in = n - 1 - present[:, ::-1].argmax(axis=1)
    ok = p["elig"] & (present.mean(axis=1) >= MIN_OR_COVERAGE)
    o_open = O[rows, np.maximum(p["first"], 0)].astype(float)
    o_close = C[rows, last_in].astype(float)
    vol = pd.Series(np.nansum(V[:, :n], axis=1), index=d.index)
    # Relative volume vs the previous 14 valid sessions; the OR must include the 09:30 print;
    # computed within each data-source regime (never across the IEX seam).
    rv_ok = ok & ~np.isnan(V[:, 0])
    base = (vol.where(rv_ok).groupby(d["source"].to_numpy())
            .transform(lambda s: s.dropna().rolling(LOOKBACK).mean().shift(1).reindex(s.index)))
    return dict(ok=ok,
                hi=np.nanmax(H[:, :n], axis=1).astype(float), lo=np.nanmin(L[:, :n], axis=1).astype(float),
                open=o_open, close=o_close,
                dir=np.where(ok, np.sign(np.round(o_close - o_open, 6)), 0).astype(int),
                rv=(vol / base).where(rv_ok & (base > 0)).to_numpy())


t0 = time.perf_counter()
panels = {sym: build_panel(raw_1m[sym]) for sym in TRADE_SYMBOLS}
del raw_1m; gc.collect()                       # 1-min frames are no longer needed (≈ 1 GB freed)
FEAT = {s: daily_features(p["daily"]) for s, p in panels.items()}
ORC = {s: {n: or_cache(p, n) for n in OR_WINDOWS} for s, p in panels.items()}
SCALE = {m: {s: price_scale(s, p["dates"], m) for s, p in panels.items()} for m in ("split", "split+div")}
CAL = pd.DatetimeIndex(sorted(set().union(*[set(p["dates"]) for p in panels.values()])))
YEARS = CAL.year.to_numpy()
R_SPY = panels[BENCHMARK]["daily"]["close"].reindex(CAL).ffill().pct_change().fillna(0.0).to_numpy()
assert "VXX" not in panels and all("vxx" not in s.lower() for s in panels)


# ── 4. Strategy definition ──
@dataclass(frozen=True)
class Logic:
    n: int                 # opening-range minutes
    dir_mode: str          # "candle" | "both" | "candle+gap" | "candle+trend"
    buffer: float          # entry beyond the OR, × ATR
    confirm: str           # "touch" (stop order) | "close" (bar-close confirmation)
    cutoff: int            # last entry column (minutes after 09:30)
    stop: tuple            # ("atr", k) | ("or_opp", 0) | ("or_mid", 0)
    be_R: float            # move stop to breakeven after +x R (0 = off)
    trail_atr: float       # trailing stop, × ATR from best price (0 = off)
    target_R: float        # profit target in R (0 = none)
    exit_col: int          # time exit column (389 = 15:59 bar → 16:00)

    def valid(self) -> bool:
        return self.cutoff > self.n and self.exit_col > self.n + 5 and not (self.be_R and self.trail_atr)


# ── 5. Vectorised simulator: one Logic × one asset, all sessions at once ──
#     Prices are mapped to "signed" space (shorts negated) so one code path serves both directions.
#     fill="next_open": bar-close confirmations fill at the NEXT bar's open (sensitivity).
def simulate(sym: str, lg: Logic, slip_ticks: float = SLIPPAGE_TICKS, scale: str = "split",
             fill: str = "close") -> pd.DataFrame:
    p, f, oc = panels[sym], FEAT[sym], ORC[sym][lg.n]
    kfac = SCALE[scale][sym]
    atr_all = f["atr"].to_numpy() * kfac                     # ATR in traded dollars
    base_ok = oc["ok"] & np.isfinite(atr_all) & (atr_all > 0) & (f["open"].to_numpy() * kfac > MIN_PRICE)
    cd = oc["dir"]
    if lg.dir_mode == "candle":
        dsel = cd
    elif lg.dir_mode == "candle+gap":
        dsel = np.where(np.sign(np.nan_to_num(f["gap_atr"].to_numpy())) == cd, cd, 0)
    elif lg.dir_mode == "candle+trend":
        dsel = np.where(np.nan_to_num(f["trend"].to_numpy()) == cd, cd, 0)
    else:
        dsel = np.where(base_ok, 2, 0)                      # "both": evaluate both sides
    rows = np.flatnonzero(base_ok & (dsel != 0))
    if rows.size == 0:
        return pd.DataFrame()

    n, slip = lg.n, slip_ticks * TICK
    kr = kfac[rows].astype(np.float32)[:, None]
    O, H, L, C = (p[k][rows, n:lg.exit_col + 1] * kr for k in ("O", "H", "L", "C"))
    j = np.arange(O.shape[1])[None, :]
    r_ = np.arange(len(rows))
    xj = lg.exit_col - n                                     # slice column of the time exit
    entry_lim = min(lg.cutoff, lg.exit_col - 1) - n          # ex-ante: no dependence on later bars
    a = atr_all[rows]
    hi, lo = oc["hi"][rows] * kfac[rows], oc["lo"][rows] * kfac[rows]
    trig_long = np.ceil(np.round((hi + lg.buffer * a) / TICK, 6)) * TICK
    trig_short = np.floor(np.round((lo - lg.buffer * a) / TICK, 6)) * TICK
    nxt = _next_true(~np.isnan(O)) if (lg.confirm == "close" and fill == "next_open") else None

    def scan(sign: float):
        ts = sign * (trig_long if sign > 0 else trig_short)
        Hs, Cs, Os = (H, C, O) if sign > 0 else (-L, -C, -O)
        win = j <= entry_lim
        if lg.confirm == "touch":
            e = _first_true((Hs >= ts[:, None] - TOL) & win)
            fill_ = np.maximum(ts, np.round(Os[r_, np.maximum(e, 0)] / TICK) * TICK) + slip
        elif fill == "close":
            e = _first_true((Cs > ts[:, None] + TOL) & win)
            fill_ = np.round(Cs[r_, np.maximum(e, 0)] / TICK) * TICK + slip
        else:                                                # confirm on the close, fill at next open
            e0 = _first_true((Cs > ts[:, None] + TOL) & win)
            e = np.where((e0 >= 0) & (e0 + 1 < O.shape[1]), nxt[r_, np.minimum(e0 + 1, O.shape[1] - 1)], -1)
            e = np.where(e > xj - 1, -1, e)
            fill_ = np.round(Os[r_, np.maximum(e, 0)] / TICK) * TICK + slip
        return e, fill_

    if lg.dir_mode == "both":
        eL, fL = scan(1.0)
        eS, fS = scan(-1.0)
        eL_, eS_ = np.where(eL < 0, 10**6, eL), np.where(eS < 0, 10**6, eS)
        d = np.where(eL_ < eS_, 1.0, np.where(eS_ < eL_, -1.0, 0.0))     # same-bar double break → skip
        e = np.where(d > 0, eL, np.where(d < 0, eS, -1))
        fill_s = np.where(d > 0, fL, fS)
    else:
        d = dsel[rows].astype(float)
        e = np.full(len(rows), -1)
        fill_s = np.full(len(rows), np.nan)
        for sgn in (1.0, -1.0):
            mk = d == sgn
            if mk.any():
                ee, ff = scan(sgn)
                e, fill_s = np.where(mk, ee, e), np.where(mk, ff, fill_s)
    trig_ok = (e >= 0) & (d != 0)

    dd = d[:, None]
    Hs, Ls = np.where(dd > 0, H, -L), np.where(dd > 0, L, -H)
    Os, Cs = np.where(dd > 0, O, -O), np.where(dd > 0, C, -C)

    kind, k = lg.stop
    if kind == "atr":
        stop_s = fill_s - k * a
    elif kind == "or_opp":
        stop_s = np.where(d > 0, lo, -hi) - TICK
    else:
        stop_s = np.where(d > 0, (hi + lo) / 2, -(hi + lo) / 2)
    stop_s = np.floor(np.round(stop_s / TICK, 6)) * TICK      # rounded away from entry
    risk = fill_s - stop_s
    trig_ok &= np.isfinite(risk) & (risk >= MIN_STOP_TICKS * TICK)

    ec = np.maximum(e, 0)[:, None]
    first_live = ec if (lg.confirm == "touch" or fill == "next_open") else ec + 1   # a fill bar can stop out
    fav = np.where(j >= first_live, Hs, -np.inf)
    run = np.maximum.accumulate(np.nan_to_num(fav, nan=-np.inf), axis=1)
    prev_best = np.maximum(np.concatenate([np.full((len(rows), 1), -np.inf), run[:, :-1]], axis=1),
                           fill_s[:, None])
    level = np.broadcast_to(stop_s[:, None], Hs.shape).copy()
    if lg.be_R > 0:
        level = np.where(prev_best - fill_s[:, None] >= lg.be_R * risk[:, None],
                         np.maximum(level, fill_s[:, None]), level)
    if lg.trail_atr > 0:
        level = np.maximum(level, prev_best - lg.trail_atr * a[:, None])

    js = _first_true((j >= first_live) & (Ls <= level + TOL))
    lv = level[r_, np.maximum(js, 0)]
    o_s = np.round(Os[r_, np.maximum(js, 0)] / TICK) * TICK
    stop_fill = np.where(js == e, lv, np.minimum(lv, o_s)) - slip
    if lg.target_R > 0:
        tgt = np.ceil(np.round((fill_s + lg.target_R * risk) / TICK, 6)) * TICK
        jt = _first_true((j >= ec + 1) & (Hs >= tgt[:, None] + TICK - TOL))
        tgt_fill = np.maximum(tgt, np.round(Os[r_, np.maximum(jt, 0)] / TICK) * TICK)
    else:
        jt, tgt_fill = np.full(len(rows), -1), np.full(len(rows), np.nan)
    big = 10**6
    js_, jt_ = np.where(js >= 0, js, big), np.where(jt >= 0, jt, big)
    by_stop = (js_ < big) & (js_ <= jt_)
    by_tgt = (jt_ < big) & (jt_ < js_)

    # time exit: close of the exit-minute bar; else the first print after it (next bar's open);
    # else (no later print in the session) the last earlier print — flagged "stale"
    pr = rows
    xc = lg.exit_col
    k1 = kfac[pr]
    c_at = p["C"][pr, xc] * k1
    after = p["O"][pr, xc + 1:] if xc + 1 < N_BARS else np.full((len(pr), 0), np.nan, np.float32)
    na = _first_true(~np.isnan(after)) if after.shape[1] else np.full(len(pr), -1)
    o_after = np.where(na >= 0, after[r_, np.maximum(na, 0)] * k1 if after.shape[1] else np.nan, np.nan)
    lastp = np.where(~np.isnan(C), j, -1).max(axis=1)          # last print inside the slice
    c_last = C[r_, np.maximum(lastp, 0)]
    time_px = np.where(np.isfinite(c_at), c_at, np.where(na >= 0, o_after, c_last))
    stale = ~np.isfinite(c_at) & (na < 0)
    time_col = np.where(np.isfinite(c_at), xc, np.where(na >= 0, xc + 1 + na, lastp + n))
    eod_fill = np.round(np.where(dd[:, 0] > 0, time_px, -time_px) / TICK) * TICK - slip
    exit_s = np.where(by_stop, stop_fill, np.where(by_tgt, tgt_fill, eod_fill))
    exit_col = np.where(by_stop, js + n, np.where(by_tgt, jt + n, time_col))
    stale &= ~(by_stop | by_tgt)

    pnl_ps = exit_s - fill_s - 2 * COMMISSION                   # $ per share after costs
    trig_ok &= np.isfinite(pnl_ps)
    fr = f.iloc[rows]
    out = pd.DataFrame({
        "asset": sym, "date": p["dates"][rows], "dir": d.astype(int), "triggered": trig_ok,
        "entry_col": e + n, "exit_col": exit_col, "entry": np.abs(fill_s), "risk_ps": risk,
        "pnl_ps": pnl_ps, "R": np.where(trig_ok, pnl_ps / np.where(risk > 0, risk, np.nan), np.nan),
        "cost_R": (2 * slip + 2 * COMMISSION) / np.where(risk > 0, risk, np.nan),
        "rv": oc["rv"][rows], "or_atr": (hi - lo) / a, "gap_atr": fr["gap_atr"].to_numpy(),
        "atr_ratio": fr["atr_ratio"].to_numpy(), "nr7": fr["nr7"].to_numpy(),
        "atr": a, "avg_vol": fr["avg_vol"].to_numpy(), "stale_exit": stale,
        "reason": np.where(by_stop, "stop", np.where(by_tgt, "target", "time")),
    })
    out["cal_idx"] = CAL.get_indexer(out["date"])
    return out


# ── 6. Asset × session grid (the dynamic layer works on tensors [asset, session, window]) ──
A, T, NW = len(TRADE_SYMBOLS), len(CAL), len(OR_WINDOWS)
WIDX = {w: i for i, w in enumerate(OR_WINDOWS)}
ROW = {s: CAL.get_indexer(p["dates"]) for s, p in panels.items()}      # panel row → CAL column

def to_grid(per_asset: Dict[str, np.ndarray], fill=np.nan, dtype=np.float32) -> np.ndarray:
    """Stack per-asset arrays aligned to each panel's dates into an [A, T] grid."""
    g = np.full((A, T), fill, dtype=dtype)
    for ai, s in enumerate(TRADE_SYMBOLS):
        g[ai, ROW[s]] = per_asset[s]
    return g

ATR_G = to_grid({s: FEAT[s]["atr"].to_numpy() for s in TRADE_SYMBOLS})          # CSV units
VALID = to_grid({s: panels[s]["elig"] & np.isfinite(FEAT[s]["atr"].to_numpy())
                 & (FEAT[s]["atr"].to_numpy() > 0)
                 & (FEAT[s]["open"].to_numpy() * SCALE["split"][s] > MIN_PRICE)
                 for s in TRADE_SYMBOLS}, fill=False, dtype=bool)                # known at 09:35
GAP_G = to_grid({s: FEAT[s]["gap_atr"].to_numpy() for s in TRADE_SYMBOLS})
ORATR_G = np.stack([to_grid({s: (ORC[s][w]["hi"] - ORC[s][w]["lo"]) / FEAT[s]["atr"].to_numpy()
                             for s in TRADE_SYMBOLS}) for w in OR_WINDOWS], axis=2)       # [A, T, W]
IS_EQUITY = np.array([s in EQUITY_ETFS for s in TRADE_SYMBOLS])
POST_SEAM = CAL >= IEX_SEAM


@dataclass(frozen=True)
class Spec:
    """ORB mechanics shared by every window; the dynamic layer only chooses `n`."""
    dir_mode: str = "candle"
    buffer: float = 0.0
    confirm: str = "touch"
    cutoff: int = 120              # last entry 11:30
    stop: tuple = ("or_opp", 0)
    be_R: float = 0.0
    exit_col: int = 389

    def logic(self, n: int) -> Logic:
        return Logic(n, self.dir_mode, self.buffer, self.confirm, self.cutoff, self.stop,
                     self.be_R, 0.0, 0.0, self.exit_col)

    def key(self) -> str:
        return hashlib.md5(repr(asdict(self)).encode()).hexdigest()[:12]


def _src_hash(*objs) -> str:
    """Hash of the source of every function (njit kernels via .py_func) and, for a class, its fields
    with defaults plus the source of its methods — per object, so one unreadable source cannot
    collapse the whole key to a constant (Jupyter cannot return a class's source)."""
    parts = []
    for o in objs:
        o = getattr(o, "py_func", o)
        if inspect.isclass(o):
            flds = getattr(o, "__dataclass_fields__", {})
            parts.append(o.__qualname__ + repr([(n, str(f.type), "-" if type(f.default).__name__ == "_MISSING_TYPE"
                                                 else repr(f.default)) for n, f in flds.items()]))
            members = [m for m in vars(o).values() if inspect.isfunction(m)]
        else:
            members = [o]
        for m in members:
            try:
                parts.append(inspect.getsource(m))
            except (OSError, TypeError):
                parts.append(m.__qualname__)
    return hashlib.sha1(("".join(parts) + ENGINE_VERSION).encode()).hexdigest()[:10]

OUTCOME_KEYS = ("R", "lev", "dir", "ecol", "xcol", "cost")


def _simulate_grid(spec: Spec, scale: str, fill: str) -> dict:
    R = np.full((A, T, NW), np.nan, dtype=np.float32)
    lev, cost = R.copy(), R.copy()
    dirn = np.zeros((A, T, NW), dtype=np.int8)
    ecol = np.full((A, T, NW), -1, dtype=np.int16)
    xcol = ecol.copy()
    jobs = [(wi, w, ai, s) for wi, w in enumerate(OR_WINDOWS) for ai, s in enumerate(TRADE_SYMBOLS)]
    with ThreadPoolExecutor(N_JOBS) as ex:
        sims = list(ex.map(lambda jb: simulate(jb[3], spec.logic(jb[1]), scale=scale, fill=fill), jobs))
    for (wi, w, ai, s), t in zip(jobs, sims):
        if t.empty:
            continue
        t = t.loc[t["triggered"]]
        ci = t["cal_idx"].to_numpy()
        R[ai, ci, wi] = t["R"].to_numpy()
        lev[ai, ci, wi] = (t["entry"] / t["risk_ps"]).to_numpy()
        cost[ai, ci, wi] = t["cost_R"].to_numpy()
        dirn[ai, ci, wi] = t["dir"].to_numpy()
        ecol[ai, ci, wi] = t["entry_col"].to_numpy()
        xcol[ai, ci, wi] = t["exit_col"].to_numpy()
    return dict(R=R, lev=lev, dir=dirn, ecol=ecol, xcol=xcol, cost=cost)


ENGINE_HASH = _src_hash(build_panel, panel_from_arrays, daily_features, or_cache, simulate, price_scale,
                        _first_true, _next_true, to_grid, _simulate_grid, Logic, Spec)
# every constant the simulated outcomes depend on, besides the code (ENGINE_HASH) and the data (DATA_SIG)
MODEL_SIG = repr((TICK, COMMISSION, SLIPPAGE_TICKS, MIN_PRICE, MIN_STOP_TICKS, tuple(OR_WINDOWS), N_BARS,
                  RTH_OPEN_MIN, sorted((k, tuple(v)) for k, v in SPLITS.items()), sorted(DIV_YIELD_EST.items()),
                  LOOKBACK, MIN_OR_COVERAGE, MIN_SESSION_COVERAGE, COVERAGE_LOOKBACK, MAX_OPEN_DELAY, MIN_FIRST5,
                  TOL, str(IEX_SEAM.date()),
                  # the price levels themselves: SEAM_FACTOR is measured from the data in Cell 2, outside the code hash
                  hashlib.sha1(b"".join(SCALE[m][s].tobytes() for m in sorted(SCALE) for s in TRADE_SYMBOLS)).hexdigest()))


def window_outcomes(spec: Spec, scale: str = "split", fill: str = "close", use_cache: bool = True) -> dict:
    """Simulate every window for every traded asset → tensors [A, T, W]:
         R    : trade result in R after costs (NaN = no trade that window/day)
         lev  : entry price / risk per share (position-sizing input)
         dir  : +1 long / -1 short;  ecol / xcol : entry / exit minute;  cost : round-trip cost in R
       'candle+gap' is derived exactly from 'candle' (same trade iff the gap agrees with the candle)."""
    if spec.dir_mode == "candle+gap":
        base = window_outcomes(replace(spec, dir_mode="candle"), scale, fill, use_cache)
        agree = (np.sign(np.nan_to_num(GAP_G))[..., None] == base["dir"]) & (base["dir"] != 0)
        return {k: np.where(agree, v, np.nan if v.dtype.kind == "f" else (0 if k == "dir" else -1)).astype(v.dtype)
                for k, v in base.items()}
    key = hashlib.sha1(repr((asdict(spec), scale, fill, MODEL_SIG, tuple(TRADE_SYMBOLS), ENGINE_HASH,
                             DATA_SIG)).encode()).hexdigest()[:16]
    path = CACHE_DIR / f"wo3_{key}.npz"
    if use_cache and path.exists():
        z = np.load(path)
        out = {k: z[k] for k in OUTCOME_KEYS}
        assert out["R"].shape == (A, T, NW) and str(z["cal_end"]) == str(CAL[-1].date())
        return out
    out = _simulate_grid(spec, scale, fill)
    np.savez_compressed(path, cal_end=str(CAL[-1].date()), **out)
    return out


def trade_book(spec: Spec, pick: np.ndarray, scale: str = "split") -> pd.DataFrame:
    """Full trade records for a [A, T] pick matrix (window index, -1 = no trade)."""
    parts = []
    for wi, w in enumerate(OR_WINDOWS):
        for ai, s in enumerate(TRADE_SYMBOLS):
            if not (pick[ai] == wi).any():
                continue
            t = simulate(s, spec.logic(w), scale=scale)
            if t.empty:
                continue
            t = t.loc[t["triggered"] & (pick[ai, t["cal_idx"].to_numpy()] == wi)]
            parts.append(t.assign(window=w))
    return (pd.concat(parts, ignore_index=True).sort_values(["cal_idx", "entry_col", "asset"], ignore_index=True)
            if parts else pd.DataFrame())


# ── 7. Portfolio sizing with a CAUSAL gross-notional cap ──
@njit(cache=True)
def _causal_alloc(t_idx, ecol, xcol, notional, gross_cap):
    """Trades sorted by (session, entry minute, asset). Each trade gets the fraction of its
    notional that fits under gross_cap given trades still open when it enters (first come,
    first served) — nothing about later trades is used."""
    n = len(t_idx)
    alloc = np.ones(n)
    i = 0
    while i < n:
        j = i
        while j < n and t_idx[j] == t_idx[i]:
            j += 1
        for a in range(i, j):
            used = 0.0
            for b in range(i, a):
                if xcol[b] >= ecol[a]:
                    used += notional[b] * alloc[b]
            room = gross_cap - used
            alloc[a] = min(1.0, max(room, 0.0) / notional[a]) if notional[a] > 0 else 0.0
        i = j
    return alloc


def grid_returns(o: dict, weight: np.ndarray, risk: float = RISK, pos_cap: float = POS_CAP,
                 gross_cap: float = GROSS_CAP, R: np.ndarray | None = None, stats_out: dict | None = None) -> np.ndarray:
    """Daily portfolio returns. weight [A, T, W] ≥ 0 = share of `risk` given to each window's trade
    (a pick is one-hot with weight 1; the all-window ensemble is 1/5 each). Per trade: notional ≤ pos_cap ×
    equity; open notional ≤ gross_cap × equity, allocated causally in entry order."""
    R = o["R"] if R is None else R
    sel = (weight > 0) & np.isfinite(R)
    ai, ti, wi = np.nonzero(sel)
    if ai.size == 0:
        return np.zeros(T)
    wt = weight[ai, ti, wi].astype(np.float64)
    lv = o["lev"][ai, ti, wi].astype(np.float64)
    rr = R[ai, ti, wi].astype(np.float64)
    ec, xc = o["ecol"][ai, ti, wi].astype(np.int64), o["xcol"][ai, ti, wi].astype(np.int64)
    w = np.minimum(1.0, pos_cap / (risk * wt * lv))
    notional = risk * wt * w * lv
    order = np.lexsort((ai, ec, ti))
    alloc = np.empty_like(notional)
    alloc[order] = _causal_alloc(ti[order], ec[order], xc[order], notional[order], gross_cap)
    if stats_out is not None:
        stats_out["bind_rate"] = float((alloc < 1 - 1e-12).mean())
        stats_out["n_trades"] = int(ai.size)
    return np.bincount(ti, weights=risk * wt * w * alloc * rr, minlength=T)


def run_risk(t: pd.DataFrame, rk: dict, equity0: float = ACCOUNT_EQUITY) -> np.ndarray:
    """Sequential, integer-share simulation with risk controls (keys: risk, gross_lev, max_trades,
    daily_loss, dd_throttle, eq_filter; optional pos_cap = max notional per trade × equity)."""
    t = t.sort_values(["cal_idx", "entry_col", "asset"])
    ci, ent, rps, pps = (t[k].to_numpy() for k in ("cal_idx", "entry", "risk_ps", "pnl_ps"))
    ecol, xcol = t["entry_col"].to_numpy(), t["exit_col"].to_numpy()
    pos_cap = rk.get("pos_cap", math.inf)
    day_pnl = np.zeros(len(CAL))
    equity = peak = equity0
    starts = np.flatnonzero(np.r_[True, ci[1:] != ci[:-1]]) if len(ci) else np.array([], int)
    ends = np.r_[starts[1:], len(ci)] if len(ci) else np.array([], int)
    last_day, eq_window = 0, []
    for s, e_ in zip(starts, ends):
        day = ci[s]
        eq_window = (eq_window + [equity] * (day - last_day))[-50:]   # equity carried over idle sessions
        last_day = day
        if equity <= 0:
            break
        if rk["eq_filter"] and len(eq_window) >= 50 and equity < np.mean(eq_window) * (1 - 1e-9):
            continue
        mult = 0.5 if rk["dd_throttle"] and equity < peak * (1 - rk["dd_throttle"]) else 1.0
        sod, taken, opened = equity, 0, []                         # opened: (exit_col, notional, pnl)
        for i in range(s, e_):
            if taken >= rk["max_trades"]:
                break
            if rk["daily_loss"] and sum(pn for xc, _, pn in opened if xc < ecol[i]) <= -rk["daily_loss"] * sod:
                break
            in_use = sum(nv for xc, nv, _ in opened if xc >= ecol[i])
            q = math.floor(min(rk["risk"] * mult * sod / rps[i],
                               pos_cap * sod / ent[i],
                               max(rk["gross_lev"] * sod - in_use, 0.0) / ent[i]))
            if q <= 0:
                continue
            opened.append((xcol[i], q * ent[i], q * pps[i]))
            taken += 1
        day_pnl[day] = sum(pn for _, _, pn in opened)
        equity = sod + day_pnl[day]
        peak = max(peak, equity)
    eq = equity0 + np.cumsum(day_pnl)
    prev = np.r_[equity0, eq[:-1]]
    return np.where(prev > 0, eq / prev - 1, 0.0)


# ── 8. Metrics ──
def summarize(r: np.ndarray, mask: np.ndarray) -> tuple:
    """(Sharpe, CAGR, max drawdown) of daily returns r[mask]; fast path for the search."""
    x = r[mask]
    sd = x.std()
    g = np.r_[1.0, np.cumprod(1 + x)]
    sharpe = x.mean() / sd * np.sqrt(252) if sd > 0 else np.nan
    cagr = g[-1] ** (252 / len(x)) - 1 if g[-1] > 0 else -1.0
    return sharpe, cagr, (g / np.maximum.accumulate(g) - 1).min()


def perf_stats(r: pd.Series, bench: pd.Series, trades: pd.DataFrame | None = None) -> dict:
    """Full performance report for a daily return series (every session, zeros included)."""
    g = pd.concat([pd.Series([1.0], index=[r.index[0] - pd.Timedelta(days=1)]), (1 + r).cumprod()])
    dd = g / g.cummax() - 1
    trough = dd.idxmin()
    rec = g.loc[trough:][g.loc[trough:] >= g.loc[:trough].max()]
    monthly = (1 + r).resample("ME").prod() - 1
    yearly = (1 + r).resample("YE").prod() - 1
    beta = np.cov(r, bench)[0, 1] / bench.var() if bench.var() > 0 else np.nan
    down = np.sqrt(np.mean(np.minimum(r, 0) ** 2))
    cagr = g.iloc[-1] ** (252 / len(r)) - 1 if g.iloc[-1] > 0 else -1.0
    out = {
        "Total return %": 100 * (g.iloc[-1] - 1), "CAGR %": 100 * cagr,
        "Volatility %": 100 * r.std() * np.sqrt(252),
        "Sharpe": r.mean() / r.std() * np.sqrt(252) if r.std() > 0 else np.nan,
        "Sortino": r.mean() / down * np.sqrt(252) if down > 0 else np.nan,
        "Max drawdown %": 100 * dd.min(), "Calmar": cagr / abs(dd.min()) if dd.min() < 0 else np.nan,
        "Recovery (sessions)": len(r.loc[trough:rec.index[0]]) - 1 if len(rec) else np.nan,
        "Worst day %": 100 * r.min(), "Worst month %": 100 * monthly.min(),
        "Best year %": 100 * yearly.max(), "Worst year %": 100 * yearly.min(),
        "Up months %": 100 * (monthly > 0).mean(),
        "Corr. to SPY": r.corr(bench), "Beta to SPY": beta,
        "Alpha % (ann.)": 100 * (r.mean() - beta * bench.mean()) * 252,
    }
    if trades is not None:
        out["Trades"] = len(trades)
        out["Hit ratio %"] = 100 * (trades["R"] > 0).mean() if len(trades) else np.nan
    return out


# ── 9. Diagnostics: eligibility by data regime, and what the old look-ahead filter did ──
STATIC_WINNER = Spec("candle+gap", 0.02, "close", 120, ("or_mid", 0), 2.0, 270)   # Static ORB System winner
rows = []
for s in TRADE_SYMBOLS:
    p = panels[s]
    post = p["dates"] >= IEX_SEAM
    rows.append({"asset": s, "eligible % pre-IEX": 100 * p["elig"][~post].mean(),
                 "eligible % IEX era": 100 * p["elig"][post].mean() if post.any() else np.nan,
                 "old filter % IEX era": 100 * p["lookahead_ok"][post].mean() if post.any() else np.nan})
ELIG_TABLE = pd.DataFrame(rows).set_index("asset")
ELIG_TABLE["IEX-era status"] = np.where(ELIG_TABLE["eligible % IEX era"] < 50,
                                        "not testable after 2022-03-07", "ok")
NOT_TESTABLE_IEX = set(ELIG_TABLE.index[ELIG_TABLE["IEX-era status"] != "ok"])

look = []
for s in sorted(EQUITY_ETFS & set(TRADE_SYMBOLS)):
    t = simulate(s, STATIC_WINNER.logic(15))
    if t.empty:
        continue
    t = t.loc[t["triggered"] & (t["gap_atr"].abs() >= 0.3) & (t["or_atr"] <= 0.5)]   # Static winner's filters
    old = panels[s]["lookahead_ok"][panels[s]["dates"].get_indexer(t["date"])]
    for label, m in (("kept by old filter", old), ("dropped by old filter", ~old)):
        look.append({"asset": s, "group": label, "trades": int(m.sum()), "mean R": t["R"][m].mean(),
                     "sum R": t["R"][m].sum()})
LOOKAHEAD = pd.DataFrame(look)

print(f"Engine ready: {A} traded assets, {T:,} sessions ({CAL[0].date()} → {CAL[-1].date()}), "
      f"windows {OR_WINDOWS}, built in {time.perf_counter() - t0:.0f}s  ·  engine {ENGINE_HASH}")
print("\nEx-ante session eligibility by data regime:")
print(ELIG_TABLE.round(1).to_string())
print("\nStatic ORB winner (15-min, gap ≥ 0.3 ATR, OR ≤ 0.5 ATR, equity ETFs): trades on sessions the OLD look-ahead filter dropped")
print(LOOKAHEAD.pivot_table(index="asset", columns="group", values=["trades", "mean R"]).round(3).to_string())
dropped = LOOKAHEAD[LOOKAHEAD["group"] == "dropped by old filter"]
print(f"→ the old filter silently removed {int(dropped['trades'].sum())} trades worth {dropped['sum R'].sum():+.1f} R "
      f"(mean {dropped['sum R'].sum() / max(dropped['trades'].sum(), 1):+.3f} R/trade); they are now included.")

Engine ready: 10 traded assets, 5,969 sessions (2002-12-30 → 2026-09-21), windows [5, 10, 15, 30, 60], built in 6s  ·  engine c88c427d49

Ex-ante session eligibility by data regime:
       eligible % pre-IEX  eligible % IEX era  old filter % IEX era                 IEX-era status
asset                                                                                             
DIA                  90.9                12.5                  14.8  not testable after 2022-03-07
GLD                  87.5                36.8                  40.2  not testable after 2022-03-07
IEF                  37.4                 0.9                   4.6  not testable after 2022-03-07
IWM                  84.3                86.5                  92.6                             ok
QQQ                  88.2                89.0                  94.6                             ok
SHY                  43.6                 0.0                   0.0  not testable after 2022-03-07
SLV                  82.9 

## Cell 4 — Decision-time features

In [4]:
# ════════════════════════════════════════════════════════════════════════════
#  Decision-time feature set — everything is known at 09:35 (end of the shortest
#  window, before any longer opening range completes). Sources: the asset's prior
#  sessions, SPY (market context) and the first five 1-minute bars of today.
#  Built as a pure function of the panel arrays, so the look-ahead test below can
#  rebuild it from deliberately corrupted future data. VXX is never used here.
# ════════════════════════════════════════════════════════════════════════════
def intraday_stats(p: dict) -> pd.DataFrame:
    """Per-session intraday statistics from 1-min closes (same-day values; shifted by the caller)."""
    C = p["C"].astype(np.float64)
    with np.errstate(invalid="ignore", divide="ignore"):
        lr = np.diff(np.log(C), axis=1)                    # NaN across missing minutes
    ok = np.isfinite(lr)
    n = ok.sum(axis=1)
    rv = np.sqrt(np.nansum(lr ** 2, axis=1))
    a, b = lr[:, :-1], lr[:, 1:]
    m = np.isfinite(a) & np.isfinite(b)
    a, b = np.where(m, a, 0.0), np.where(m, b, 0.0)
    k = m.sum(axis=1)
    ma, mb = a.sum(1) / np.maximum(k, 1), b.sum(1) / np.maximum(k, 1)
    cov = ((a - ma[:, None]) * (b - mb[:, None]) * m).sum(1)
    va = (((a - ma[:, None]) ** 2) * m).sum(1)
    vb = (((b - mb[:, None]) ** 2) * m).sum(1)
    with np.errstate(invalid="ignore", divide="ignore"):
        ac1 = cov / np.sqrt(va * vb)
    d = p["daily"]
    eff = (d["close"] - d["open"]).abs() / (d["high"] - d["low"]).replace(0, np.nan)
    return pd.DataFrame({"id_rv": np.where(n >= 100, rv, np.nan),
                         "ac1": np.where(k >= 100, ac1, np.nan),
                         "eff": eff.to_numpy()}, index=d.index)


def asset_features(p: dict) -> pd.DataFrame:
    """09:35 feature frame of one asset — a pure function of its panel arrays."""
    d = p["daily"]
    f = daily_features(d)
    oc5 = or_cache(p, DECISION_MIN)
    ist = intraday_stats(p)
    r1 = np.log(d["close"]).diff()
    x = pd.DataFrame(index=d.index)
    # — volatility state (all through yesterday's close) —
    x["atr_pct"] = f["atr"] / d["close"].shift(1)
    x["atr_ratio"] = f["atr_ratio"]
    x["rv5"] = r1.rolling(5, min_periods=5).std().shift(1)
    x["rv20"] = r1.rolling(20, min_periods=20).std().shift(1)
    x["rv_ratio"] = x["rv5"] / x["rv20"]
    x["id_rv1"] = ist["id_rv"].shift(1)
    x["id_rv_ratio"] = ist["id_rv"].shift(1) / ist["id_rv"].rolling(20, min_periods=10).mean().shift(1)
    x["range1_atr"] = (d["high"] - d["low"]).shift(1) / f["atr"]
    x["nr7"] = f["nr7"].astype(float)
    # — trendiness / mean-reversion character of recent sessions —
    x["eff1"] = ist["eff"].shift(1)
    x["eff20"] = ist["eff"].rolling(20, min_periods=10).mean().shift(1)
    x["ac1_1"] = ist["ac1"].shift(1)
    x["ac1_20"] = ist["ac1"].rolling(20, min_periods=10).mean().shift(1)
    # — price location —
    x["ret1_atr"] = (d["close"] - d["close"].shift(1)).shift(1) / f["atr"]
    x["ret5_atr"] = (d["close"] - d["close"].shift(5)).shift(1) / f["atr"]
    x["sma20_atr"] = (d["close"] - d["close"].rolling(20, min_periods=20).mean()).shift(1) / f["atr"]
    x["sma200_atr"] = (d["close"] - d["close"].rolling(200, min_periods=150).mean()).shift(1) / f["atr"]
    # — today, known at 09:30 / 09:35 (bars 09:30–09:34 only) —
    #   the day's open is only known by 09:35 if the first print falls inside the first five minutes
    opened = (p["first"] >= 0) & (p["first"] < DECISION_MIN)
    x["gap_atr"] = f["gap_atr"].where(opened)
    ok5 = oc5["ok"]
    x["or5_atr"] = np.where(ok5, (oc5["hi"] - oc5["lo"]) / f["atr"].to_numpy(), np.nan)
    x["c5_atr"] = np.where(ok5, (oc5["close"] - oc5["open"]) / f["atr"].to_numpy(), np.nan)
    x["relvol5"] = np.where(ok5, oc5["rv"], np.nan)          # relative volume within the data-source regime
    x["c5_gap"] = np.sign(x["c5_atr"]) * np.sign(x["gap_atr"])
    x["dow"] = d.index.dayofweek.astype(float)
    return x


def market_features(spy: pd.DataFrame) -> pd.DataFrame:
    """Market context from SPY's own 09:35 frame (same value for every asset on a date)."""
    return pd.DataFrame({"spy_rv20": spy["rv20"], "spy_rv_ratio": spy["rv_ratio"],
                         "spy_gap_atr": spy["gap_atr"], "spy_c5_atr": spy["c5_atr"]})


CLASSES = sorted(set(ASSET_CLASS.values()))
CLASS_COLS = [f"is_{c}" for c in CLASSES]


def stack_features(frames: Dict[str, pd.DataFrame], cal: pd.DatetimeIndex) -> Tuple[np.ndarray, list]:
    """[A, T, F] tensor: asset features + SPY market context + asset-class one-hots."""
    mkt = market_features(frames[BENCHMARK]).reindex(cal)
    base_cols = list(frames[BENCHMARK].columns) + list(mkt.columns)
    X = np.full((len(TRADE_SYMBOLS), len(cal), len(base_cols) + len(CLASS_COLS)), np.nan, dtype=np.float32)
    for ai, s in enumerate(TRADE_SYMBOLS):
        xs = frames[s].reindex(cal).join(mkt)
        X[ai, :, :len(base_cols)] = xs[base_cols].to_numpy(np.float32)
        X[ai, :, len(base_cols):] = np.array([ASSET_CLASS[s] == c for c in CLASSES], dtype=np.float32)
    return X, base_cols + CLASS_COLS


t0 = time.perf_counter()
BASE_FEATS = {s: asset_features(panels[s]) for s in TRADE_SYMBOLS}
XB, XB_COLS = stack_features(BASE_FEATS, CAL)
assert not any("vxx" in c.lower() for c in XB_COLS)


# ── Look-ahead test: corrupt every bar after 09:34 on day t, and every later session, for ALL assets;
#    rebuild the panels and features from scratch; day t's features must be bit-for-bit unchanged. ──
def corrupt_future(p: dict, d_row: int, rng: np.random.Generator) -> dict:
    O, H, L, C, V = (p[k].copy() for k in ("O", "H", "L", "C", "V"))
    fut = np.zeros(O.shape, bool)
    fut[d_row, DECISION_MIN:] = True
    fut[d_row + 1:] = True
    k = np.exp(rng.normal(0, 0.03, O.shape)).astype(np.float32)          # one factor per bar keeps H ≥ O, C ≥ L
    drop = rng.random(O.shape) < 0.3                                       # and knock out 30 % of the bars
    for M in (O, H, L, C):
        M[fut] *= k[fut]
        M[fut & drop] = np.nan
    V[fut] *= np.exp(rng.normal(0, 1, O.shape)).astype(np.float32)[fut]
    V[fut & drop] = np.nan
    return panel_from_arrays(p["dates"], O, H, L, C, V, p["daily"]["source"].to_numpy())


def lookahead_test(n_dates: int, seed: int = SEED) -> int:
    rng = np.random.default_rng(seed)
    cand = np.flatnonzero(VALID.any(axis=0) & (YEARS >= RF_FIRST_YEAR))
    checked = 0
    for t in rng.choice(cand, n_dates, replace=False):
        day = CAL[t]
        frames = {}
        for s in TRADE_SYMBOLS:
            rows = np.flatnonzero(panels[s]["dates"] >= day)
            if not rows.size:
                frames[s] = BASE_FEATS[s]
                continue
            frames[s] = asset_features(corrupt_future(panels[s], rows[0], rng))
        Xc, _ = stack_features(frames, CAL)
        np.testing.assert_array_equal(Xc[:, t], XB[:, t], err_msg=f"look-ahead in features on {day.date()}")
        checked += len(TRADE_SYMBOLS)
    return checked


n_checked = lookahead_test(5 if QUICK else 25)
cov = pd.Series(np.isfinite(XB[VALID]).mean(axis=0), index=XB_COLS)
print(f"Base feature tensor XB: {XB.shape} (assets × sessions × features) built in {time.perf_counter() - t0:.0f}s")
print(f"Look-ahead test passed: {n_checked} asset-days rebuilt from corrupted future bars give identical 09:35 features")
print("Coverage on eligible asset-sessions (share non-NaN):")
print(cov.round(3).to_frame("coverage").T.to_string())

Base feature tensor XB: (10, 5969, 31) (assets × sessions × features) built in 55s
Look-ahead test passed: 250 asset-days rebuilt from corrupted future bars give identical 09:35 features
Coverage on eligible asset-sessions (share non-NaN):
          atr_pct  atr_ratio  rv5   rv20  rv_ratio  id_rv1  id_rv_ratio  range1_atr  nr7  eff1  eff20  ac1_1  ac1_20  ret1_atr  ret5_atr  sma20_atr  sma200_atr  gap_atr  or5_atr  c5_atr  relvol5  c5_gap  dow  spy_rv20  spy_rv_ratio  spy_gap_atr  spy_c5_atr  is_energy  is_equity  is_metals  is_rates
coverage      1.0      0.996  1.0  0.999     0.999     1.0          1.0         1.0  1.0   1.0    1.0  0.999     1.0       1.0       1.0      0.999       0.989      1.0      1.0     1.0    0.966     1.0  1.0       1.0           1.0        0.981       0.936        1.0        1.0        1.0       1.0


## Cell 5 — Dynamic layer: time-series window selectors

In [5]:
# ════════════════════════════════════════════════════════════════════════════
#  Dynamic layer — at 09:35 each asset picks ONE opening-range window (5/10/15/30/60)
#  from the TIME SERIES of how each window has performed so far.
#    u[a, t, W] = value of choosing W on session t, in units of RISK
#                 (= position weight × R; 0 if W does not trigger or is filtered; NaN = no session)
#    pick[a, t] = window index into OR_WINDOWS, −1 = flat
#  Every score at t uses outcomes of sessions < t only (two causality tests at the bottom).
# ════════════════════════════════════════════════════════════════════════════
IS_MASK = (CAL >= pd.Timestamp(f"{WF_FIRST_YEAR}-01-01")) & (CAL <= IS_END)   # 2008–2015: all selection
OOS_MASK = CAL >= OOS_START                                                   # 2016+: holdout (scored from Cell 9 on)
YEAR_LIST = np.unique(YEARS)
YI = np.searchsorted(YEAR_LIST, YEARS)                  # year index of every session
NY = len(YEAR_LIST)
DEFAULT_W = WIDX[15]            # fallback window when there is no history yet
EW_MIN = 20                     # observations before an EWMA is defined
HL_U = (40, 120, 250)           # half-lives (sessions) of the window-performance EWMAs
HL_TS = (120, 250)              # half-lives the time-series selectors may use
W0_MIN = 250                    # sessions of history before the expanding best window replaces the default
SWITCH_GAP = 20                 # incumbent selector: ≥ 20 valid sessions between switches
CLASSES = sorted(set(ASSET_CLASS.values()))
CLASS_OF = np.array([CLASSES.index(ASSET_CLASS[s]) for s in TRADE_SYMBOLS])
NC = len(CLASSES)


@dataclass(frozen=True)
class Config:
    """One spec + its OR-time filters (applied identically to every window)."""
    spec: Spec
    gap_min: float = 0.0           # |gap| / ATR ≥ gap_min   (known 09:30)
    or_atr_max: float = np.inf     # OR width / ATR ≤ max    (known when that window's range completes)

    def key(self) -> str:
        return hashlib.md5(repr((asdict(self.spec), self.gap_min, self.or_atr_max)).encode()).hexdigest()[:12]

    def label(self) -> str:
        s = self.spec
        stop = s.stop[0] if s.stop[0] != "atr" else f"atr{s.stop[1]}"
        return (f"{s.dir_mode}/{s.confirm}/buf{s.buffer}/{stop}/be{s.be_R}/x{s.exit_col}"
                f"/gap{self.gap_min}/or{self.or_atr_max}")


# ── causal EWMA on the session axis ──
def ewm_shift(x: np.ndarray, hl: float, min_periods: int = EW_MIN) -> np.ndarray:
    """x [N, T, ...] → EWMA along axis 1 over finite observations only; the value at t is the
    EWMA through the last observation strictly before t (NaN until `min_periods` observations)."""
    n, t = x.shape[:2]
    flat = np.moveaxis(x, 1, 0).reshape(t, -1).astype(np.float64)
    e = pd.DataFrame(flat).ewm(halflife=hl, ignore_na=True, min_periods=min_periods).mean().to_numpy()
    e[~np.isfinite(flat)] = np.nan
    e = pd.DataFrame(e).ffill().shift(1).to_numpy()
    return np.moveaxis(e.reshape((t, n) + x.shape[2:]), 0, 1)


def class_mean(x: np.ndarray) -> np.ndarray:
    """[A, T, ...] → [NC, T, ...]: per-date mean over the class members that have a session."""
    return np.stack([np.nanmean(x[CLASS_OF == c], axis=0) for c in range(NC)])


def to_assets(xc: np.ndarray, fallback: np.ndarray) -> np.ndarray:
    """Class-level array → each member asset; the asset's own value where the class has none."""
    y = xc[CLASS_OF]
    return np.where(np.isfinite(y), y, fallback)


def tradable_mask(cost: np.ndarray) -> np.ndarray:
    """[A, T] bool: median round-trip cost (R) over the asset's prior 250 sessions ≤ MAX_COST_R."""
    day = np.nanmedian(cost, axis=2)
    roll = pd.DataFrame(day.T).rolling(250, min_periods=50).median().shift(1).to_numpy().T
    return VALID & (roll <= MAX_COST_R)


def universes(tradable: np.ndarray) -> dict:
    return {"tradable": tradable, "equity": tradable & IS_EQUITY[:, None], "all": VALID}


# ── per asset-year sums → expanding / trailing best windows and the hindsight oracle ──
def year_sums(u: np.ndarray):
    S, SS = np.zeros((A, NY, NW)), np.zeros((A, NY, NW))
    N = np.zeros((A, NY))
    for yi in range(NY):
        x = u[:, YI == yi]
        S[:, yi], SS[:, yi] = np.nansum(x, 1), np.nansum(x ** 2, 1)
        N[:, yi] = np.isfinite(x[..., 0]).sum(1)
    return S, SS, N


def _best_mean(S: np.ndarray, N: np.ndarray, min_n: int) -> np.ndarray:
    b = (S / np.maximum(N, 1)[..., None]).argmax(-1)          # ties → the shorter window
    return np.where(N >= min_n, b, DEFAULT_W)


def w0_tables(S, SS, N) -> dict:
    """[A, NY] window tables using years < y only: expanding best mean u (asset / class scope)
    and the best trailing-3-year Sharpe of u (≥ 250 sessions, else the expanding winner)."""
    cS, cN = np.cumsum(S, 1) - S, np.cumsum(N, 1) - N
    w0a = _best_mean(cS, cN, W0_MIN)
    w0c = np.empty_like(w0a)
    for c in range(NC):
        m = CLASS_OF == c
        w0c[m] = _best_mean(cS[m].sum(0), cN[m].sum(0), W0_MIN)[None]
    w3 = w0a.copy()
    for yi in range(NY):
        lo = max(0, yi - 3)
        s, ss, n = S[:, lo:yi].sum(1), SS[:, lo:yi].sum(1), N[:, lo:yi].sum(1)[:, None]
        mu = s / np.maximum(n, 1)
        sd = np.sqrt(np.maximum(ss / np.maximum(n, 1) - mu ** 2, 0))
        sh = np.where(sd > 0, mu / np.where(sd > 0, sd, 1), -np.inf)
        w3[:, yi] = np.where(n[:, 0] >= 250, sh.argmax(-1), w0a[:, yi])
    return {"asset": w0a, "class": w0c, "trail3": w3}


# ── the time-series state of one config ──
def ts_state(u: np.ndarray, hit: np.ndarray, trig: np.ndarray) -> dict:
    """Causal window-performance time series (asset and asset-class scope) + RF window features."""
    uc = class_mean(u)
    E, Q = {}, {}
    for h in HL_U:
        E[("asset", h)] = ewm_shift(u, h)
        E[("class", h)] = to_assets(ewm_shift(uc, h), E[("asset", h)])
    for h in HL_TS:
        qa = ewm_shift(u[..., :, None] * u[..., None, :], h)                   # E[u_W u_W'] [A, T, W, W]
        Q[("asset", h)] = qa
        qc = ewm_shift(uc[..., :, None] * uc[..., None, :], h)[CLASS_OF]
        cls_ok = np.isfinite(ewm_shift(uc, h)[CLASS_OF])[..., None]           # class EWMA defined → use class Q
        Q[("class", h)] = np.where(cls_ok, qc, qa)
    e_hit, e_trig = ewm_shift(hit, 120), ewm_shift(trig, 120)
    S, SS, N = year_sums(u)
    w0 = w0_tables(S, SS, N)
    own = [E[("asset", 40)], E[("asset", 250)], E[("class", 40)], E[("class", 250)], e_hit, e_trig]
    rel = [x - np.nanmean(x, axis=2, keepdims=True) for x in own[:4]]
    wfeat = np.stack(own + rel + [np.broadcast_to(np.array(OR_WINDOWS, float), u.shape)], axis=-1).astype(np.float32)
    return dict(E=E, Q=Q, S=S, SS=SS, N=N, W0={k: v[:, YI] for k, v in w0.items()}, wfeat=wfeat)

WFEAT_COLS = ["ew_u_a40", "ew_u_a250", "ew_u_c40", "ew_u_c250", "ew_hit_a120", "ew_trig_a120",
              "rel_u_a40", "rel_u_a250", "rel_u_c40", "rel_u_c250", "window_min"]


def build_config(cfg: Config, ts: bool = True, scale: str = "split") -> dict:
    """Outcome tensors of one config (+ its time-series state unless ts=False, for Stage A). scale "split" is
    the registered price basis; "split+div" also undoes the dividend adjustment (Cell 10's re-selection)."""
    wo = window_outcomes(cfg.spec, scale)
    keep = (np.abs(np.nan_to_num(GAP_G))[..., None] >= cfg.gap_min) & ~(ORATR_G > cfg.or_atr_max)
    V3 = VALID[..., None]
    R = np.where(keep & V3, wo["R"], np.nan).astype(np.float32)               # NaN = no trade in that window
    traded = np.isfinite(R)
    wfrac = np.minimum(1.0, POS_CAP / (RISK * np.where(traded, wo["lev"], 1.0)))
    u = np.where(V3, np.where(traded, wfrac * np.nan_to_num(R), 0.0), np.nan)
    tradable = tradable_mask(wo["cost"])
    cb = dict(cfg=cfg, scale=scale, wo=wo, R=R, u=u, TRADABLE=tradable, uni=universes(tradable))
    if ts:
        hit = np.where(V3, (traded & (np.nan_to_num(R) >= 1.0)).astype(float), np.nan)
        trig = np.where(V3, traded.astype(float), np.nan)
        cb.update(ts_state(u, hit, trig))
    return cb


# ── selectors: every one returns pick [A, T] on VALID sessions (universe applied at scoring) ──
def pick_static(w: int) -> np.ndarray:
    return np.where(VALID, WIDX[w], -1)


def pick_table(cb: dict, which: str) -> np.ndarray:
    """Walk-forward static: this year's window = cb['W0'][which] (asset / class / trail3)."""
    return np.where(VALID, cb["W0"][which], -1)


def pick_argmax(cb: dict, scope: str, hl: int, gate: str) -> np.ndarray:
    """Trade the window whose EWMA of u is highest ('strongest so far'); gate='pos' requires it > 0.
    Before an EWMA exists (< 20 sessions) the expanding default window is used, ungated."""
    E = cb["E"][(scope, hl)]
    fin = np.isfinite(E).all(axis=2)
    s = np.where(np.isfinite(E), E, -np.inf)
    best, top = s.argmax(axis=2), s.max(axis=2)                               # argmax ties → shorter window
    pick = np.where(fin, best, cb["W0"][scope])
    ok = VALID & (~fin | (top > 0)) if gate == "pos" else VALID
    return np.where(ok, pick, -1)


@njit(cache=True)
def _incumbent(E, Q, W0, valid, k, alpha, min_gap, gate_pos):
    """Hold the default window W0 unless another window beats it by k standard errors of the
    EWMA difference: switch to argmax_W (m_W − k·SE_W) if > 0, else revert to W0; at most one
    switch per `min_gap` valid sessions. SE uses the EWMA effective sample size (2 − α)/α."""
    A_, T_, NW_ = E.shape
    out = np.full((A_, T_), -1, np.int64)
    fac = alpha / (2.0 - alpha)
    for a in range(A_):
        cur, prev_w0, since = -1, -1, 10 ** 9
        for t in range(T_):
            if not valid[a, t]:
                continue
            w0 = W0[a, t]
            if cur < 0 or (w0 != prev_w0 and cur == prev_w0):    # sitting on the default → follow it
                cur = w0
            prev_w0 = w0
            since += 1
            best, bscore = w0, 0.0
            e0 = E[a, t, w0]
            if np.isfinite(e0):
                for w in range(NW_):
                    if w == w0 or not np.isfinite(E[a, t, w]):
                        continue
                    m = E[a, t, w] - e0
                    v = Q[a, t, w, w] - 2.0 * Q[a, t, w, w0] + Q[a, t, w0, w0] - m * m
                    sc = m - k * np.sqrt(max(v, 0.0) * fac)
                    if sc > bscore:
                        best, bscore = w, sc
            if best != cur and since >= min_gap:
                cur, since = best, 0
            if gate_pos and not (np.isfinite(E[a, t, cur]) and E[a, t, cur] > 0):
                continue
            out[a, t] = cur
    return out


def pick_incumbent(cb: dict, scope: str, hl: int, k: float, gate: str) -> np.ndarray:
    alpha = 1 - np.exp(-np.log(2) / hl)
    return _incumbent(cb["E"][(scope, hl)], cb["Q"][(scope, hl)], cb["W0"][scope].astype(np.int64),
                      VALID, float(k), alpha, SWITCH_GAP, gate == "pos")


TS_ARGMAX = [(sc, h, g) for sc in ("asset", "class") for h in HL_TS for g in ("always", "pos")]
TS_INCUMBENT = [(sc, h, k, g) for sc in ("asset", "class") for h in HL_TS for k in (1.0, 2.0)
                for g in ("always", "pos")]


def static_bank() -> dict:
    return {f"static_{w}": pick_static(w) for w in OR_WINDOWS}


def ts_bank(cb: dict) -> dict:
    """Every walk-forward time-series selector (the RF selectors are added in Cell 7)."""
    bank = {"wf_static": pick_table(cb, "asset"), "wf_static_class": pick_table(cb, "class"),
            "wf_static3": pick_table(cb, "trail3")}
    for sc, h, g in TS_ARGMAX:
        bank[f"ts_max_{sc}_h{h}_{g}"] = pick_argmax(cb, sc, h, g)
    for sc, h, k, g in TS_INCUMBENT:
        bank[f"ts_inc_{sc}_h{h}_k{k:g}_{g}"] = pick_incumbent(cb, sc, h, k, g)
    return bank


ENSEMBLE_W = np.repeat(np.where(VALID, 1.0 / NW, 0.0)[..., None], NW, axis=2)   # every window at RISK / 5


def oracle_year(cb: dict) -> np.ndarray:
    """Hindsight: each asset-year's best window by mean u (an upper bound, not tradable)."""
    best = (cb["S"] / np.maximum(cb["N"], 1)[..., None]).argmax(-1)
    return np.where(VALID, best[:, YI], -1)


def oracle_session(cb: dict) -> np.ndarray:
    """Hindsight per session: the best window if any is positive, else flat (footnote only)."""
    s = np.where(np.isfinite(cb["u"]), cb["u"], -np.inf)
    return np.where(VALID & (s.max(2) > 0), s.argmax(2), -1)


# ── scoring ──
def as_weight(cb: dict, sel: np.ndarray, universe: str) -> np.ndarray:
    um = cb["uni"][universe]
    if sel.ndim == 3:
        return np.where(um[..., None], sel, 0.0)
    w = np.zeros((A, T, NW), np.float32)
    ai, ti = np.nonzero((sel >= 0) & um)
    w[ai, ti, sel[ai, ti]] = 1.0
    return w


def returns_of(cb: dict, sel: np.ndarray, universe: str = "tradable", **kw) -> np.ndarray:
    """Daily portfolio returns of a pick [A, T] or weight [A, T, W] (causal gross cap)."""
    return grid_returns(cb["wo"], as_weight(cb, sel, universe), R=cb["R"], **kw)


def trade_R(cb: dict, pick: np.ndarray, universe: str, mask: np.ndarray) -> np.ndarray:
    """R of the trades a pick actually takes on `mask` sessions."""
    sel = (pick >= 0) & cb["uni"][universe] & mask[None, :]
    rr = np.take_along_axis(cb["R"], np.maximum(pick, 0)[..., None], 2)[..., 0]
    return rr[sel & np.isfinite(rr)]


# ── placebos (Cells 8–9): break the alignment of decisions with outcomes WITHOUT look-ahead.
#    Each decision is delayed by k sessions — the choice made on t−k is used on t. It was made with data
#    through t−k−1, so it knows nothing about t. (Shuffling decisions within a year is NOT a valid null:
#    a later decision's trailing statistics already contain day t, so moving it onto t leaks t's outcome.)
#    A lagged selector keeps its window mix, trade rate, persistence and cross-asset agreement, and its
#    correlation with every other selector lagged by the same k; only the freshness of its information goes.
#    Where the asset had no decision on t−k (not yet eligible), the first decision after t−k is used. ──
PLACEBO_LAG = (21, 252)                       # k drawn uniformly: one month to one year of staleness
NEXT_VALID = np.full((A, T), T, dtype=np.int64)
for _a in range(A):
    _v = np.flatnonzero(VALID[_a])
    NEXT_VALID[_a] = np.r_[_v, T][np.searchsorted(_v, np.arange(T))]


def calendar_lag(pick: np.ndarray, k: int, mask: np.ndarray) -> np.ndarray:
    """pick[:, t − k] used on session t (masked), never a decision made on or after t."""
    t = np.arange(T)
    src = NEXT_VALID[:, np.maximum(t - k, 0)]
    ok = (t[None, :] >= k) & (src < t[None, :]) & mask
    return np.where(ok, np.take_along_axis(pick, np.minimum(src, T - 1), 1), -1)


def _groups(sel: np.ndarray):
    ai, ti = np.nonzero(sel)
    g = ai * NY + YI[ti]
    o = np.argsort(g, kind="stable")
    return ai[o], ti[o], g[o]


def placebo_A(pick: np.ndarray, ungated: np.ndarray, universe_mask: np.ndarray, k: int) -> np.ndarray:
    """Same trading sessions as `pick`; the window is the one its ungated selector chose k sessions earlier."""
    return np.where((pick >= 0) & universe_mask, calendar_lag(ungated, k, universe_mask), -1)


def placebo_B(pick: np.ndarray, ungated: np.ndarray, universe_mask: np.ndarray, rng: np.random.Generator,
              k: int) -> np.ndarray:
    """Random eligible sessions (same count per asset-year as `pick`), each carrying the window the
    ungated selector chose k sessions earlier."""
    out = np.full_like(pick, -1)
    lag = calendar_lag(ungated, k, universe_mask)
    ai, ti, g = _groups((pick >= 0) & universe_mask)
    ea, et, eg = _groups(universe_mask)
    ug, cnt = np.unique(g, return_counts=True)
    starts, ends = np.searchsorted(eg, ug), np.searchsorted(eg, ug, side="right")
    for j, (s0, e0) in enumerate(zip(starts, ends)):
        days = rng.choice(np.arange(s0, e0), size=cnt[j], replace=False)
        out[ea[days], et[days]] = lag[ea[days], et[days]]
    return out


# ── causality test 1: an outcome at session s < t never depends on data from t onward ──
def outcome_causality_test(n_dates: int = 4, seed: int = SEED + 1) -> int:
    rng = np.random.default_rng(seed)
    cols = ["date", "dir", "triggered", "entry_col", "exit_col", "entry", "risk_ps", "R", "cost_R"]
    cases = [(sp, w) for sp in (Spec(), STATIC_WINNER) for w in (OR_WINDOWS[0], OR_WINDOWS[-1])]
    checked = 0
    for s in [x for x in ("SPY", "TLT", "GLD") if x in panels]:
        p0, f0, o0 = panels[s], FEAT[s], ORC[s]
        base = {c: simulate(s, c[0].logic(c[1])) for c in cases}
        for r in rng.choice(np.flatnonzero(p0["dates"].year >= WF_FIRST_YEAR), n_dates, replace=False):
            O, H, L, C, V = (p0[k].copy() for k in ("O", "H", "L", "C", "V"))
            kf = np.exp(rng.normal(0, 0.05, O[r:].shape)).astype(np.float32)
            drop = rng.random(O[r:].shape) < 0.3
            for M in (O, H, L, C):
                M[r:] *= kf
            V[r:] *= np.exp(rng.normal(0, 1, V[r:].shape)).astype(np.float32)
            for M in (O, H, L, C, V):
                M[r:][drop] = np.nan
            p1 = panel_from_arrays(p0["dates"], O, H, L, C, V, p0["daily"]["source"].to_numpy())
            cut = p0["dates"][r]
            try:                                     # the simulator reads these globals → swap, then restore
                panels[s], FEAT[s] = p1, daily_features(p1["daily"])
                ORC[s] = {n: or_cache(p1, n) for n in OR_WINDOWS}
                for c in cases:
                    a, b = simulate(s, c[0].logic(c[1])), base[c]
                    pd.testing.assert_frame_equal(a.loc[a["date"] < cut, cols].reset_index(drop=True),
                                                  b.loc[b["date"] < cut, cols].reset_index(drop=True),
                                                  check_exact=True)
                    checked += 1
            finally:
                panels[s], FEAT[s], ORC[s] = p0, f0, o0
    return checked


# ── causality test 2: scores, picks and tradability at t are unchanged when every u, hit,
#    trigger and cost from t onward is replaced with noise ──
def ts_causality_test(cfg: Config, n_dates: int = 5, seed: int = SEED + 2) -> int:
    rng = np.random.default_rng(seed)
    cb = build_config(cfg)
    base, wf0 = ts_bank(cb), cb["wfeat"]
    tr0 = cb["TRADABLE"]
    V3 = VALID[..., None]
    for t in rng.choice(np.flatnonzero(YEARS >= WF_FIRST_YEAR), n_dates, replace=False):
        u, cost = cb["u"].copy(), cb["wo"]["cost"].copy()
        u[:, t:] = np.where(V3[:, t:], rng.normal(0, 2, u[:, t:].shape), np.nan)
        cost[:, t:] = rng.uniform(0, 1, cost[:, t:].shape)
        hit = np.where(V3, (np.nan_to_num(u) >= 1.0).astype(float), np.nan)
        trig = np.where(V3, (np.nan_to_num(u) != 0).astype(float), np.nan)
        hit[:, :t] = np.where(V3[:, :t], (np.nan_to_num(cb["R"][:, :t]) >= 1).astype(float), np.nan)
        trig[:, :t] = np.where(V3[:, :t], np.isfinite(cb["R"][:, :t]).astype(float), np.nan)
        cb2 = {**cb, "u": u, **ts_state(u, hit, trig)}
        bank = ts_bank(cb2)
        for name, pk in base.items():
            assert np.array_equal(pk[:, :t + 1], bank[name][:, :t + 1]), f"{name} looks ahead at {CAL[t].date()}"
        np.testing.assert_array_equal(cb2["wfeat"][:, :t + 1], wf0[:, :t + 1])
        assert np.array_equal(tradable_mask(cost)[:, :t + 1], tr0[:, :t + 1])
    return n_dates * len(base)


def ewm_brute_check(x: np.ndarray, hl: float, n_cols: int = 6, seed: int = SEED) -> None:
    """The shifted EWMA at t equals a hand-computed EWMA of the observations strictly before t."""
    rng = np.random.default_rng(seed)
    e = ewm_shift(x, hl)
    lam = np.exp(-np.log(2) / hl)
    for _ in range(n_cols):
        a, w = rng.integers(A), rng.integers(NW)
        col = x[a, :, w]
        for t in rng.integers(EW_MIN * 3, T, 5):
            obs = col[:t][np.isfinite(col[:t])]
            if len(obs) < EW_MIN:
                continue
            wts = lam ** np.arange(len(obs))[::-1]
            assert np.isclose(e[a, t, w], (wts * obs).sum() / wts.sum(), rtol=1e-9), (a, w, t)


# ── run: the Static ORB winner's config (15-min winner of the previous project) as a first look ──
t0 = time.perf_counter()
CFG_STATIC = Config(STATIC_WINNER, 0.3, 0.5)
cbS = build_config(CFG_STATIC)
ewm_brute_check(cbS["u"], 120)
n_out = outcome_causality_test()
n_ts = ts_causality_test(CFG_STATIC)

rows = []
banks = {**static_bank(), **ts_bank(cbS), "oracle_year (hindsight)": oracle_year(cbS)}
for name, pk in banks.items():
    st = {}
    r = returns_of(cbS, pk, "tradable", stats_out=st)
    rr = trade_R(cbS, pk, "tradable", IS_MASK)
    rows.append({"selector": name, "IS Sharpe": summarize(r, IS_MASK)[0], "IS trades": rr.size,
                 "IS mean R": rr.mean() if rr.size else np.nan, "cap bind %": 100 * st.get("bind_rate", 0)})
st = {}
r = returns_of(cbS, ENSEMBLE_W, "tradable", stats_out=st)
rows.append({"selector": "ensemble (all 5 at RISK/5)", "IS Sharpe": summarize(r, IS_MASK)[0],
             "IS trades": int(np.isfinite(cbS["R"][cbS["uni"]["tradable"] & IS_MASK[None]]).sum()),
             "IS mean R": np.nanmean(cbS["R"][cbS["uni"]["tradable"] & IS_MASK[None]]),
             "cap bind %": 100 * st["bind_rate"]})
FIRST_LOOK = pd.DataFrame(rows).set_index("selector")

trad_share = pd.DataFrame({s: [cbS["TRADABLE"][i, VALID[i] & IS_MASK].mean() if (VALID[i] & IS_MASK).any() else np.nan,
                               cbS["TRADABLE"][i, VALID[i] & OOS_MASK].mean() if (VALID[i] & OOS_MASK).any() else np.nan]
                           for i, s in enumerate(TRADE_SYMBOLS)}, index=["IS", "2016+"]).T
print(f"Dynamic layer ready in {time.perf_counter() - t0:.0f}s  ·  IS = {WF_FIRST_YEAR}–{IS_END.year} "
      f"(holdout {OOS_START.year}+ is not scored until Cell 9)")
print(f"Causality: EWMA matches a hand computation · {n_out} asset/spec/window outcome series unchanged when all "
      f"data from t onward is corrupted · {n_ts} selector series unchanged at t when u/cost ≥ t are noise")
print(f"\nShare of eligible sessions that pass the tradability rule (median cost ≤ {MAX_COST_R} R, prior 250 sessions) "
      f"— config {CFG_STATIC.label()}:")
print((100 * trad_share).round(1).T.to_string())
print(f"\nFirst look, Static ORB winner config, tradable universe, IS only:")
print(FIRST_LOOK.round(3).to_string())
pk = ts_bank(cbS)["ts_inc_asset_h250_k1_always"]
print("\nIS window shares of ts_inc_asset_h250_k1_always:",
      {f"{w}m": round(float((pk[VALID & IS_MASK[None]] == i).mean()), 3) for i, w in enumerate(OR_WINDOWS)})

Dynamic layer ready in 5s  ·  IS = 2008–2015 (holdout 2016+ is not scored until Cell 9)
Causality: EWMA matches a hand computation · 48 asset/spec/window outcome series unchanged when all data from t onward is corrupted · 135 selector series unchanged at t when u/cost ≥ t are noise

Share of eligible sessions that pass the tradability rule (median cost ≤ 0.15 R, prior 250 sessions) — config candle+gap/close/buf0.02/or_mid/be2.0/x270/gap0.3/or0.5:
        DIA   GLD  IEF    IWM    QQQ  SHY   SLV    SPY   TLT   USO
IS     98.2  95.9  0.0  100.0   24.6  0.0  17.5  100.0   0.0  28.7
2016+  95.1  66.8  0.0  100.0  100.0  0.0   9.0  100.0  35.9   4.6

First look, Static ORB winner config, tradable universe, IS only:
                             IS Sharpe  IS trades  IS mean R  cap bind %
selector                                                                
static_5                         0.845       1469      0.193       0.396
static_10                        0.999       1331      0.181  

## Cell 6 — Stage A: ORB mechanics screen (2008–2015)

In [6]:
# ════════════════════════════════════════════════════════════════════════════
#  Stage A — choose the ORB mechanics the dynamic layer will switch windows for.
#  Screened on 2008–2015 ONLY with the five static windows (no dynamic selector is
#  run here, so the choice cannot favour any selector). Score = mean IS Sharpe of
#  the five static windows (tradable universe). Carried forward: the Static ORB
#  winner (reference), the Stage-A winner, then configs added greedily by score,
#  each with an IS return-series correlation < 0.7 to all those already chosen.
# ════════════════════════════════════════════════════════════════════════════
GRID_SPECS = [Spec(dm, buf, cf, 120, st, be, ex)
              for dm, cf, buf, st, be, ex in itertools.product(
                  ("candle", "candle+gap", "both"), ("touch", "close"), (0.0, 0.02),
                  (("or_opp", 0), ("or_mid", 0), ("atr", 0.10), ("atr", 0.25)), (0.0, 2.0), (270, 389))]
GRID_FILTERS = [(g, o) for g in (0.0, 0.3) for o in (np.inf, 0.5)]
if QUICK:
    GRID_SPECS = [s for s in GRID_SPECS if s.buffer == 0.02 and s.exit_col == 270 and s.be_R == 2.0] + [STATIC_WINNER]
    GRID_SPECS = list(dict.fromkeys(GRID_SPECS))
GRID_CONFIGS = [Config(sp, g, o) for sp in GRID_SPECS for g, o in GRID_FILTERS]
MIN_IS_TRADES = 250             # every static window needs this many IS trades
MAX_CORR = 0.7
STAGEA_UNIS = ("tradable", "equity")
assert CFG_STATIC in GRID_CONFIGS


def stage_a(scale: str = "split") -> tuple:
    """Every grid config's five static windows on 2008–2015 → (table, tradable-universe IS returns
    [configs × windows, IS sessions], cache file name or None). Cell 10 re-runs it on "split+div"."""
    key = hashlib.sha1(repr(([c.key() for c in GRID_CONFIGS], STAGEA_UNIS, str(IS_MASK.sum()), ENGINE_HASH,
                             MODEL_SIG, DATA_SIG, RISK, POS_CAP, GROSS_CAP, MAX_COST_R, sorted(EQUITY_ETFS),
                             sorted(ASSET_CLASS.items()), STAGEA_HASH, scale)).encode()).hexdigest()[:12]
    path = CACHE_DIR / f"stageA3_{key}.npz"
    if path.exists():
        z = np.load(path, allow_pickle=True)
        return pd.DataFrame(z["table"].tolist(), columns=z["cols"].tolist()), z["is_ret"], path.name
    t0, recs, is_ret = time.perf_counter(), [], []
    for i, cfg in enumerate(GRID_CONFIGS):
        cb = build_config(cfg, ts=False, scale=scale)
        for uni in STAGEA_UNIS:
            for w in OR_WINDOWS:
                pk = pick_static(w)
                r = returns_of(cb, pk, uni)
                sh, cagr, mdd = summarize(r, IS_MASK)
                recs.append({"config": cfg.key(), "label": cfg.label(), "universe": uni, "window": w,
                             "IS Sharpe": sh, "IS CAGR %": 100 * cagr, "IS MDD %": 100 * mdd,
                             "IS trades": trade_R(cb, pk, uni, IS_MASK).size})
                if uni == "tradable":
                    is_ret.append(r[IS_IDX].astype(np.float32))
        if (i + 1) % 96 == 0:
            print(f"  {scale}: {i + 1}/{len(GRID_CONFIGS)} configs  ({time.perf_counter() - t0:.0f}s)")
    table, is_ret = pd.DataFrame(recs), np.stack(is_ret)
    np.savez_compressed(path, table=np.array(table.to_numpy(), dtype=object), cols=np.array(table.columns), is_ret=is_ret)
    print(f"Stage A ({scale} prices): {len(GRID_CONFIGS)} configs × {NW} windows × {len(STAGEA_UNIS)} universes in "
          f"{time.perf_counter() - t0:.0f}s")
    return table, is_ret, None


def choose_configs(table: pd.DataFrame, is_ret: np.ndarray) -> tuple:
    """The finalist rule → (chosen keys, per-config score, eligible configs ranked, mean-of-statics IS series)."""
    tr = table[table["universe"] == "tradable"]
    score = tr.groupby("config").agg(label=("label", "first"), mean_static_IS=("IS Sharpe", "mean"),
                                     min_static_IS=("IS Sharpe", "min"), best_static_IS=("IS Sharpe", "max"),
                                     min_trades=("IS trades", "min"))
    score["mean_static_IS_equity"] = table[table["universe"] == "equity"].groupby("config")["IS Sharpe"].mean()
    score = score.loc[[c.key() for c in GRID_CONFIGS]]
    score["row"] = np.arange(len(score))
    ranked = score[score["min_trades"] >= MIN_IS_TRADES].sort_values("mean_static_IS", ascending=False)
    mean_series = is_ret.reshape(len(GRID_CONFIGS), NW, -1).mean(axis=1)           # mean-of-statics IS returns
    chosen = [CFG_STATIC.key()]
    if ranked.index[0] not in chosen:
        chosen.append(ranked.index[0])              # the Stage-A winner is always carried forward
    for k in ranked.index:
        if len(chosen) >= TOP_K:
            break
        if k in chosen:
            continue
        c = [np.corrcoef(mean_series[score.loc[k, "row"]], mean_series[score.loc[j, "row"]])[0, 1] for j in chosen]
        if max(c) < MAX_CORR:
            chosen.append(k)
    return chosen, score, ranked, mean_series


# the cached table depends on the scoring code as well as the engine (MIN_IS_TRADES only filters it afterwards)
STAGEA_HASH = _src_hash(build_config, tradable_mask, universes, pick_static, as_weight, returns_of, trade_R,
                        grid_returns, _causal_alloc, summarize, window_outcomes, Config, stage_a)
IS_IDX = np.flatnonzero(IS_MASK)
stageA, IS_RET_A, _hit = stage_a()
if _hit:
    print(f"Stage A loaded from cache ({_hit})")

# ── score every config on its static windows (tradable universe) and choose the finalists ──
tr = stageA[stageA["universe"] == "tradable"]
chosen, score, ranked, mean_series = choose_configs(stageA, IS_RET_A)
CFG_BY_KEY = {c.key(): c for c in GRID_CONFIGS}
TOP_CONFIGS = [CFG_BY_KEY[k] for k in chosen]

# ── trial bookkeeping for the multiple-testing corrections in Cell 8 ──
N_TRIALS_A = len(stageA)                                  # every static evaluation, both universes
sv = np.linalg.svd((mean_series - mean_series.mean(1, keepdims=True))
                   / mean_series.std(1, keepdims=True).clip(1e-12), compute_uv=False)
ev = sv ** 2 / (sv ** 2).sum()
N_EFF_A = float(1 / (ev ** 2).sum())                      # participation ratio of the config return space

TOP_TABLE = score.loc[chosen, ["label", "mean_static_IS", "min_static_IS", "best_static_IS",
                               "mean_static_IS_equity", "min_trades"]]
TOP_TABLE.insert(0, "rank", [int(np.flatnonzero(ranked.index == k)[0]) + 1 if k in ranked.index else -1 for k in chosen])
corr_top = pd.DataFrame(np.corrcoef(mean_series[score.loc[chosen, "row"].to_numpy()]),
                        index=[f"C{i + 1}" for i in range(len(chosen))], columns=[f"C{i + 1}" for i in range(len(chosen))])
TOP_TABLE.index = [f"C{i + 1}" for i in range(len(chosen))]

print(f"\nTrials in Stage A: {N_TRIALS_A:,} static evaluations ({len(GRID_CONFIGS)} configs × {NW} windows × "
      f"{len(STAGEA_UNIS)} universes) · effective number of independent configs ≈ {N_EFF_A:.1f}")
print(f"{len(ranked)} of {len(GRID_CONFIGS)} configs have ≥ {MIN_IS_TRADES} IS trades in every static window")
print(f"\nConfigs carried forward (C1 = Static ORB winner, C2 = Stage-A winner; the rest greedy by mean static IS "
      f"Sharpe with correlation < {MAX_CORR} to every config already chosen):")
print(TOP_TABLE.round(3).to_string())
print("\nIS correlation of their mean-of-statics return series:")
print(corr_top.round(2).to_string())
print("\nStatic IS Sharpe by window for the chosen configs (tradable universe):")
print(tr[tr["config"].isin(chosen)].pivot_table(index="config", columns="window", values="IS Sharpe")
      .loc[chosen].set_axis(TOP_TABLE.index).round(3).to_string())
print("\nDistribution over all configs of the mean static IS Sharpe:")
print(score["mean_static_IS"].describe().round(3).to_frame().T.to_string())

Stage A loaded from cache (stageA3_4c915c04769e.npz)



Trials in Stage A: 7,680 static evaluations (768 configs × 5 windows × 2 universes) · effective number of independent configs ≈ 3.6
724 of 768 configs have ≥ 250 IS trades in every static window

Configs carried forward (C1 = Static ORB winner, C2 = Stage-A winner; the rest greedy by mean static IS Sharpe with correlation < 0.7 to every config already chosen):
    rank                                                     label  mean_static_IS  min_static_IS  best_static_IS  mean_static_IS_equity  min_trades
C1    63   candle+gap/close/buf0.02/or_mid/be2.0/x270/gap0.3/or0.5           0.864          0.312           1.093                  0.910         555
C2     1  candle+gap/close/buf0.02/atr0.25/be0.0/x389/gap0.3/or0.5           1.351          0.825           1.654                  1.250         830
C3    47      candle/close/buf0.02/atr0.25/be2.0/x389/gap0.3/or0.5           0.997          0.707           1.368                  1.064        1709
C4    59   candle+gap/close/buf0.02/or_m

## Cell 7 — Random-forest window selectors (walk-forward)

In [7]:
# ════════════════════════════════════════════════════════════════════════════
#  Random-forest window selectors, walk-forward: refit every January from 2006 on
#  all eligible, tradable asset-sessions BEFORE Jan 1 of the predicted year.
#   • stacked   : RandomForestRegressor on rows (asset, session, W) → μ̂_W = E[u_W | x]
#                 trade W* = argmax_W μ̂_W
#   • multiclass: RandomForestClassifier → P(class | x), class = best W by u (or "none");
#                 expected value EV_W = Σ_c P(c | x) · M[c, W], M = training mean of u_W in class c
#  Gates: always | score > 0 | score > q50 / q75 of the previous three years' genuine
#  walk-forward out-of-sample scores (never training or OOB predictions).
#  An asset whose own and class training pool has < 500 sessions trades its expanding
#  default window W0, ungated. Predictions start 2006 (to calibrate gates); scoring 2008+.
# ════════════════════════════════════════════════════════════════════════════
RF_YEARS = [y for y in YEAR_LIST if y >= RF_FIRST_YEAR]
RF_PARAMS = {"stacked": dict(n_estimators=100 if QUICK else 300, cap=10_000 if QUICK else 25_000, leaf=250),
             "multiclass": dict(n_estimators=100 if QUICK else 300, cap=5_000 if QUICK else 10_000, leaf=100)}
RF_GATES = ("always", "0", "q50", "q75")
MIN_COVER = 500                 # training-pool sessions needed for the asset (or its class)
NONE = NW                       # multiclass "no window wins" class
FORM_TAG = {"stacked": "st", "multiclass": "mc"}


def rf_design(cb: dict) -> dict:
    """Feature tensors: stacked [A, T, W, F] and multiclass [A, T, F]."""
    wf = cb["wfeat"]
    Xs = np.concatenate([np.broadcast_to(XB[:, :, None, :], (A, T, NW, XB.shape[2])), wf], axis=3)
    own = wf[..., :6]                                                    # the six own-window features
    ew = own[..., 1]                                                     # ew_u_a250 per window
    lead = np.where(np.isfinite(ew).any(axis=2), np.argmax(np.nan_to_num(ew, nan=-np.inf), axis=2), np.nan)
    lead = lead[..., None].astype(np.float32)
    Xm = np.concatenate([XB, own.reshape(A, T, -1), lead], axis=2)
    cols_s = XB_COLS + WFEAT_COLS
    cols_m = XB_COLS + [f"{c}_{w}m" for w in OR_WINDOWS for c in WFEAT_COLS[:6]] + ["lead_window"]
    assert Xs.shape[3] == len(cols_s) and Xm.shape[2] == len(cols_m)
    return dict(stacked=Xs.astype(np.float32), multiclass=Xm.astype(np.float32), cols_s=cols_s, cols_m=cols_m)


def rf_seed(cfg: Config, form: str, y: int) -> int:
    return int(hashlib.sha1(f"{cfg.key()}|{form}|{y}".encode()).hexdigest()[:8], 16) % 2 ** 31


def fit_year(cb: dict, X: dict, form: str, y: int, u: np.ndarray | None = None):
    """Fit on VALID & TRADABLE sessions of years < y; score every VALID session of year y.
    Returns (te flat indices, score [n, W], train_end, n_train_rows, model)."""
    u = cb["u"] if u is None else u
    prm = RF_PARAMS[form]
    tr = VALID & cb["TRADABLE"] & (YEARS < y)[None, :]
    te = VALID & (YEARS == y)[None, :]
    train_end = CAL[np.flatnonzero(tr.any(axis=0)).max()]
    assert train_end < pd.Timestamp(f"{y}-01-01")
    seed = rf_seed(cb["cfg"], form, y)
    if form == "stacked":
        Xtr, ytr = X["stacked"][tr].reshape(-1, X["stacked"].shape[3]), u[tr].reshape(-1)
        m = RandomForestRegressor(n_estimators=prm["n_estimators"], max_samples=min(len(ytr), prm["cap"]),
                                  min_samples_leaf=prm["leaf"], max_features="sqrt", n_jobs=N_JOBS,
                                  random_state=seed).fit(Xtr, ytr)
        sc = m.predict(X["stacked"][te].reshape(-1, X["stacked"].shape[3])).reshape(-1, NW)
    else:
        ut = u[tr]
        lab = np.where(ut.max(1) > 0, ut.argmax(1), NONE)
        m = RandomForestClassifier(n_estimators=prm["n_estimators"], max_samples=min(len(lab), prm["cap"]),
                                   min_samples_leaf=prm["leaf"], max_features="sqrt", class_weight=None,
                                   n_jobs=N_JOBS, random_state=seed).fit(X["multiclass"][tr], lab)
        M = np.stack([ut[lab == c].mean(0) for c in m.classes_])                 # [classes, W]
        sc = m.predict_proba(X["multiclass"][te]) @ M                             # EV_W
    return np.flatnonzero(te.ravel()), sc.astype(np.float32), train_end, int(tr.sum()), m


RF_HASH = _src_hash(intraday_stats, asset_features, market_features, stack_features, ewm_shift, class_mean,
                    to_assets, ts_state, build_config, tradable_mask, year_sums, w0_tables, rf_design, rf_seed, fit_year)


def rf_cache_path(cb: dict, form: str, y: int) -> Path:
    key = hashlib.sha1(json.dumps({"cfg": cb["cfg"].key(), "scale": cb["scale"], "form": form, "year": int(y),
                                   "prm": RF_PARAMS[form],
                                   "code": RF_HASH, "engine": ENGINE_HASH, "model": MODEL_SIG, "data": DATA_SIG, "quick": QUICK,
                                   "syms": TRADE_SYMBOLS, "xb": XB_COLS, "wf": WFEAT_COLS, "cost": MAX_COST_R,
                                   "risk": [RISK, POS_CAP], "sk": sklearn.__version__,
                                   "cls": sorted(ASSET_CLASS.items()),
                                   "ew": [EW_MIN, list(HL_U), list(HL_TS), W0_MIN]}, sort_keys=True).encode()
                       ).hexdigest()[:16]
    return CACHE_DIR / f"rf3_{key}.npz"


def wf_scores(cb: dict, X: dict, form: str) -> dict:
    """Walk-forward scores [A, T, W] (NaN where not predicted) + per-year training metadata. The
    multiclass forest also keeps its class probabilities [A, T, W + 1] (last column = 'no window
    wins'): P(window W is the session's best) is what Cell 12 compares with volatility."""
    S = np.full((A, T, NW), np.nan, np.float32)
    P = np.full((A, T, NW + 1), np.nan, np.float32) if form == "multiclass" else None
    meta = {}
    for y in RF_YEARS:
        path = rf_cache_path(cb, form, y)
        te_now = np.flatnonzero((VALID & (YEARS == y)[None, :]).ravel())
        z = np.load(path) if path.exists() else None
        if z is not None and (P is None or "proba" in z):
            idx, sc, train_end, n_tr = z["idx"], z["score"], pd.Timestamp(str(z["train_end"])), int(z["n_train"])
            pr = z["proba"] if P is not None else None
            assert np.array_equal(idx, te_now), f"cached rows changed: {path.name}"
        else:
            idx, sc, train_end, n_tr, m = fit_year(cb, X, form, y)
            if z is not None:                                       # older cache without probabilities
                assert np.array_equal(z["score"], sc), f"refit is not deterministic: {path.name}"
            extra = {}
            if P is not None:
                pr = np.zeros((len(idx), NW + 1), np.float32)
                pr[:, m.classes_] = m.predict_proba(X["multiclass"].reshape(A * T, -1)[idx])
                extra["proba"] = pr
            np.savez_compressed(path, idx=idx, score=sc, train_end=str(train_end.date()), n_train=n_tr, **extra)
        assert train_end < pd.Timestamp(f"{y}-01-01")
        S.reshape(A * T, NW)[idx] = sc
        if P is not None:
            P.reshape(A * T, NW + 1)[idx] = pr
        meta[y] = {"train_end": train_end.date(), "train_sessions": n_tr}
    return {"score": S, "meta": meta, **({"proba": P} if P is not None else {})}


def coverage(cb: dict) -> np.ndarray:
    """[A, T] bool: in year y the asset's own (or its class's) training pool has ≥ MIN_COVER sessions."""
    pool = VALID & cb["TRADABLE"]
    per_year = np.stack([pool[:, YI == yi].sum(1) for yi in range(NY)], axis=1)        # [A, NY]
    before = np.cumsum(per_year, 1) - per_year
    cls = np.stack([before[CLASS_OF == c].sum(0) for c in range(NC)])[CLASS_OF]
    return ((before >= MIN_COVER) | (cls >= MIN_COVER))[:, YI]


def gate_thresholds(cb: dict, S: np.ndarray, cov: np.ndarray, q: float) -> np.ndarray:
    """Per-session threshold: q-quantile of max-scores on tradable, covered sessions of years y−3..y−1,
    each scored by its own walk-forward model (NaN when no such history)."""
    fin = np.isfinite(S).all(axis=2)
    top = np.where(fin, S.max(axis=2, initial=-np.inf, where=np.isfinite(S)), np.nan)
    ok = VALID & cb["TRADABLE"] & cov & fin
    thr = np.full(T, np.nan)
    for y in RF_YEARS:
        prior = ok & ((YEARS >= y - 3) & (YEARS <= y - 1))[None, :]
        if prior.sum() >= 250:
            thr[YEARS == y] = np.quantile(top[prior], q)
    return thr


def rf_bank(cb: dict, preds: dict) -> dict:
    """name → pick [A, T] for both formulations × the four gates."""
    cov = coverage(cb)
    bank = {}
    for form, pr in preds.items():
        S = pr["score"]
        fin = np.isfinite(S).all(axis=2) & VALID & cov
        best = np.argmax(np.where(np.isfinite(S), S, -np.inf), axis=2)
        top = np.where(fin, S.max(axis=2, initial=-np.inf, where=np.isfinite(S)), -np.inf)
        fallback = np.where(VALID & ~fin, cb["W0"]["asset"], -1)           # uncovered → W0, ungated
        for g in RF_GATES:
            if g == "always":
                h = np.full(T, -np.inf)
            elif g == "0":
                h = np.zeros(T)
            else:
                h = gate_thresholds(cb, S, cov, int(g[1:]) / 100)
            go = fin & (top > np.nan_to_num(h, nan=np.inf)[None, :])        # no calibration history → flat
            bank[f"rf_{FORM_TAG[form]}_{g}"] = np.where(go, best, fallback)
    return bank


def refit_test(cb: dict, X: dict, form: str, y: int) -> None:
    """Re-fit year y after replacing every outcome from Jan 1 of y onward with noise:
    the year-y predictions must be bit-identical (training never touches year ≥ y)."""
    rng = np.random.default_rng(SEED + 3)
    u2 = cb["u"].copy()
    fut = (YEARS >= y)[None, :, None] & np.isfinite(u2)
    u2[fut] = rng.normal(0, 3, fut.sum())
    i1, s1, *_ = fit_year(cb, X, form, y)
    i2, s2, *_ = fit_year(cb, X, form, y, u=u2)
    assert np.array_equal(i1, i2) and np.array_equal(s1, s2), "RF training used outcomes from the predicted year"


def _row_spearman(a: np.ndarray, b: np.ndarray) -> float:
    """Mean across rows of the Spearman correlation between a[i] and b[i] (rows with no spread skipped)."""
    ra, rb = stats.rankdata(a, axis=1), stats.rankdata(b, axis=1)
    ra, rb = ra - ra.mean(1, keepdims=True), rb - rb.mean(1, keepdims=True)
    den = np.sqrt((ra ** 2).sum(1) * (rb ** 2).sum(1))
    ok = den > 0
    return float(np.mean((ra * rb).sum(1)[ok] / den[ok])) if ok.any() else np.nan


def rf_diagnostics(cb: dict, preds: dict) -> pd.DataFrame:
    """Per in-sample year (2006–2015, tradable sessions): pooled rank IC of score vs realised u, mean
    within-session rank IC across the five windows, how often the pick is the session's best window
    (vs always naming that year's most frequent best window), and mean u of the picked window minus
    the expanding default window W0. Rank ICs understate skill on skewed payoffs (a window that wins
    big but rarely ranks low on most days), so the last column is the economic one."""
    rows = []
    trad = VALID & cb["TRADABLE"]
    u, w0 = cb["u"], cb["W0"]["asset"]
    for form, pr in preds.items():
        S = pr["score"]
        best = np.argmax(np.where(np.isfinite(S), S, -np.inf), axis=2)
        for y in [y for y in RF_YEARS if y <= IS_END.year]:
            m = trad & (YEARS == y)[None, :] & np.isfinite(S).all(axis=2)
            if m.sum() < 50:
                continue
            s, uu, b = S[m], u[m], best[m]
            live = uu.max(1) > 0
            up = np.take_along_axis(uu, b[:, None], 1)[:, 0]
            u0 = np.take_along_axis(uu, w0[m][:, None], 1)[:, 0]
            rows.append({"form": form, "year": y, "sessions": int(m.sum()),
                         "IC pooled": stats.spearmanr(s.ravel(), uu.ravel())[0],
                         "IC within session": _row_spearman(s, uu),
                         "picked = best": float((b[live] == uu[live].argmax(1)).mean()),
                         "modal best": float(np.bincount(uu[live].argmax(1), minlength=NW).max() / max(live.sum(), 1)),
                         "u picked − u W0": float(up.mean() - u0.mean())})
    return pd.DataFrame(rows)


def rf_config(cfg: Config, scale: str = "split") -> tuple:
    """One finalist on one price basis → (outcome tensors + TS state, forest design, walk-forward scores of
    both formulations, diagnostics). Cell 10 calls it again on "split+div"."""
    cb = build_config(cfg, scale=scale)
    X = rf_design(cb)
    preds = {form: wf_scores(cb, X, form) for form in ("stacked", "multiclass")}
    return cb, X, preds, rf_diagnostics(cb, preds)


# ── run the walk-forward forests on every Stage-A finalist ──
CB, X_RF, PRED, RF_DIAG = {}, {}, {}, {}
for i, cfg in enumerate(TOP_CONFIGS):
    t0 = time.perf_counter()
    k = cfg.key()
    CB[k], X_RF[k], PRED[k], RF_DIAG[k] = rf_config(cfg)
    cb, X = CB[k], X_RF[k]
    d = RF_DIAG[k]
    is_ = d[(d["year"] >= WF_FIRST_YEAR) & (d["year"] <= IS_END.year)].groupby("form").mean(numeric_only=True)
    print(f"[C{i + 1}] {cfg.label()}  ({time.perf_counter() - t0:.0f}s)")
    for form in ("stacked", "multiclass"):
        r = is_.loc[form]
        print(f"     {form:10s} IS 2008–15: IC pooled {r['IC pooled']:+.3f} · IC within session "
              f"{r['IC within session']:+.3f} · picked = best {r['picked = best']:.1%} (always naming the year's "
              f"modal best window: {r['modal best']:.1%}) · u(picked) − u(W0) {r['u picked − u W0']:+.4f} R/session")

refit_test(CB[TOP_CONFIGS[0].key()], X_RF[TOP_CONFIGS[0].key()], "stacked", 2012)
meta = PRED[TOP_CONFIGS[0].key()]["stacked"]["meta"]
print(f"\nRefit test passed: the 2012 stacked forest is bit-identical when every outcome from 2012 onward is noise.")
print("Training windows (C1, stacked):", ", ".join(f"{y}: ≤{m['train_end']} ({m['train_sessions']:,} asset-sessions)"
                                                  for y, m in list(meta.items())[::4]))
cov1 = coverage(CB[TOP_CONFIGS[0].key()])
print("RF-covered share of eligible sessions by asset, 2008+ (C1):",
      {s: round(float(cov1[a, VALID[a] & (YEARS >= WF_FIRST_YEAR)].mean()), 2) for a, s in enumerate(TRADE_SYMBOLS)})

[C1] candle+gap/close/buf0.02/or_mid/be2.0/x270/gap0.3/or0.5  (0s)
     stacked    IS 2008–15: IC pooled -0.017 · IC within session +0.014 · picked = best 16.6% (always naming the year's modal best window: 44.7%) · u(picked) − u(W0) -0.0044 R/session
     multiclass IS 2008–15: IC pooled -0.107 · IC within session -0.119 · picked = best 50.4% (always naming the year's modal best window: 44.7%) · u(picked) − u(W0) +0.0024 R/session


[C2] candle+gap/close/buf0.02/atr0.25/be0.0/x389/gap0.3/or0.5  (0s)
     stacked    IS 2008–15: IC pooled -0.012 · IC within session +0.007 · picked = best 15.3% (always naming the year's modal best window: 61.7%) · u(picked) − u(W0) -0.0116 R/session
     multiclass IS 2008–15: IC pooled -0.051 · IC within session -0.025 · picked = best 66.0% (always naming the year's modal best window: 61.7%) · u(picked) − u(W0) +0.0025 R/session


[C3] candle/close/buf0.02/atr0.25/be2.0/x389/gap0.3/or0.5  (0s)
     stacked    IS 2008–15: IC pooled -0.003 · IC within session +0.001 · picked = best 12.6% (always naming the year's modal best window: 61.1%) · u(picked) − u(W0) -0.0080 R/session
     multiclass IS 2008–15: IC pooled -0.128 · IC within session -0.040 · picked = best 20.2% (always naming the year's modal best window: 61.1%) · u(picked) − u(W0) +0.0066 R/session


[C4] candle+gap/close/buf0.02/or_mid/be2.0/x389/gap0.0/orinf  (0s)
     stacked    IS 2008–15: IC pooled +0.045 · IC within session +0.032 · picked = best 17.1% (always naming the year's modal best window: 40.6%) · u(picked) − u(W0) +0.0059 R/session
     multiclass IS 2008–15: IC pooled -0.117 · IC within session -0.124 · picked = best 46.1% (always naming the year's modal best window: 40.6%) · u(picked) − u(W0) -0.0050 R/session



Refit test passed: the 2012 stacked forest is bit-identical when every outcome from 2012 onward is noise.
Training windows (C1, stacked): 2006: ≤2005-12-30 (583 asset-sessions), 2010: ≤2009-12-31 (3,466 asset-sessions), 2014: ≤2013-12-31 (7,299 asset-sessions), 2018: ≤2017-12-29 (11,508 asset-sessions), 2022: ≤2021-12-31 (16,090 asset-sessions), 2026: ≤2025-12-31 (19,568 asset-sessions)
RF-covered share of eligible sessions by asset, 2008+ (C1): {'DIA': 1.0, 'GLD': 0.88, 'IEF': 0.0, 'IWM': 1.0, 'QQQ': 1.0, 'SHY': 0.0, 'SLV': 0.87, 'SPY': 1.0, 'TLT': 0.19, 'USO': 0.86}


## Cell 8 — Bootstrap selection of the best variation (2008–2015 only)

In [8]:
# ════════════════════════════════════════════════════════════════════════════
#  Bootstrap selection — 2008–2015 ONLY. Nothing in this cell reads a 2016+ return.
#   S*  (static benchmark, per spec): for each asset, the static window with the best
#       10th-percentile bootstrap IS Sharpe of that asset's standalone series.
#   D*  (the dynamic strategy): among every time-series and random-forest selector on
#       every Stage-A finalist, the one maximising the 10th percentile of the PAIRED
#       bootstrap distribution of SR(D) − SR(S* of the same spec).
#   One shared set of stationary-bootstrap resamples (Politis–White block length) is used
#   for every series. D*, S*, the vol-matching factor and the test are then written to a
#   pre-registration file BEFORE the holdout is touched (Cell 9).
#  Data snooping: SPA_c / RC / StepM (vs S* and vs the all-window ensemble), a model
#  confidence set, and deflated Sharpe ratios that count Stage A and Stage B trials.
# ════════════════════════════════════════════════════════════════════════════
UNI = "tradable"
IS_IDX, OOS_IDX = np.flatnonzero(IS_MASK), np.flatnonzero(OOS_MASK)
N_IS = len(IS_IDX)
ALT_BLOCKS = (10, 20, 60)
GAMMA = 0.5772156649                     # Euler–Mascheroni
DYN_PREFIX = ("ts_max", "ts_inc", "rf_")
CONTROL_NAMES = ("wf_static", "wf_static_class", "wf_static3")


# ── bootstrap machinery ──
def pw_block(x: np.ndarray) -> float:
    """Politis–White (2004) automatic mean block length for the stationary bootstrap,
    with the Patton–Politis–White (2009) correction."""
    x = np.asarray(x, float) - np.mean(x)
    n = len(x)
    kn = max(5, int(np.sqrt(np.log10(n))))
    mmax = int(np.ceil(np.sqrt(n))) + kn
    bmax = np.ceil(min(3 * np.sqrt(n), n / 3))
    acv = np.array([x[: n - k] @ x[k:] / n for k in range(mmax + 1)])
    rho = acv[1:] / acv[0]
    insig = np.abs(rho) < 2 * np.sqrt(np.log10(n) / n)
    mhat = next((m for m in range(mmax - kn + 1) if insig[m:m + kn].all()), None)
    M = mmax if mhat is None else min(2 * max(mhat, 1), mmax)
    k = np.arange(-M, M + 1)
    lam = np.clip(2 * (1 - np.abs(k) / M), 0, 1)                  # flat-top lag window
    R = acv[np.abs(k)]
    G, g0 = np.sum(lam * np.abs(k) * R), np.sum(lam * R)
    b = (2 * G ** 2 / (2 * g0 ** 2)) ** (1 / 3) * n ** (1 / 3) if g0 > 0 else 1.0
    return float(min(max(b, 1.0), bmax))


def sb_counts(n: int, B: int, L: float, seed: int) -> np.ndarray:
    """Stationary bootstrap (Politis–Romano), mean block L, circular. Returns the [B, n]
    matrix of how often each date is drawn, so that every bootstrap mean is C @ x / n."""
    rng = np.random.default_rng(seed)
    idx = np.empty((B, n), np.int32)
    idx[:, 0] = rng.integers(0, n, B)
    new = rng.random((B, n)) < 1.0 / L
    jump = rng.integers(0, n, (B, n), dtype=np.int32)
    for t in range(1, n):
        idx[:, t] = np.where(new[:, t], jump[:, t], (idx[:, t - 1] + 1) % n)
    flat = (idx + (np.arange(B, dtype=np.int64) * n)[:, None]).ravel()
    return np.bincount(flat, minlength=B * n).reshape(B, n).astype(np.float32)


def boot_sharpe(C: np.ndarray, X: np.ndarray) -> np.ndarray:
    """[B, K] annualised bootstrap Sharpe of the columns of X [n, K] (−inf where flat)."""
    n = X.shape[0]
    m1, m2 = C @ X / n, C @ (X ** 2) / n
    sd = np.sqrt(np.maximum(m2 - m1 ** 2, 0))
    return np.where(sd > 1e-12, m1 / np.where(sd > 1e-12, sd, 1) * np.sqrt(252), -np.inf)


def sharpe(x: np.ndarray) -> float:
    sd = np.std(x)
    return float(np.mean(x) / sd * np.sqrt(252)) if sd > 1e-12 else np.nan


# ── series ──
def only(pick: np.ndarray, rows: np.ndarray) -> np.ndarray:
    """The pick restricted to a subset of assets (everything else flat)."""
    return np.where(rows[:, None], pick, -1)


def series_of(cb: dict, preds: dict) -> tuple:
    """One finalist's picks and daily returns → (bank, returns, per-asset statics, per-class statics), keyed by
    (config, …): every static / TS / RF selector, the all-window ensemble, and the standalone statics behind S*.
    Cell 10 calls it again on dividend-corrected prices."""
    k = cb["cfg"].key()
    bank = {(k, nm): pk for nm, pk in {**static_bank(), **ts_bank(cb), **rf_bank(cb, preds)}.items()}
    ret = {kn: returns_of(cb, pk, UNI) for kn, pk in bank.items()}
    ret[(k, "ensemble")] = returns_of(cb, ENSEMBLE_W, UNI)
    a_ret = {(k, a, w): returns_of(cb, only(pick_static(w), np.arange(A) == a), UNI) for a in range(A) for w in OR_WINDOWS}
    c_ret = {(k, ci, w): returns_of(cb, only(pick_static(w), CLASS_OF == ci), UNI) for ci in range(NC) for w in OR_WINDOWS}
    return bank, ret, a_ret, c_ret


t0 = time.perf_counter()
CFG_TAG = {c.key(): f"C{i + 1}" for i, c in enumerate(TOP_CONFIGS)}
# (config, name) → pick [A, T] / daily returns [T]; (config, asset | class, window) → standalone static returns
BANK, RET, ASSET_RET, CLASS_RET = {}, {}, {}, {}
for c in TOP_CONFIGS:
    for into, part in zip((BANK, RET, ASSET_RET, CLASS_RET), series_of(CB[c.key()], PRED[c.key()])):
        into.update(part)
print(f"Series built: {len(RET)} portfolio variants + {len(ASSET_RET) + len(CLASS_RET)} per-asset/class statics "
      f"({time.perf_counter() - t0:.0f}s)")

# ── one shared bootstrap over the 2008–2015 sessions ──
dyn_keys = [kn for kn in RET if kn[1].startswith(DYN_PREFIX)]
PW_ALL = np.array([pw_block(RET[kn][IS_IDX] - RET[(kn[0], "ensemble")][IS_IDX]) for kn in dyn_keys])
BLOCK_L = float(np.median(PW_ALL))
C_IS = sb_counts(N_IS, N_BOOT, BLOCK_L, SEED + 10)
print(f"Stationary bootstrap: B = {N_BOOT:,} resamples of the {N_IS:,} sessions of 2008–2015, mean block "
      f"{BLOCK_L:.1f} sessions (median Politis–White length over the {len(dyn_keys)} dynamic-minus-ensemble "
      f"differentials; range {PW_ALL.min():.1f}–{PW_ALL.max():.1f}); robustness blocks {ALT_BLOCKS}")


def q10(X: np.ndarray, C: np.ndarray = None) -> np.ndarray:
    return np.quantile(boot_sharpe(C_IS if C is None else C, X), 0.10, axis=0)


# S*_asset / S*_class / S*_common per spec, chosen by the 10th-percentile rule
S_STAR, S_STAR_TABLE = {}, []
for c in TOP_CONFIGS:
    k, cb = c.key(), CB[c.key()]
    common = OR_WINDOWS[int(np.argmax(q10(np.stack([RET[(k, f"static_{w}")][IS_IDX] for w in OR_WINDOWS], 1))))]
    wa = np.full(A, WIDX[common])
    for a in range(A):
        Xa = np.stack([ASSET_RET[(k, a, w)][IS_IDX] for w in OR_WINDOWS], 1)
        if (Xa != 0).sum(0).min() >= 100:                        # ≥ 100 IS trading days in every window
            wa[a] = int(np.argmax(q10(Xa)))
    wc = np.full(NC, WIDX[common])
    for ci in range(NC):
        Xc = np.stack([CLASS_RET[(k, ci, w)][IS_IDX] for w in OR_WINDOWS], 1)
        if (Xc != 0).sum(0).min() >= 100:
            wc[ci] = int(np.argmax(q10(Xc)))
    picks = {"S*_asset": np.where(VALID, wa[:, None], -1), "S*_class": np.where(VALID, wc[CLASS_OF][:, None], -1),
             "S*_common": pick_static(common)}
    for nm, pk in picks.items():
        BANK[(k, nm)] = pk
        RET[(k, nm)] = returns_of(cb, pk, UNI)
    S_STAR[k] = {"asset": {TRADE_SYMBOLS[a]: OR_WINDOWS[wa[a]] for a in range(A)},
                 "class": {CLASSES[ci]: OR_WINDOWS[wc[ci]] for ci in range(NC)}, "common": common}
    S_STAR_TABLE.append({"config": CFG_TAG[k], **{s: OR_WINDOWS[wa[a]] for a, s in enumerate(TRADE_SYMBOLS)},
                         "common": common})

# ── D*: the paired 10th-percentile rule ──
def sel_table(C: np.ndarray) -> pd.DataFrame:
    rows = []
    for c in TOP_CONFIGS:
        k = c.key()
        names = [nm for (kk, nm) in RET if kk == k and nm.startswith(DYN_PREFIX)]
        X = np.stack([RET[(k, nm)][IS_IDX] for nm in names], 1)
        b = RET[(k, "S*_asset")][IS_IDX]
        sd, sdD = boot_sharpe(C, X), boot_sharpe(C, b[:, None])
        diff = sd - sdD
        for j, nm in enumerate(names):
            rows.append({"config": CFG_TAG[k], "key": k, "selector": nm, "IS Sharpe": sharpe(X[:, j]),
                         "S* IS Sharpe": sharpe(b), "q10 SR": np.quantile(sd[:, j], 0.1),
                         "q10 ΔSR vs S*": np.quantile(diff[:, j], 0.1), "median ΔSR": np.median(diff[:, j]),
                         "P(ΔSR > 0)": float((diff[:, j] > 0).mean())})
    return pd.DataFrame(rows).sort_values("q10 ΔSR vs S*", ascending=False, ignore_index=True)


SEL = sel_table(C_IS)
D_KEY, D_NAME = SEL.loc[0, "key"], SEL.loc[0, "selector"]
D_CFG = CFG_BY_KEY[D_KEY]
r_D_is, r_S_is = RET[(D_KEY, D_NAME)][IS_IDX], RET[(D_KEY, "S*_asset")][IS_IDX]
VOL_K = round(float(r_S_is.std() / r_D_is.std()), 6)          # frozen vol-matching factor

# robustness of the choice to the block length (selection itself stays on the PW block)
ALT_PICK = {}
for L in ALT_BLOCKS:
    st = sel_table(sb_counts(N_IS, N_BOOT, L, SEED + 10 + L))
    ALT_PICK[L] = (CFG_TAG[st.loc[0, "key"]], st.loc[0, "selector"],
                   int(np.flatnonzero((SEL["key"] == st.loc[0, "key"]) & (SEL["selector"] == st.loc[0, "selector"]))[0]) + 1)

# the "best variation" overall (any family), by absolute 10th-percentile Sharpe — descriptive, no inference
ALL_KEYS = [kn for kn in RET]
Q10_ALL = pd.Series(q10(np.stack([RET[kn][IS_IDX] for kn in ALL_KEYS], 1)),
                    index=pd.MultiIndex.from_tuples([(CFG_TAG[k], n) for k, n in ALL_KEYS], names=["config", "selector"]))

# ── data-snooping tests on 2008–2015 (vol-matched differentials, shared resamples) ──
def spa_tests(D: np.ndarray, C: np.ndarray) -> dict:
    """Hansen SPA (c, l, u), White RC and Romano–Wolf StepM for H0: no column of D beats 0."""
    n = D.shape[0]
    dbar = D.mean(0)
    dstar = C @ D / n                                                    # [B, K]
    om = np.sqrt(n) * dstar.std(0)
    om = np.where(om > 1e-12, om, np.inf)
    t = np.sqrt(n) * dbar / om
    T_spa = max(t.max(), 0.0)
    thr = -np.sqrt(om ** 2 / n * 2 * np.log(np.log(n)))
    out = {}
    for nm, mu in (("SPA_c", np.where(dbar >= thr, dbar, 0.0)), ("SPA_l", np.maximum(dbar, 0.0)), ("SPA_u", dbar)):
        Tb = np.maximum((np.sqrt(n) * (dstar - mu) / om).max(1), 0.0)
        out[nm] = float((1 + (Tb >= T_spa).sum()) / (len(Tb) + 1))
    T_rc = np.sqrt(n) * dbar.max()
    out["RC"] = float((1 + ((np.sqrt(n) * (dstar - dbar)).max(1) >= T_rc).sum()) / (len(dstar) + 1))
    active, rejected = np.ones(len(dbar), bool), np.zeros(len(dbar), bool)
    Z = np.sqrt(n) * (dstar - dbar) / om
    while active.any():
        crit = np.quantile(Z[:, active].max(1), 0.95)
        new = active & (t > crit)
        if not new.any():
            break
        rejected |= new
        active &= ~new
    out["StepM rejections"] = int(rejected.sum())
    out["StepM rejected"] = rejected
    return out


def mcs(Lmat: np.ndarray, C: np.ndarray) -> np.ndarray:
    """Hansen–Lunde–Nason model confidence set, T_max statistic, run to the last model so every
    model gets its MCS p-value (the α-MCS = models with p ≥ α)."""
    n, K = Lmat.shape
    Lb, Lbar = C @ Lmat / n, Lmat.mean(0)
    alive, pvals, pmax = np.ones(K, bool), np.ones(K), 0.0
    while alive.sum() > 1:
        idx = np.flatnonzero(alive)
        di = Lbar[idx] - Lbar[idx].mean()
        db = Lb[:, idx] - Lb[:, idx].mean(1, keepdims=True) - di
        se = np.sqrt(np.maximum((db ** 2).mean(0), 1e-30))
        tt = di / se
        pmax = max(pmax, float((((db / se).max(1)) >= tt.max()).mean()))
        worst = idx[int(np.argmax(tt))]
        pvals[worst], alive[worst] = pmax, False
    return pvals


def vm(x: np.ndarray, ref: np.ndarray) -> np.ndarray:
    return x * (ref.std() / x.std()) if x.std() > 1e-12 else np.zeros_like(x)


D_vs_S = np.stack([vm(RET[kn][IS_IDX], RET[(kn[0], "S*_asset")][IS_IDX]) - RET[(kn[0], "S*_asset")][IS_IDX]
                   for kn in dyn_keys], 1)
D_vs_E = np.stack([vm(RET[kn][IS_IDX], RET[(kn[0], "ensemble")][IS_IDX]) - RET[(kn[0], "ensemble")][IS_IDX]
                   for kn in dyn_keys], 1)
SPA_S, SPA_E = spa_tests(D_vs_S, C_IS), spa_tests(D_vs_E, C_IS)
SPA_BLOCKS = {L: spa_tests(D_vs_S, sb_counts(N_IS, N_BOOT, L, SEED + 20 + L))["SPA_c"] for L in ALT_BLOCKS}

mcs_keys = list(RET)
L_mcs = -np.stack([RET[kn][IS_IDX] / max(RET[kn][IS_IDX].std(), 1e-12) for kn in mcs_keys], 1)   # unit-vol losses
MCS_P = pd.Series(mcs(L_mcs, C_IS), index=pd.MultiIndex.from_tuples([(CFG_TAG[k], n) for k, n in mcs_keys]))


# ── deflated Sharpe ratios ──
def dsr(x: np.ndarray, n_trials: float, var_sr: float) -> float:
    """Bailey & López de Prado (2014): P(true SR > the expected max of n_trials null SRs)."""
    sr = x.mean() / x.std()
    sk, ku = stats.skew(x), stats.kurtosis(x, fisher=False)
    sr0 = (np.sqrt(var_sr) * ((1 - GAMMA) * stats.norm.ppf(1 - 1 / n_trials)
                              + GAMMA * stats.norm.ppf(1 - 1 / (n_trials * np.e)))) if n_trials > 1 else 0.0
    return float(stats.norm.cdf((sr - sr0) * np.sqrt(len(x) - 1) / np.sqrt(1 - sk * sr + (ku - 1) / 4 * sr ** 2)))


def n_eff(X: np.ndarray) -> float:
    Z = (X - X.mean(0)) / np.maximum(X.std(0), 1e-12)
    ev = np.linalg.eigvalsh(Z @ Z.T if Z.shape[0] < Z.shape[1] else Z.T @ Z)
    ev = np.clip(ev, 0, None)
    return float(ev.sum() ** 2 / (ev ** 2).sum())


TRIALS_IS = np.concatenate([IS_RET_A.T, np.stack([RET[kn][IS_IDX] for kn in RET], 1)], 1)   # Stage A + Stage B
TRIALS_IS = TRIALS_IS[:, TRIALS_IS.std(0) > 1e-12]
N_TRIALS = N_TRIALS_A + len(RET)
N_EFF_ALL = n_eff(TRIALS_IS)
V_SR = float(np.var(TRIALS_IS.mean(0) / TRIALS_IS.std(0)))
dd_star = vm(r_D_is, r_S_is) - r_S_is
N_EFF_D, V_SR_D = n_eff(D_vs_S), float(np.var(D_vs_S.mean(0) / D_vs_S.std(0)))
DSR = {"absolute (N_eff)": dsr(r_D_is, N_EFF_ALL, V_SR), "absolute (all trials)": dsr(r_D_is, N_TRIALS, V_SR),
       "D* vs S* (N_eff)": dsr(dd_star, N_EFF_D, V_SR_D), "D* vs S* (all trials)": dsr(dd_star, len(dyn_keys), V_SR_D)}

# ── minimum detectable effect (one-sided 5 %, 80 % power) ──
RHO_DS = float(np.corrcoef(r_D_is, r_S_is)[0, 1])
N_OOS = len(OOS_IDX)
MDE = {nm: 2.486 * np.sqrt(252 * 2 * (1 - RHO_DS) / n) for nm, n in (("2008–2015", N_IS), ("holdout", N_OOS))}
NW_LAG = int(np.floor(4 * (N_OOS / 100) ** (2 / 9)))

# ── selection-aware placebo (2008–2015): re-run the whole D* choice with every selector's decisions
#    delayed by one shared random lag k (Cell 5) — each keeps its window mix, trade rate, persistence and
#    its correlation with the other 127; only the freshness of its information is destroyed. The null is
#    "a decision made today is no better than the same rule's decision from 1–12 months ago". ──
def max_q10_delta(ret_by_key: dict, C: np.ndarray) -> float:
    best = -np.inf
    for c in TOP_CONFIGS:
        k = c.key()
        X = np.stack([ret_by_key[kn][IS_IDX] for kn in dyn_keys if kn[0] == k], 1)
        diff = boot_sharpe(C, X) - boot_sharpe(C, RET[(k, "S*_asset")][IS_IDX][:, None])
        best = max(best, float(np.quantile(diff, 0.1, axis=0).max()))
    return best


N_SEL_PLACEBO = 50 if QUICK else 300
OBS_SEL = float(SEL.loc[0, "q10 ΔSR vs S*"])
rng_p = np.random.default_rng(SEED + 30)
SEL_PLACEBO = np.empty(N_SEL_PLACEBO)
t0 = time.perf_counter()
for i in range(N_SEL_PLACEBO):
    k = int(rng_p.integers(PLACEBO_LAG[0], PLACEBO_LAG[1] + 1))
    fake = {kn: returns_of(CB[kn[0]], calendar_lag(BANK[kn], k, VALID & IS_MASK[None, :]), UNI)
            for kn in dyn_keys}
    SEL_PLACEBO[i] = max_q10_delta(fake, C_IS)
P_SEL_PLACEBO = float((1 + (SEL_PLACEBO >= OBS_SEL).sum()) / (N_SEL_PLACEBO + 1))
T_SEL_PLACEBO = time.perf_counter() - t0

# ── pre-registration: one file per traded universe, written once, verified on every later run. Only the
#    discrete choices that decide the holdout verdict are hashed; the two frozen numbers (vol-matching k,
#    bootstrap block) are registered next to them and READ BACK on later runs, so float noise can neither
#    break the check nor move them; dates, data and code signatures are recorded alongside. ──
N_PLACEBO = 200 if QUICK else 2000
NW_RULE = "floor(4 * (N / 100) ** (2 / 9)), N = holdout sessions at test time"     # = NW_LAG; hashed as a rule
PLACEBO_DESC = (f"calendar lag (Cell 5), k ~ uniform{{{PLACEBO_LAG[0]}..{PLACEBO_LAG[1]}}} sessions per draw. "
                "A: D*'s own holdout trade sessions, each carrying the window D*'s ungated sibling selector chose "
                "k sessions earlier (never a decision made on or after the session); B: random eligible holdout "
                "sessions, same count per (asset, year), carrying those lagged windows. "
                f"{N_PLACEBO} draws each; statistic = holdout Sharpe; p = (1 + #{{placebo >= observed}}) / (draws + 1)")
PREREG_BODY = {
    "version": 3,
    "selection_period": [str(CAL[IS_IDX[0]].date()), str(CAL[IS_IDX[-1]].date())],
    "holdout_start": str(OOS_START.date()), "universe": UNI, "symbols": list(TRADE_SYMBOLS),
    "D_star": {"config": D_KEY, "config_label": D_CFG.label(), "selector": D_NAME},
    "S_star_asset": S_STAR[D_KEY]["asset"],
    "hypothesis": "one-sided H0: E[d_t] <= 0, d_t = k * r_D*(t) - r_S*(t), daily, every holdout session",
    "tests": {"primary": "Newey-West t of mean d_t, one-sided 5%", "newey_west_lags": [NW_RULE, 20],
              "secondary": "paired stationary bootstrap of SR(D*) - SR(S*), one-sided p and 90% CI",
              "robust_if": f"same sign of mean d_t before and after {IEX_SEAM.date()}"},
    "placebo": PLACEBO_DESC,
    "bootstrap": {"B": N_BOOT, "block": "stationary; mean length registered below", "seed": SEED},
    "quick": QUICK,
}
PREREG_PARAMS = {"vol_match_k": VOL_K, "bootstrap_block": round(BLOCK_L, 4)}   # registered, not hashed
# What happened before this file existed — kept in every registration so the record is honest.
PREREG_HISTORY = [
    "2026-10-16T13:27:01: development registration sha256 321cd354ce3ef16e… (same selection rules; written "
    "before the 2022-03-07 dividend-basis repair, with within-(asset, year) permutation placebos). Its holdout "
    "was evaluated during development: FAIL.",
    "Afterwards the permutation placebos were replaced by calendar-lag placebos (permuting decisions within a year "
    "leaks the session's own outcome, Cell 5) and the seam was repaired (Cell 2). Every registration since was "
    "therefore written after a holdout result had been seen: the holdout is no longer blind.",
]
PREREG_INFO = {"holdout_end_at_registration": str(CAL[OOS_IDX[-1]].date()), "nw_lag_at_registration": NW_LAG,
               "hashes": {"data": DATA_SIG, "engine": ENGINE_HASH, "model": MODEL_SIG, "rf_code": RF_HASH},
               "seam_factors": {s: round(f, 6) for s, f in SEAM_FACTOR.items()},
               "rf": {"params": RF_PARAMS, "gates": list(RF_GATES), "min_cover": MIN_COVER},
               "history": PREREG_HISTORY}
PREREG_HASH = hashlib.sha256(json.dumps(PREREG_BODY, sort_keys=True, default=str).encode()).hexdigest()
UNIV_TAG = hashlib.sha1(",".join(TRADE_SYMBOLS).encode()).hexdigest()[:8]
_q = "_quick" if QUICK else ""
prereg_path = PREREG_PATH.with_name(f"{PREREG_PATH.stem}_{UNIV_TAG}{_q}{PREREG_PATH.suffix}")
# other registrations: <stem>_<8 hex>[_quick][_superseded_YYYYMMDD].json, plus the legacy untagged <stem>.json;
# each is classed by the universe recorded inside it (other renamed files are ignored)
OTHER_REGS, SAME_UNIV_REGS = [], []
for p in sorted(PREREG_PATH.parent.glob(f"{PREREG_PATH.stem}*{PREREG_PATH.suffix}")):
    if p == prereg_path or not re.fullmatch(rf"{re.escape(PREREG_PATH.stem)}(_[0-9a-f]{{8}})?{_q}(_superseded_\d{{8}})?\.json", p.name):
        continue
    try:
        _syms = json.loads(p.read_text())["body"].get("symbols")
    except (OSError, ValueError, KeyError, TypeError):
        _syms = None
    (SAME_UNIV_REGS if _syms == list(TRADE_SYMBOLS) else OTHER_REGS).append(p.name)
if prereg_path.exists():
    reg = json.loads(prereg_path.read_text())
    if reg["sha256"] != PREREG_HASH:
        changed = [k for k in PREREG_BODY if json.dumps(PREREG_BODY[k], sort_keys=True, default=str)
                   != json.dumps(reg["body"].get(k), sort_keys=True, default=str)]
        _arch = prereg_path.with_name(f"{prereg_path.stem}_superseded_{pd.Timestamp.now():%Y%m%d}{prereg_path.suffix}")
        raise RuntimeError(f"The 2008–2015 selection no longer matches the pre-registration written "
                           f"{reg['written']} (changed: {changed}). To re-register deliberately, archive the old "
                           f"file (e.g. rename it to {_arch.name}) — keep it as a record — and re-run.")
    for nm, val in PREREG_PARAMS.items():
        frozen = float(reg.get("params", {}).get(nm, val))
        if abs(frozen - val) > 1e-3 * abs(frozen):
            warnings.warn(f"{nm}: this run computes {val:.6g}, the registration froze {frozen:.6g} — using the frozen value")
        PREREG_PARAMS[nm] = frozen
    PREREG_WRITTEN = reg["written"]
    print(f"Pre-registration verified: {prereg_path.name} · sha256 {PREREG_HASH[:16]}… · written {PREREG_WRITTEN} "
          f"· frozen k = {PREREG_PARAMS['vol_match_k']:.4f}, block = {PREREG_PARAMS['bootstrap_block']:.2f}")
else:
    PREREG_WRITTEN = pd.Timestamp.now().isoformat(timespec="seconds")
    prereg_path.write_text(json.dumps({"written": PREREG_WRITTEN, "sha256": PREREG_HASH, "body": PREREG_BODY,
                                       "params": PREREG_PARAMS, "info": PREREG_INFO}, indent=2, default=str))
    print(f"Pre-registration WRITTEN: {prereg_path.name} · sha256 {PREREG_HASH[:16]}… · universe {', '.join(TRADE_SYMBOLS)}")
    if SAME_UNIV_REGS:
        print(f"   An earlier registration for this universe exists ({', '.join(SAME_UNIV_REGS)}); see the history below.")
    if OTHER_REGS:
        print("!" * 100 + f"\n  Registrations for other universes already exist ({', '.join(OTHER_REGS)}).\n"
              "  Their holdouts overlap this one, so this registration is NOT blind — read Cell 9 as a re-test,\n"
              "  not a fresh pre-registered test.\n" + "!" * 100)
# Cells 9–12 use the registered values on every run, including the one that writes the file
VOL_K, BLOCK_L = PREREG_PARAMS["vol_match_k"], PREREG_PARAMS["bootstrap_block"]
print("   Registration history: " + " ".join(PREREG_HISTORY))

# ── report (all numbers 2008–2015) ──
print(f"\nS* (static benchmark) — per-asset window chosen by the 10th-percentile IS Sharpe rule:")
print(pd.DataFrame(S_STAR_TABLE).set_index("config").to_string())
print(f"\nTop dynamic variants by q10 of the paired bootstrap ΔSR vs S* (same spec), 2008–2015, "
      f"{len(SEL)} candidates:")
print(SEL.drop(columns="key").head(12).round(3).to_string())
print(f"\n→ D* = {CFG_TAG[D_KEY]} · {D_NAME}   ({D_CFG.label()})")
print(f"   vol-matching factor k = sd(S*)/sd(D*) = {VOL_K:.3f} (frozen) · IS correlation D*–S* ρ = {RHO_DS:.2f}")
print("   Choice under other block lengths: " + ", ".join(f"L={L}: {c} {n} (rank {r} under PW)" for L, (c, n, r)
                                                       in ALT_PICK.items()))
print(f"\nData-snooping tests on 2008–2015, {len(dyn_keys)} dynamic variants (vol-matched differentials):")
for nm, res in (("vs S* (own spec)", SPA_S), ("vs all-window ensemble", SPA_E)):
    print(f"   {nm:24s} SPA_c p = {res['SPA_c']:.3f} · SPA_l {res['SPA_l']:.3f} · SPA_u {res['SPA_u']:.3f} · "
          f"RC {res['RC']:.3f} · StepM rejects {res['StepM rejections']}")
print(f"   Primary (dynamic vs max(S*, ensemble)): p = {max(SPA_S['SPA_c'], SPA_E['SPA_c']):.3f}")
print("   SPA_c vs S* by block length: " + ", ".join(f"L={L}: {p:.3f}" for L, p in SPA_BLOCKS.items()))
print(f"   Selection-aware placebo ({N_SEL_PLACEBO} draws, {T_SEL_PLACEBO:.0f}s): delaying every selector's decisions "
      f"by a shared random {PLACEBO_LAG[0]}–{PLACEBO_LAG[1]} sessions and re-running the D* choice reaches the real "
      f"best q10 ΔSR {OBS_SEL:+.3f} with p = {P_SEL_PLACEBO:.3f} (placebo median {np.median(SEL_PLACEBO):+.3f})")
print("      → " + ("fresh decisions beat stale copies of the same rules (the timing carries information)"
                    if P_SEL_PLACEBO < 0.05 else
                    "fresh decisions do not beat stale copies of the same rules (no evidence the timing carries information)")
      + (f"; even the best still trails S* at q10 (ΔSR {OBS_SEL:+.3f})" if OBS_SEL < 0 else
         f"; the best leads S* at q10 (ΔSR {OBS_SEL:+.3f})"))
print(f"90 % model confidence set (unit-vol losses, {len(MCS_P)} models): {(MCS_P >= 0.10).sum()} survive — "
      f"2008–2015 cannot separate these Sharpe ratios"
      + ("" if (MCS_P >= 0.10).sum() < len(MCS_P) else " (none eliminated: the set is uninformative)"))
print("   S*_asset in the MCS: " + ", ".join(f"{CFG_TAG[c.key()]} {'yes' if MCS_P.get((CFG_TAG[c.key()], 'S*_asset'), 0) >= 0.10 else 'no'}"
                                          for c in TOP_CONFIGS)
      + f" · D* in the MCS: {'yes' if MCS_P[(CFG_TAG[D_KEY], D_NAME)] >= 0.10 else 'no'}")
print(f"\nDeflated Sharpe (trials: Stage A {N_TRIALS_A:,} + Stage B {len(RET)} = {N_TRIALS:,}; "
      f"N_eff = {N_EFF_ALL:.1f}; differential N_eff = {N_EFF_D:.1f}):")
for nm, v in DSR.items():
    print(f"   {nm:24s} {v:.3f}")
print("   (absolute DSRs are upper bounds: the Static ORB project already mined 2008–2015)")
print(f"\nMinimum detectable ΔSharpe (one-sided 5 %, 80 % power, ρ = {RHO_DS:.2f}): "
      f"2008–2015 {MDE['2008–2015']:.2f} · holdout ({N_OOS:,} sessions) {MDE['holdout']:.2f}. "
      f"A non-rejection rules out only gains larger than this.")
print("\nBest variation of any family by absolute q10 Sharpe, 2008–2015 (descriptive; no inference):")
print(Q10_ALL.sort_values(ascending=False).head(10).round(3).to_frame("q10 SR").to_string())

Series built: 164 portfolio variants + 280 per-asset/class statics (1s)
Stationary bootstrap: B = 2,000 resamples of the 2,015 sessions of 2008–2015, mean block 1.3 sessions (median Politis–White length over the 128 dynamic-minus-ensemble differentials; range 1.0–3.3); robustness blocks (10, 20, 60)


Pre-registration verified: preregistration_71316e0b.json · sha256 22d4360dfd25db3a… · written 2026-10-16T17:19:44 · frozen k = 1.2265, block = 1.25
   Registration history: 2026-10-16T13:27:01: development registration sha256 321cd354ce3ef16e… (same selection rules; written before the 2022-03-07 dividend-basis repair, with within-(asset, year) permutation placebos). Its holdout was evaluated during development: FAIL. Afterwards the permutation placebos were replaced by calendar-lag placebos (permuting decisions within a year leaks the session's own outcome, Cell 5) and the seam was repaired (Cell 2). Every registration since was therefore written after a holdout result had been seen: the holdout is no longer blind.

S* (static benchmark) — per-asset window chosen by the 10th-percentile IS Sharpe rule:
        DIA  GLD  IEF  IWM  QQQ  SHY  SLV  SPY  TLT  USO  common
config                                                          
C1       10   30   15   15   15   15   15   10   15   15 

## Cell 9 — Holdout test (2016+)

In [9]:
# ════════════════════════════════════════════════════════════════════════════
#  Holdout 2016-01-01 → last session: the pre-registered test of D* against S*, run once.
#    d_t = k·r_D*(t) − r_S*(t)   (k frozen in Cell 8 so both legs carry the same 2008–2015 volatility)
#    primary:   one-sided Newey–West t of mean d_t;  secondary: paired stationary bootstrap of ΔSharpe
#  Then what the switching and the gating each contribute (2×2), the controls, two placebos that
#  destroy the timing of D*'s choices, a days-only placebo for the gate (descriptive), the development
#  registration's leaky permutation method re-applied for comparison, how much of the hindsight-best
#  switch D* captures, and charts.
#  Every number below this line uses 2016+ returns; nothing here feeds back into a choice.
# ════════════════════════════════════════════════════════════════════════════
reg = json.loads(prereg_path.read_text())
assert reg["sha256"] == PREREG_HASH and reg["body"]["D_star"] == PREREG_BODY["D_star"], "pre-registration mismatch"
assert NW_LAG == int(np.floor(4 * (N_OOS / 100) ** (2 / 9)))      # the registered rule (Cell 8)
cbD, uniD = CB[D_KEY], CB[D_KEY]["uni"][UNI]
pick_D, pick_S = BANK[(D_KEY, D_NAME)], BANK[(D_KEY, "S*_asset")]
r_D, r_S = RET[(D_KEY, D_NAME)], RET[(D_KEY, "S*_asset")]
d_all = VOL_K * r_D - r_S
PRE_SEAM, POST_SEAM = OOS_IDX[CAL[OOS_IDX] < IEX_SEAM], OOS_IDX[CAL[OOS_IDX] >= IEX_SEAM]


def nw(x: np.ndarray, lags: int) -> tuple:
    """(annualised mean, Newey–West t, one-sided p for H0: E[x] ≤ 0)."""
    res = sm.OLS(np.asarray(x, float), np.ones(len(x))).fit(cov_type="HAC", cov_kwds={"maxlags": lags})
    t = float(res.tvalues[0])
    return 252 * float(np.mean(x)), t, float(stats.norm.sf(t))


def paired_dsr(C: np.ndarray, a: np.ndarray, b: np.ndarray) -> np.ndarray:
    sb = boot_sharpe(C, np.stack([a, b], 1))
    return sb[:, 0] - sb[:, 1]


# ── 1. the pre-registered test ──
C_OOS = sb_counts(N_OOS, N_BOOT, BLOCK_L, SEED + 40)
x_D, x_S, x_d = r_D[OOS_IDX], r_S[OOS_IDX], d_all[OOS_IDX]
DSR_OBS = sharpe(x_D) - sharpe(x_S)
bd = paired_dsr(C_OOS, x_D, x_S)
HOLDOUT = {"mean d_t (ann. %)": 100 * nw(x_d, NW_LAG)[0],
           **{f"NW t (lag {L})": nw(x_d, L)[1] for L in (NW_LAG, 20)},
           **{f"NW p one-sided (lag {L})": nw(x_d, L)[2] for L in (NW_LAG, 20)},
           "SR D*": sharpe(x_D), "SR S*": sharpe(x_S), "ΔSR": DSR_OBS,
           "bootstrap p one-sided": float((1 + ((bd - DSR_OBS) >= DSR_OBS).sum()) / (N_BOOT + 1)),
           "ΔSR 90% CI low": float(np.quantile(bd, 0.05)), "ΔSR 90% CI high": float(np.quantile(bd, 0.95))}
ERA = pd.DataFrame([{"era": nm, "sessions": len(ix), "mean d_t (ann. %)": 100 * nw(d_all[ix], NW_LAG)[0],
                     "NW t": nw(d_all[ix], NW_LAG)[1], "SR D*": sharpe(r_D[ix]), "SR S*": sharpe(r_S[ix])}
                    for nm, ix in ((f"2016 → {(IEX_SEAM - pd.Timedelta(days=1)).date()}", PRE_SEAM),
                                   (f"{IEX_SEAM.date()} → {CAL[-1].date()}", POST_SEAM))]).set_index("era")
SAME_SIGN = bool(np.sign(ERA["mean d_t (ann. %)"]).nunique() == 1)
PASS = HOLDOUT[f"NW p one-sided (lag {NW_LAG})"] < 0.05 and HOLDOUT["mean d_t (ann. %)"] > 0 and SAME_SIGN
if D_NAME.startswith("rf_"):                                   # sessions the forest itself scored
    cov = coverage(cbD)
    d_cov = VOL_K * returns_of(cbD, np.where(cov, pick_D, -1), UNI) - returns_of(cbD, np.where(cov, pick_S, -1), UNI)
    COVERED = nw(d_cov[OOS_IDX], NW_LAG)
else:
    COVERED = None


# ── 2. decomposition: {switch, fixed S*} × {gated, ungated}, plus controls (descriptive, not pre-registered) ──
def sibling(name: str, gate: str) -> str:
    return name.rsplit("_", 1)[0] + "_" + gate


GATED_NAME = D_NAME if not D_NAME.endswith("_always") else sibling(D_NAME, "0" if D_NAME.startswith("rf_") else "pos")
UNGATED_NAME = sibling(D_NAME, "always")
GATE_ON = BANK[(D_KEY, GATED_NAME)] >= 0
W0_pick = np.where(VALID, cbD["W0"]["asset"], -1)
BEST_ANY = Q10_ALL.idxmax()                                    # (config label, selector)
BEST_ANY_KEY = next(kn for kn in RET if (CFG_TAG[kn[0]], kn[1]) == BEST_ANY)
ARMS = {
    f"switch + gate ({GATED_NAME})": RET[(D_KEY, GATED_NAME)],
    f"switch, no gate ({UNGATED_NAME})": RET[(D_KEY, UNGATED_NAME)],
    "fixed S* + D*'s gate": returns_of(cbD, np.where(GATE_ON, pick_S, -1), UNI),
    "fixed S*, no gate (= S*)": r_S,
    "expanding-best W0 + D*'s gate": returns_of(cbD, np.where(GATE_ON, W0_pick, -1), UNI),
    "all-window ensemble + D*'s gate": returns_of(cbD, ENSEMBLE_W * GATE_ON[..., None], UNI),
    "all-window ensemble": RET[(D_KEY, "ensemble")],
    "slow switching control (wf_static3)": RET[(D_KEY, "wf_static3")],
    "S*_class": RET[(D_KEY, "S*_class")], "S*_common": RET[(D_KEY, "S*_common")],
    f"best of any family by IS q10 ({BEST_ANY[0]} {BEST_ANY[1]})": RET[BEST_ANY_KEY],
}
rows = []
for nm, r in ARMS.items():
    k_arm = r_S[IS_IDX].std() / r[IS_IDX].std() if r[IS_IDX].std() > 0 else 0.0
    sh, cagr, mdd = summarize(r, OOS_MASK)
    m, t, _ = nw(k_arm * r[OOS_IDX] - x_S, NW_LAG)
    rows.append({"variant": nm, "holdout Sharpe": sh, "CAGR %": 100 * cagr, "MDD %": 100 * mdd,
                 "active days %": 100 * (r[OOS_IDX] != 0).mean(), "vs S*: mean d (ann. %)": 100 * m, "NW t": t})
ARMS_TABLE = pd.DataFrame(rows).set_index("variant")

# ── 3. placebos (no look-ahead, Cell 5): D*'s own days (not flat) carrying the window its ungated selector
#    chose k sessions earlier (A — is today's window choice better than a stale one?) · random eligible
#    days, same count per asset-year, carrying those stale windows (B — adds: are D*'s days better?) ──
um_oos = uniD & OOS_MASK[None, :]
pick_U = BANK[(D_KEY, UNGATED_NAME)]
assert np.all((pick_U == pick_D) | (pick_D < 0)), "the gated selector must trade its ungated sibling's window"
rngA, rngB = np.random.default_rng(SEED + 50), np.random.default_rng(SEED + 51)
lag = lambda g: int(g.integers(PLACEBO_LAG[0], PLACEBO_LAG[1] + 1))
t0 = time.perf_counter()
PL_A = np.array([sharpe(returns_of(cbD, placebo_A(pick_D, pick_U, um_oos, lag(rngA)), UNI)[OOS_IDX])
                 for _ in range(N_PLACEBO)])
PL_B = np.array([sharpe(returns_of(cbD, placebo_B(pick_D, pick_U, um_oos, rngB, lag(rngB)), UNI)[OOS_IDX])
                 for _ in range(N_PLACEBO)])
P_PL = {nm: float((1 + (pl >= sharpe(x_D)).sum()) / (N_PLACEBO + 1)) for nm, pl in (("A", PL_A), ("B", PL_B))}
T_PL = time.perf_counter() - t0


# ── 3a. days-only placebo for the gate (descriptive, NOT registered): does D*'s gate pick better days for a
#    FIXED window? Random eligible days, same count per asset-year as the gate's holdout days, each trading
#    S*'s window — against the "fixed S* + D*'s gate" arm. (Placebo B moves the days AND stales the window.)
#    A class-scope gate opens every asset of a class on the same sessions, so the days are drawn jointly
#    within each (class, year); drawing each asset independently would add diversification the gate never had. ──
def placebo_days(pick: np.ndarray, fixed: np.ndarray, universe_mask: np.ndarray, rng: np.random.Generator,
                 group_of: np.ndarray) -> np.ndarray:
    """Random eligible days, same count per asset-year as `pick`, each trading `fixed`'s window: one random order
    of the year's sessions per (group, year), each asset taking its first n eligible sessions in that order."""
    out = np.full_like(pick, -1)
    on = (pick >= 0) & universe_mask
    for yi in np.unique(YI[on.any(0)]):
        cols = np.flatnonzero(YI == yi)
        for gr in np.unique(group_of):
            order = cols[rng.permutation(len(cols))]
            for a in np.flatnonzero(group_of == gr):
                n = int(on[a, cols].sum())
                if n:
                    days = order[universe_mask[a, order]][:n]
                    out[a, days] = fixed[a, days]
    return out


assert np.all(pick_S[um_oos] >= 0), "S* must have a window on every eligible holdout cell"
GATE_PICK = np.where(GATE_ON, pick_S, -1)
GATE_SR, S_SR = sharpe(ARMS["fixed S* + D*'s gate"][OOS_IDX]), sharpe(x_S)
GATE_SCOPE = "class" if "_class_" in GATED_NAME else "asset"
_grp = CLASS_OF if GATE_SCOPE == "class" else np.arange(A)
rngD = np.random.default_rng(SEED + 54)
PL_D = np.array([sharpe(returns_of(cbD, placebo_days(GATE_PICK, pick_S, um_oos, rngD, _grp), UNI)[OOS_IDX])
                 for _ in range(N_PLACEBO)])
P_PL["D"] = float((1 + (PL_D >= GATE_SR).sum()) / (N_PLACEBO + 1))
GATE_DAYS_SHARE = float(GATE_ON[um_oos].mean())


# ── 3b. for comparison only: the development registration's permutation method (Cell 8 history),
#    re-applied to this run's D* — D*'s windows permuted within (asset, year). It leaks: a window chosen
#    later in the year was chosen with trailing statistics that already contain the session it is moved
#    onto, which helps the placebo, so these p-values are biased upward (against D*). Not this run's test. ──
def perm_A(pick: np.ndarray, universe_mask: np.ndarray, rng: np.random.Generator) -> np.ndarray:
    ai, ti, g = _groups((pick >= 0) & universe_mask)
    perm = np.lexsort((rng.random(len(g)), g))
    out = np.full_like(pick, -1)
    out[ai, ti] = pick[ai[perm], ti[perm]]
    return out


def perm_B(pick: np.ndarray, universe_mask: np.ndarray, rng: np.random.Generator) -> np.ndarray:
    out = np.full_like(pick, -1)
    ai, ti, g = _groups((pick >= 0) & universe_mask)
    ea, et, eg = _groups(universe_mask)
    ug, cnt = np.unique(g, return_counts=True)
    starts, ends = np.searchsorted(eg, ug), np.searchsorted(eg, ug, side="right")
    ws, gs = pick[ai, ti], np.r_[0, np.cumsum(cnt)]
    for j, (s0, e0) in enumerate(zip(starts, ends)):
        days = rng.choice(np.arange(s0, e0), size=cnt[j], replace=False)
        out[ea[days], et[days]] = rng.permutation(ws[gs[j]:gs[j + 1]])
    return out


rngC, rngE = np.random.default_rng(SEED + 52), np.random.default_rng(SEED + 53)
PL_A0 = np.array([sharpe(returns_of(cbD, perm_A(pick_D, um_oos, rngC), UNI)[OOS_IDX]) for _ in range(N_PLACEBO)])
PL_B0 = np.array([sharpe(returns_of(cbD, perm_B(pick_D, um_oos, rngE), UNI)[OOS_IDX]) for _ in range(N_PLACEBO)])
P_PL0 = {nm: float((1 + (pl >= sharpe(x_D)).sum()) / (N_PLACEBO + 1)) for nm, pl in (("A", PL_A0), ("B", PL_B0))}

# ── 4. oracles, in u units (RISK per session): how much of the hindsight-best switch D* captures ──
u = np.nan_to_num(cbD["u"])
ai, ti = np.nonzero((pick_D >= 0) & um_oos)
uD, uS, uO = u[ai, ti, pick_D[ai, ti]], u[ai, ti, pick_S[ai, ti]], u[ai, ti].max(1)
ORACLE = {"D* trade sessions": len(ai), "mean u D*": uD.mean(), "mean u S* (same sessions)": uS.mean(),
          "mean u oracle switch": uO.mean(),
          "capture share": (uD.mean() - uS.mean()) / (uO.mean() - uS.mean()) if uO.mean() > uS.mean() else np.nan,
          "oracle_year Sharpe (ceiling)": summarize(returns_of(cbD, oracle_year(cbD), UNI), OOS_MASK)[0],
          "oracle_session Sharpe (footnote)": summarize(returns_of(cbD, oracle_session(cbD), UNI), OOS_MASK)[0]}

# ── report ──
print(f"Pre-registration {prereg_path.name} (written {PREREG_WRITTEN}) verified.  D* = {CFG_TAG[D_KEY]} {D_NAME}  "
      f"vs  S* = per-asset static windows {S_STAR[D_KEY]['asset']}")
print(f"Holdout {CAL[OOS_IDX[0]].date()} → {CAL[OOS_IDX[-1]].date()} ({N_OOS:,} sessions), k = {VOL_K:.3f}, "
      f"minimum detectable ΔSR ≈ {MDE['holdout']:.2f}")
print("Caveat — not a blind test: " + " ".join(PREREG_HISTORY) + "\n")
print(pd.Series(HOLDOUT).round(4).to_string())
print(f"\nBy data era (the result must have the same sign in both):")
print(ERA.round(3).to_string())
if COVERED is not None:
    print(f"\nRF-covered (asset, session) cells only: mean d_t {100 * COVERED[0]:+.2f} %/yr · NW t {COVERED[1]:+.2f}")
verdict = ("PASS — D* beats the static benchmark out of sample" if PASS else
           "FAIL — no evidence that switching windows beats the per-asset static window out of sample")
print(f"\nPre-registered verdict: {verdict} (one-sided NW p = {HOLDOUT[f'NW p one-sided (lag {NW_LAG})']:.3f}, "
      f"same sign in both eras: {SAME_SIGN}).")
if not PASS:
    print(f"   Non-rejection only rules out holdout ΔSharpe gains larger than ≈ {MDE['holdout']:.2f}.")
print(f"\nWhat switching and gating each contribute, holdout (descriptive; k re-matched per row on 2008–2015):")
print(ARMS_TABLE.round(3).to_string())
print(f"\nRegistered placebos ({N_PLACEBO:,} draws each, {T_PL:.0f}s) — holdout Sharpe of D* {sharpe(x_D):.3f}:")
print(f"   A  D*'s own days (not flat), window its ungated selector chose {PLACEBO_LAG[0]}–{PLACEBO_LAG[1]} sessions earlier: "
      f"median {np.median(PL_A):.3f}, p = {P_PL['A']:.4f}")
print(f"   B  random eligible days (same count per asset-year), same stale windows: "
      f"median {np.median(PL_B):.3f}, p = {P_PL['B']:.4f}")
_gate_lbl = f"D*'s gate" if GATED_NAME == D_NAME else f"the gate of D*'s gated sibling {GATED_NAME} (D* is ungated)"
print(f"   D  (descriptive, not registered) S*'s fixed window on the days {_gate_lbl} trades ({GATE_DAYS_SHARE:.1%} of "
      f"eligible holdout cells): holdout Sharpe {GATE_SR:.3f} (S* on every eligible day: {S_SR:.3f}) · on random eligible "
      f"days instead (same count per asset-year, drawn jointly per {GATE_SCOPE}-year as the gate's scope): median "
      f"{np.median(PL_D):.3f}, p = {P_PL['D']:.4f}")
print(f"   For comparison only — the development registration's permutation method, re-applied to this run's D* "
      f"(leaky; biased against D*): A within-year shuffle median {np.median(PL_A0):.3f}, p = {P_PL0['A']:.4f} · "
      f"B random days + D*'s window mix median {np.median(PL_B0):.3f}, p = {P_PL0['B']:.4f}")
print(f"\nOracles (u = position weight × R per session, holdout, D*'s trade sessions):")
print(pd.Series(ORACLE).round(4).to_string())

# ── charts: equity and drawdown 2008+ (IS shaded), and D*'s window mix by year ──
show = {f"D* ({D_NAME})": r_D, "S* (per-asset static)": r_S, "all-window ensemble": RET[(D_KEY, "ensemble")],
        f"best any ({BEST_ANY[1]})": RET[BEST_ANY_KEY], f"{BENCHMARK} buy & hold": R_SPY}
ev = CAL >= pd.Timestamp(f"{WF_FIRST_YEAR}-01-01")
fig = make_subplots(rows=2, cols=1, shared_xaxes=True, row_heights=[0.68, 0.32], vertical_spacing=0.04)
for nm, r in show.items():
    g = np.cumprod(1 + r[ev])
    fig.add_trace(go.Scatter(x=CAL[ev], y=g, name=nm, mode="lines"), 1, 1)
    fig.add_trace(go.Scatter(x=CAL[ev], y=100 * (g / np.maximum.accumulate(g) - 1), name=nm, mode="lines",
                             showlegend=False, line=dict(width=1)), 2, 1)
for rr in (1, 2):
    fig.add_vrect(x0=CAL[IS_IDX[0]], x1=CAL[IS_IDX[-1]], fillcolor="grey", opacity=0.12, line_width=0, row=rr, col=1)
fig.update_yaxes(type="log", title="growth of $1", row=1, col=1)
fig.update_yaxes(title="drawdown %", row=2, col=1)
fig.update_layout(height=650, title=f"D* vs static benchmark vs {BENCHMARK} — strategies at {RISK:.2%} risk per trade, "
                                    f"shaded = 2008–2015 selection period", hovermode="x unified")
fig.show()

yrs = YEARS[ev]
mix = pd.DataFrame({f"{w}m": [((pick_D[:, ev] == i) & uniD[:, ev])[:, yrs == y].sum() for y in np.unique(yrs)]
                    for i, w in enumerate(OR_WINDOWS)}, index=np.unique(yrs))
mix["flat"] = [((pick_D[:, ev] < 0) & uniD[:, ev] & VALID[:, ev])[:, yrs == y].sum() for y in np.unique(yrs)]
mix = mix.div(mix.sum(1), axis=0)
fig = go.Figure([go.Bar(x=mix.index, y=100 * mix[c], name=c) for c in mix.columns])
fig.update_layout(barmode="stack", height=380, yaxis_title="% of tradable asset-sessions",
                  title=f"D*'s window choice by year ({D_NAME})")
fig.show()

Pre-registration preregistration_71316e0b.json (written 2026-10-16T17:19:44) verified.  D* = C1 ts_inc_class_h250_k2_pos  vs  S* = per-asset static windows {'DIA': 10, 'GLD': 30, 'IEF': 15, 'IWM': 15, 'QQQ': 15, 'SHY': 15, 'SLV': 15, 'SPY': 10, 'TLT': 15, 'USO': 15}
Holdout 2016-01-04 → 2026-09-21 (2,694 sessions), k = 1.226, minimum detectable ΔSR ≈ 0.49
Caveat — not a blind test: 2026-10-16T13:27:01: development registration sha256 321cd354ce3ef16e… (same selection rules; written before the 2022-03-07 dividend-basis repair, with within-(asset, year) permutation placebos). Its holdout was evaluated during development: FAIL. Afterwards the permutation placebos were replaced by calendar-lag placebos (permuting decisions within a year leaks the session's own outcome, Cell 5) and the seam was repaired (Cell 2). Every registration since was therefore written after a holdout result had been seen: the holdout is no longer blind.

mean d_t (ann. %)          2.0549
NW t (lag 8)               1

## Cell 10 — Robustness

In [10]:
# ════════════════════════════════════════════════════════════════════════════
#  Robustness of the pre-registered result — descriptive; nothing here changes D*.
#   1. Walk-forward meta-selection: the WHOLE Cell 8 rule (S*_asset by the 10th-percentile
#      Sharpe, D* by the 10th percentile of the paired ΔSR) is re-run every January from 2012
#      on 2008 … y−1 only, traded in year y, and the years are chained. The gross cap resets
#      every session, so chaining different selectors year by year is legitimate. 2012–2015
#      are NOT clean (the four Stage-A finalists were chosen on 2008–2015); from 2016 every
#      choice in the pipeline is ex-ante. A second run starts training in 2009 (drops 2008).
#   2. Execution sensitivities on the frozen D* and S*: bar-close entries filled at the NEXT
#      bar's open, round-trip costs doubled, and prices with the dividend adjustment undone
#      (calibrated to the seam factors, Cell 3). Every pick and k stay fixed; only the fills, the
#      costs and (dividend case) the price level each R is measured on change. The dividend case
#      runs twice: on the frozen universe, and with the cost-based universe re-derived.
#   3. Seed stability: Cell 8's rule re-run on 2008–2015 with 20 other bootstrap seeds. Where
#      candidates sit within Monte Carlo noise of each other, the choice is a coin toss.
#   4. Corrected price basis: the split-only engine overstates fixed per-share costs in R before
#      the end of the file — unevenly across assets, years and stop widths — so it also shaped
#      Cells 6–8's choices. Stage A, the forests, S* and D* are re-made with the same rules on prices
#      with the estimated dividend adjustment undone, and tested like Cell 9. Not pre-registered: it is
#      what the same rules choose on those prices. (Slow on a cold cache: one more Stage A, and one more
#      forest run per finalist.)
# ════════════════════════════════════════════════════════════════════════════
MS_FIRST = 2012
B_MS = 200 if QUICK else 1000
MS_YEARS = [int(y) for y in YEAR_LIST if y >= MS_FIRST]


def s_star_on(k: str, tr: np.ndarray, C: np.ndarray, W: dict) -> tuple:
    """Cell 8's S*_asset rule for one finalist on the sessions tr → (window index per asset, r_S*)."""
    q10s = lambda X: np.quantile(boot_sharpe(C, X), 0.10, axis=0)
    common = OR_WINDOWS[int(np.argmax(q10s(np.stack([W["RET"][(k, f"static_{w}")][tr] for w in OR_WINDOWS], 1))))]
    wa = np.full(A, WIDX[common])
    for a in range(A):
        Xa = np.stack([W["ASSET_RET"][(k, a, w)][tr] for w in OR_WINDOWS], 1)
        if (Xa != 0).sum(0).min() >= 100:
            wa[a] = int(np.argmax(q10s(Xa)))
    return wa, returns_of(W["CB"][k], np.where(VALID, wa[:, None], -1), UNI)


def select_on(tr: np.ndarray, C: np.ndarray, W: dict = None, ranks: list = None) -> tuple:
    """Cell 8's rule on the sessions tr with bootstrap counts C → (q10 ΔSR, config key, selector, S* windows, r_S*).
    W = the finalists and their series (default: Cells 6–8's); ranks, if given, collects (q10 ΔSR, key, selector)
    of every candidate."""
    W = WORLD if W is None else W
    best = None
    for c in W["configs"]:
        k = c.key()
        wa, r_s = s_star_on(k, tr, C, W)
        names = [nm for (kk, nm) in W["dyn"] if kk == k]
        X = np.stack([W["RET"][(k, nm)][tr] for nm in names], 1)
        diff = np.quantile(boot_sharpe(C, X) - boot_sharpe(C, r_s[tr][:, None]), 0.10, axis=0)
        j = int(np.argmax(diff))
        if ranks is not None:
            ranks.extend((float(q), k, nm) for q, nm in zip(diff, names))
        if best is None or diff[j] > best[0]:
            best = (float(diff[j]), k, names[j], wa, r_s)
    return best


WORLD = dict(configs=TOP_CONFIGS, CB=CB, RET=RET, ASSET_RET=ASSET_RET, dyn=dyn_keys)


def wf_meta(first_train: int) -> tuple:
    """Yearly re-selection → chained daily returns of the chosen D and S*, their vol-matched
    differential d_t = k_y·r_D − r_S*, and one row per traded year."""
    rD, rS, dd = np.zeros(T), np.zeros(T), np.zeros(T)
    rows = []
    for y in MS_YEARS:
        tr = np.flatnonzero((YEARS >= first_train) & (YEARS <= y - 1))
        te = YEARS == y
        C = sb_counts(len(tr), B_MS, BLOCK_L, SEED + 60 + y + 100 * (first_train - WF_FIRST_YEAR))
        q, k, nm, wa, r_s = select_on(tr, C)
        r_d = RET[(k, nm)]
        k_y = r_s[tr].std() / r_d[tr].std() if r_d[tr].std() > 1e-12 else 0.0
        rD[te], rS[te], dd[te] = r_d[te], r_s[te], k_y * r_d[te] - r_s[te]
        rows.append({"year": y, "trained on": f"{first_train}–{y - 1}", "config": CFG_TAG[k], "selector": nm,
                     "q10 ΔSR": q, "k": k_y, "= frozen D*": (k, nm) == (D_KEY, D_NAME),
                     "S* windows": " ".join(f"{TRADE_SYMBOLS[a]}{OR_WINDOWS[wa[a]]}" for a in range(A)),
                     "SR D": sharpe(r_d[te]), "SR S*": sharpe(r_s[te]), "d (ann. %)": 100 * 252 * dd[te].mean()})
    return rD, rS, dd, pd.DataFrame(rows).set_index("year")


t0 = time.perf_counter()
WF_META = {start: wf_meta(start) for start in (WF_FIRST_YEAR, WF_FIRST_YEAR + 1)}
T_META = time.perf_counter() - t0
ERAS_MS = {f"{MS_FIRST}–{OOS_START.year - 1} (not clean)": (YEARS >= MS_FIRST) & (YEARS < OOS_START.year),
           f"{OOS_START.year}+ (ex-ante)": YEARS >= OOS_START.year,
           f"{MS_FIRST}+": YEARS >= MS_FIRST}
rows = []
for start, (rD, rS, dd, _) in WF_META.items():
    for era, m in ERAS_MS.items():
        ix = np.flatnonzero(m)
        mu, t, p = nw(dd[ix], NW_LAG)
        rows.append({"training starts": start, "era": era, "sessions": len(ix), "SR D (re-chosen)": sharpe(rD[ix]),
                     "SR S* (re-chosen)": sharpe(rS[ix]), "mean d (ann. %)": 100 * mu, "NW t": t, "p one-sided": p})
META_TABLE = pd.DataFrame(rows).set_index(["training starts", "era"])

# ── 3. seed stability of the frozen choice (same sessions, same rule, other bootstrap draws) ──
N_SEEDS = 5 if QUICK else 20
_chk = select_on(IS_IDX, C_IS)
assert (_chk[1], _chk[2]) == (D_KEY, D_NAME) and np.array_equal(np.where(VALID, _chk[3][:, None], -1), pick_S), \
    "select_on must reproduce Cell 8's frozen D* and S* with Cell 8's own bootstrap draws"
STAB = []
for i in range(N_SEEDS):
    q, k, nm, wa, r_s = select_on(IS_IDX, sb_counts(N_IS, N_BOOT, BLOCK_L, SEED + 1000 + i))
    STAB.append({"config": CFG_TAG[k], "selector": nm,
                 "S* = frozen": bool(np.all(np.where(VALID, wa[:, None], -1) == pick_S)) if k == D_KEY else False})
STAB = pd.DataFrame(STAB)
STAB_COUNTS = STAB.groupby(["config", "selector"]).size().sort_values(ascending=False)


# ── 2. execution sensitivities (frozen decisions) ──
def config_R(cfg: Config, wo: dict) -> np.ndarray:
    """R tensor of a config from any outcome set — the same filters build_config applies."""
    keep = (np.abs(np.nan_to_num(GAP_G))[..., None] >= cfg.gap_min) & ~(ORATR_G > cfg.or_atr_max)
    return np.where(keep & VALID[..., None], wo["R"], np.nan).astype(np.float32)


t1 = time.perf_counter()
SENS_SETS = {"baseline (pre-registered)": (cbD["wo"], cbD["R"])}
if D_CFG.spec.confirm == "close":
    wo_no = window_outcomes(D_CFG.spec, "split", "next_open")
    SENS_SETS["bar-close entries fill at the next bar's open"] = (wo_no, config_R(D_CFG, wo_no))
SENS_SETS["round-trip costs doubled"] = (cbD["wo"], cbD["R"] - cbD["wo"]["cost"])
wo_div = window_outcomes(D_CFG.spec, "split+div")
DIV_FIXED, DIV_UNI = "dividend adjustment undone (seam-calibrated)", "dividend adjustment undone, universe re-derived"
SENS_SETS[DIV_FIXED] = SENS_SETS[DIV_UNI] = (wo_div, config_R(D_CFG, wo_div))
UNI_DIV = universes(tradable_mask(wo_div["cost"]))[UNI]
_cb_div = {**cbD, "uni": {UNI: UNI_DIV}}
_share = lambda U, ix: 100 * U[:, ix].sum(1) / np.maximum(VALID[:, ix].sum(1), 1)
DIV_UNI_TABLE = pd.DataFrame({"tradable % IS (split only)": _share(cbD["uni"][UNI], IS_IDX),
                              "tradable % IS (dividends undone)": _share(UNI_DIV, IS_IDX),
                              "tradable % holdout (split only)": _share(cbD["uni"][UNI], OOS_IDX),
                              "tradable % holdout (dividends undone)": _share(UNI_DIV, OOS_IDX)}, index=TRADE_SYMBOLS)
NEVER_DIV = [s for a, s in enumerate(TRADE_SYMBOLS) if not UNI_DIV[a].any()]
wD, wS = as_weight(cbD, pick_D, UNI), as_weight(cbD, pick_S, UNI)
SENS_W = {DIV_UNI: (as_weight(_cb_div, pick_D, UNI), as_weight(_cb_div, pick_S, UNI))}
rows, SENS_RET = [], {}
for nm, (wo, R) in SENS_SETS.items():
    wd, ws = SENS_W.get(nm, (wD, wS))
    rd, rs = grid_returns(wo, wd, R=R), grid_returns(wo, ws, R=R)
    SENS_RET[nm] = (rd, rs)
    mu, t, p = nw(VOL_K * rd[OOS_IDX] - rs[OOS_IDX], NW_LAG)
    same = len({np.sign((VOL_K * rd - rs)[ix].mean()) for ix in (PRE_SEAM, POST_SEAM)}) == 1
    rows.append({"variant": nm, "IS SR D*": sharpe(rd[IS_IDX]), "IS SR S*": sharpe(rs[IS_IDX]),
                 "holdout SR D*": sharpe(rd[OOS_IDX]), "holdout SR S*": sharpe(rs[OOS_IDX]),
                 "holdout ΔSR": sharpe(rd[OOS_IDX]) - sharpe(rs[OOS_IDX]), "mean d (ann. %)": 100 * mu,
                 "NW t": t, "p one-sided": p, "passes": bool(p < 0.05 and mu > 0 and same)})
SENS_TABLE = pd.DataFrame(rows).set_index("variant")
assert SENS_TABLE.iloc[0]["passes"] == PASS, "the baseline row must reproduce Cell 9's verdict"
assert np.allclose(SENS_RET["baseline (pre-registered)"][0], r_D) and np.allclose(SENS_RET["baseline (pre-registered)"][1], r_S)
T_SENS = time.perf_counter() - t1

# ── 4. the whole selection re-made on dividend-corrected prices: Stage A → forests → Cell 8's rule ──
t2 = time.perf_counter()
_tab_div, _isr_div, _ = stage_a("split+div")
TOP_DIV = [CFG_BY_KEY[k] for k in choose_configs(_tab_div, _isr_div)[0]]
CB_DIV, PRED_DIV, RET_DIV, ASSET_DIV = {}, {}, {}, {}
for c in TOP_DIV:
    CB_DIV[c.key()], _X, PRED_DIV[c.key()], _ = rf_config(c, "split+div")
    del _X                                                        # the design matrices are large and not needed again
    _, _ret, _aret, _ = series_of(CB_DIV[c.key()], PRED_DIV[c.key()])
    CB_DIV[c.key()] = {kk: CB_DIV[c.key()][kk] for kk in ("cfg", "scale", "wo", "R", "uni")}   # all returns_of reads
    RET_DIV.update(_ret)
    ASSET_DIV.update(_aret)
del _tab_div, _isr_div
DYN_DIV = [kn for kn in RET_DIV if kn[1].startswith(DYN_PREFIX)]
BLOCK_DIV = float(np.median([pw_block(RET_DIV[kn][IS_IDX] - RET_DIV[(kn[0], "ensemble")][IS_IDX]) for kn in DYN_DIV]))
C_DIV = sb_counts(N_IS, N_BOOT, BLOCK_DIV, SEED + 10)               # Cell 8's seed: only the prices differ
WORLD_DIV = dict(configs=TOP_DIV, CB=CB_DIV, RET=RET_DIV, ASSET_RET=ASSET_DIV, dyn=DYN_DIV)
RANK_DIV = []
_, DIV_KEY, DIV_NAME, WA_DIV, r_S_div = select_on(IS_IDX, C_DIV, WORLD_DIV, RANK_DIV)
RANK_DIV.sort(key=lambda x: -x[0])
r_D_div = RET_DIV[(DIV_KEY, DIV_NAME)]
D_RANK_DIV = next((i + 1 for i, (_, k, nm) in enumerate(RANK_DIV) if (k, nm) == (D_KEY, D_NAME)), None)
_TAG = lambda k: CFG_TAG.get(k, "new")
_win = lambda wa: " ".join(f"{s}{OR_WINDOWS[wa[a]]}" for a, s in enumerate(TRADE_SYMBOLS))
_kis = lambda rd, rs: round(float(rs[IS_IDX].std() / rd[IS_IDX].std()), 6)       # Cell 8's vol-matching rule


def resel_row(nm: str, rd: np.ndarray, rs: np.ndarray, k: float, d_lab: str, wa: np.ndarray) -> dict:
    """Cell 9's test on any D / S* pair: NW p of mean d_t = k·r_D − r_S* over the holdout, and its pass rule."""
    d = k * rd - rs
    mu, t, p = nw(d[OOS_IDX], NW_LAG)
    same = len({np.sign(d[ix].mean()) for ix in (PRE_SEAM, POST_SEAM)}) == 1
    return {"variant": nm, "D": d_lab, "S* windows": _win(wa), "k": k, "IS SR D": sharpe(rd[IS_IDX]),
            "IS SR S*": sharpe(rs[IS_IDX]), "holdout SR D": sharpe(rd[OOS_IDX]), "holdout SR S*": sharpe(rs[OOS_IDX]),
            "holdout ΔSR": sharpe(rd[OOS_IDX]) - sharpe(rs[OOS_IDX]), "mean d (ann. %)": 100 * mu, "NW t": t,
            "p one-sided": p, "passes": bool(p < 0.05 and mu > 0 and same)}


_wa_reg = np.array([WIDX[S_STAR[D_KEY]["asset"][s]] for s in TRADE_SYMBOLS])
_lab_D = f"{CFG_TAG[D_KEY]} {D_NAME}"
rows = [resel_row("registered (split-only prices)", r_D, r_S, VOL_K, _lab_D, _wa_reg),
        resel_row("same picks and k, dividends undone", *SENS_RET[DIV_FIXED], VOL_K, _lab_D, _wa_reg)]
if D_KEY in CB_DIV:
    _wa_d, _rs_d = s_star_on(D_KEY, IS_IDX, C_DIV, WORLD_DIV)
    _rd_d = RET_DIV[(D_KEY, D_NAME)]
    rows.append(resel_row("D*'s rule and S*'s rule re-run on corrected prices", _rd_d, _rs_d, _kis(_rd_d, _rs_d), _lab_D, _wa_d))
rows.append(resel_row("whole selection re-made on corrected prices", r_D_div, r_S_div, _kis(r_D_div, r_S_div),
                      f"{_TAG(DIV_KEY)} {DIV_NAME}", WA_DIV))
RESEL_TABLE = pd.DataFrame(rows).set_index("variant")
assert RESEL_TABLE.iloc[0]["passes"] == PASS, "resel_row must reproduce Cell 9's verdict on the registered pair"
RESEL = RESEL_TABLE.iloc[-1]
RESEL_PASS = bool(RESEL["passes"])
SAME_FINALISTS = {c.key() for c in TOP_DIV} == {c.key() for c in TOP_CONFIGS}
T_DIV = time.perf_counter() - t2

# ── report ──
for start, (_, _, _, tab) in WF_META.items():
    print(f"Walk-forward meta-selection, training from {start} (B = {B_MS:,} resamples per year, block {BLOCK_L:.1f}):")
    print(tab.round(3).to_string())
    print(f"   frozen D* re-chosen in {int(tab['= frozen D*'].sum())} of {len(tab)} years · "
          f"{tab[['config', 'selector']].drop_duplicates().shape[0]} distinct choices\n")
print(f"Chained results ({T_META:.0f}s) — d_t = k_y · r_D − r_S*, k_y matched on each year's training window:")
print(META_TABLE.round(3).to_string())
print(f"\nSeed stability — Cell 8's rule on 2008–2015 re-run with {N_SEEDS} other bootstrap seeds (B = {N_BOOT:,}):")
print("   " + " · ".join(f"{c} {n}: {v}/{N_SEEDS}" for (c, n), v in STAB_COUNTS.items()))
print(f"   frozen D* re-chosen {int(((STAB['config'] == CFG_TAG[D_KEY]) & (STAB['selector'] == D_NAME)).sum())}/{N_SEEDS}; "
      f"when it is, S* matches the frozen S* in {int(STAB['S* = frozen'].sum())} — the 2016 row above trains on the same "
      f"2008–2015 and differs from the frozen choice only through these draws")
print(f"\nExecution sensitivities on the frozen D* = {CFG_TAG[D_KEY]} {D_NAME} and S* (k = {VOL_K:.3f}; {T_SENS:.0f}s):")
print(SENS_TABLE.round(3).to_string())
print(f"\nCost-based universe ({UNI}) with the dividend adjustment undone — % of eligible sessions tradable "
      f"(changed assets only; never tradable either way: {', '.join(NEVER_DIV) or 'none'}):")
_chg = DIV_UNI_TABLE[(DIV_UNI_TABLE.iloc[:, [0, 2]].to_numpy() != DIV_UNI_TABLE.iloc[:, [1, 3]].to_numpy()).any(1)]
print(_chg.round(1).to_string() if len(_chg) else "   no asset changes")
if D_CFG.spec.confirm != "close":
    print("   (D* enters on a touch, so the next-open variant does not apply)")

print(f"\nCorrected price basis — Stage A, the forests and Cell 8's rule re-run with the dividend adjustment undone "
      f"({T_DIV:.0f}s; not pre-registered):")
print("   finalists: " + " · ".join(f"{_TAG(c.key())} {c.label()}" for c in TOP_DIV) +
      (f"  (the same {len(TOP_DIV)} as Cell 6)" if SAME_FINALISTS else
       f"  ({sum(c.key() in CB for c in TOP_DIV)} of {len(TOP_DIV)} shared with Cell 6; 'new' = not a Cell 6 finalist)"))
print(f"   D* on corrected prices: {_TAG(DIV_KEY)} {DIV_NAME} (q10 ΔSR vs its S* {RANK_DIV[0][0]:+.3f}; block "
      f"{BLOCK_DIV:.1f}) · " + (f"the registered D* ranks {D_RANK_DIV} of {len(RANK_DIV)}" if D_RANK_DIV else
                                "the registered D*'s config is not a finalist on corrected prices"))
print(RESEL_TABLE.round(3).to_string())
print("   k is frozen in the first two rows and re-matched on 2008–2015 by Cell 8's rule in the re-chosen ones; "
      "'passes' = Cell 9's rule (NW p < 0.05, mean d > 0, same sign either side of the seam).")

fig = go.Figure()
ev = YEARS >= MS_FIRST
fig.add_trace(go.Scatter(x=CAL[ev], y=100 * np.cumsum(d_all[ev]), name="frozen D* vs S* (pre-registered)", mode="lines"))
for start, (_, _, dd, _) in WF_META.items():
    fig.add_trace(go.Scatter(x=CAL[ev], y=100 * np.cumsum(dd[ev]), name=f"re-chosen every January, training from {start}",
                             mode="lines"))
fig.add_trace(go.Scatter(x=CAL[ev], y=100 * np.cumsum((RESEL["k"] * r_D_div - r_S_div)[ev]), mode="lines",
                         name="whole selection re-made on dividend-corrected prices", line_dash="dot"))
fig.add_vline(x=OOS_START, line_dash="dash", line_color="grey")
fig.add_vline(x=IEX_SEAM, line_dash="dot", line_color="grey")
fig.add_vrect(x0=CAL[np.flatnonzero(ev)[0]], x1=OOS_START, fillcolor="grey", opacity=0.10, line_width=0)
fig.update_layout(height=420, hovermode="x unified", yaxis_title="cumulative d_t (% of equity)",
                  title="Dynamic minus static, vol-matched — shaded 2012–2015 is not clean; dashed = holdout start; "
                        "dotted = IEX seam")
fig.show()

Walk-forward meta-selection, training from 2008 (B = 1,000 resamples per year, block 1.3):
     trained on config                     selector  q10 ΔSR      k  = frozen D*                                                   S* windows   SR D  SR S*  d (ann. %)
year                                                                                                                                                                   
2012  2008–2011     C2     ts_max_class_h120_always   -0.588  0.866        False  DIA30 GLD30 IEF30 IWM30 QQQ30 SHY30 SLV30 SPY30 TLT30 USO30 -0.143  0.834      -7.646
2013  2008–2012     C4                      rf_st_0   -0.518  1.360        False  DIA10 GLD30 IEF30 IWM30 QQQ30 SHY30 SLV30 SPY10 TLT30 USO30  0.799  1.081      -1.622
2014  2008–2013     C2  ts_inc_class_h250_k2_always   -0.253  1.034        False  DIA15 GLD15 IEF15 IWM15 QQQ15 SHY15 SLV15 SPY15 TLT15 USO30  3.375  2.715       4.666
2015  2008–2014     C1     ts_inc_class_h250_k2_pos   -0.303  0.982  

## Cell 11 — Every asset vs buy & hold (individual charts)

In [11]:
# ════════════════════════════════════════════════════════════════════════════
#  Every traded asset on its own vs buy & hold.
#   D* and S* restricted to one asset, run as a $100k single-asset account with Cell 1's
#   ACCOUNT_RISK (1 % of equity risked per trade, ≤ 4× notional, integer shares, one trade a
#   day) — the size a trader would actually run, not the 0.25 % portfolio research sizing.
#   Buy & hold = the asset's session-close to session-close return on the dividend-adjusted
#   CSV prices (≈ total return, no costs); idle cash earns nothing in the ORB accounts, which
#   matters most against SHY / IEF. 2008–2015 is shaded: D* and S* were CHOSEN there;
#   only 2016+ is out of sample. Per-asset holdout tests of D* vs S* (vol-matched on the
#   asset's own 2008–2015 account returns) are Holm-corrected; B&H comparisons are descriptive.
#   An asset trades only on sessions in D*'s tradable universe (Cell 5: median round-trip cost over
#   the prior 250 sessions ≤ MAX_COST_R of the stop) — where that never holds the ORB is flat and
#   only buy & hold is shown.
# ════════════════════════════════════════════════════════════════════════════
EV_START = pd.Timestamp(f"{WF_FIRST_YEAR}-01-01")
MIN_TEST_SESS, MIN_TEST_TRADES = 250, 100
STAT_COLS = ["Total return %", "CAGR %", "Volatility %", "Sharpe", "Sortino", "Max drawdown %", "Calmar",
             "Corr. to SPY", "Beta to SPY", "Trades", "Hit ratio %"]


def asset_pick(pick: np.ndarray, a: int) -> np.ndarray:
    """The pick restricted to asset a, the tradable universe and sessions on which the config
    really trades that window (trade_book re-simulates without the gap / OR-width filters)."""
    rr = np.take_along_axis(cbD["R"], np.maximum(pick, 0)[..., None], 2)[..., 0]
    return np.where(uniD & (np.arange(A) == a)[:, None] & (pick >= 0) & np.isfinite(rr), pick, -1)


def account(pick: np.ndarray, a: int) -> tuple:
    pk = asset_pick(pick, a)
    tb = trade_book(D_CFG.spec, pk)
    if tb.empty:
        return np.zeros(T), tb
    rr = np.take_along_axis(cbD["R"], np.maximum(pk, 0)[..., None], 2)[..., 0]
    hit = np.zeros(T, bool)
    hit[tb["cal_idx"].to_numpy()] = True                       # picked sessions whose breakout triggered
    pos = pk[a] >= 0
    assert not (hit & ~pos).any() and np.allclose(rr[a, pos & ~hit], 0.0) \
        and np.allclose(tb["R"], rr[a, tb["cal_idx"]], atol=1e-4), f"{TRADE_SYMBOLS[a]}: trade book ≠ outcome grid"
    return run_risk(tb, ACCOUNT_RISK), tb


def bh(sym: str) -> np.ndarray:
    c = panels[sym]["daily"]["close"].reindex(CAL).ffill()
    return c.pct_change().fillna(0.0).to_numpy()


t0 = time.perf_counter()
bench = pd.Series(R_SPY, index=CAL)
ASSET_ACC, ROWS_H, ROWS_F, TESTS = {}, [], [], []
for a, s in enumerate(TRADE_SYMBOLS):
    first = max(EV_START, panels[s]["dates"][0])
    ev = CAL >= first
    (rD, tD), (rS, tS), rB = account(pick_D, a), account(pick_S, a), bh(s)
    ASSET_ACC[s] = {"D*": rD, "S*": rS, "B&H": rB, "trades": {"D*": tD, "S*": tS}, "first": first}
    for period, m, out in (("holdout", ev & OOS_MASK, ROWS_H), (f"{first.year}+", ev, ROWS_F)):
        for nm, r in (("D*", rD), ("S*", rS), ("B&H", rB)):
            tr = ASSET_ACC[s]["trades"].get(nm)
            tr = tr.loc[m[tr["cal_idx"].to_numpy()]] if tr is not None and len(tr) else (None if nm == "B&H" else tr)
            st = perf_stats(pd.Series(r[m], index=CAL[m]), bench[m], tr)
            out.append({"asset": s, "strategy": nm, **{k: st.get(k, np.nan) for k in STAT_COLS}})
    # holdout test of D* vs S* on this asset, vol-matched on its own 2008–2015 account returns
    isx, oos = np.flatnonzero(ev & IS_MASK), np.flatnonzero(ev & OOS_MASK)
    n_sess = int((uniD[a] & OOS_MASK).sum())
    n_tr = int((tD["cal_idx"].to_numpy() >= OOS_IDX[0]).sum()) if len(tD) else 0
    if n_sess >= MIN_TEST_SESS and n_tr >= MIN_TEST_TRADES and rD[isx].std() > 0 and rS[isx].std() > 0:
        k_a = rS[isx].std() / rD[isx].std()
        mu, t, p = nw(k_a * rD[oos] - rS[oos], NW_LAG)
        TESTS.append({"asset": s, "holdout tradable sessions": n_sess, "D* holdout trades": n_tr, "k": k_a,
                      "SR D*": sharpe(rD[oos]), "SR S*": sharpe(rS[oos]), "SR B&H": sharpe(rB[oos]),
                      "mean d (ann. %)": 100 * mu, "NW t": t, "p one-sided": p})
    else:
        TESTS.append({"asset": s, "holdout tradable sessions": n_sess, "D* holdout trades": n_tr, "k": np.nan,
                      "SR D*": sharpe(rD[oos]), "SR S*": sharpe(rS[oos]), "SR B&H": sharpe(rB[oos]),
                      "mean d (ann. %)": np.nan, "NW t": np.nan, "p one-sided": np.nan})
ACC_HOLDOUT = pd.DataFrame(ROWS_H).set_index(["asset", "strategy"])
ACC_FULL = pd.DataFrame(ROWS_F).set_index(["asset", "strategy"])
ASSET_TESTS = pd.DataFrame(TESTS).set_index("asset")
tested = ASSET_TESTS["p one-sided"].notna()
p_sorted = ASSET_TESTS.loc[tested, "p one-sided"].sort_values()
holm = np.maximum.accumulate([(len(p_sorted) - i) * p for i, p in enumerate(p_sorted)])
ASSET_TESTS["Holm p"] = pd.Series(np.minimum(holm, 1.0), index=p_sorted.index)
ASSET_TESTS["IEX era"] = ["not testable" if s in NOT_TESTABLE_IEX else "ok" for s in ASSET_TESTS.index]
day_cost = np.nanmedian(cbD["wo"]["cost"], axis=2)
COVER = pd.DataFrame({"median cost (R)": [np.nanmedian(day_cost[a, VALID[a] & (CAL >= EV_START)]) for a in range(A)],
                      f"tradable % {EV_START.year}+": [100 * uniD[a, CAL >= EV_START].sum() / max(VALID[a, CAL >= EV_START].sum(), 1)
                                                      for a in range(A)],
                      "tradable % holdout": [100 * uniD[a, OOS_MASK].sum() / max(VALID[a, OOS_MASK].sum(), 1) for a in range(A)],
                      f"tradable % {IEX_SEAM.date()}+": [100 * uniD[a, CAL >= IEX_SEAM].sum() / max(VALID[a, CAL >= IEX_SEAM].sum(), 1)
                                                         for a in range(A)]},
                     index=TRADE_SYMBOLS)
COVER["eligible sessions holdout"] = VALID[:, OOS_MASK].sum(1)
COVER[f"eligible sessions {IEX_SEAM.date()}+"] = VALID[:, CAL >= IEX_SEAM].sum(1)
NEVER = [s for s in TRADE_SYMBOLS if not uniD[TRADE_SYMBOLS.index(s)].any()]
TRADED = list(ASSET_TESTS.index[ASSET_TESTS["D* holdout trades"] >= MIN_TEST_TRADES])
T_ACC = time.perf_counter() - t0

# ── report ──
print(f"Single-asset $ {ACCOUNT_EQUITY:,.0f} accounts at {ACCOUNT_RISK['risk']:.0%} risk per trade "
      f"(D* = {CFG_TAG[D_KEY]} {D_NAME}, S* = per-asset static windows), {T_ACC:.0f}s")
print(f"\nTradable-universe coverage (% of the asset's eligible sessions with median cost ≤ {MAX_COST_R} R; "
      f"eligible = Cell 3's opening-bar rule, which the sparse IEX feed often fails):")
print(COVER.round(2).T.to_string())
if NEVER:
    print(f"   never tradable: {', '.join(NEVER)} — their opening ranges are too narrow for the costs; D* and S* stay flat")
print(f"\nHOLDOUT {OOS_START.date()} → {CAL[-1].date()} — the only out-of-sample comparison:")
print(ACC_HOLDOUT.round(2).to_string())
print(f"\nD* vs S* per asset, holdout (one-sided NW, lag {NW_LAG}; tested only with ≥ {MIN_TEST_SESS} tradable "
      f"sessions and ≥ {MIN_TEST_TRADES} D* trades; Holm over {int(tested.sum())} assets):")
print(ASSET_TESTS.round(3).to_string())
bt = ASSET_TESTS.loc[TRADED]
print(f"\nOn the {len(TRADED)} assets D* traded ≥ {MIN_TEST_TRADES} times in the holdout ({', '.join(TRADED)}), its holdout "
      f"Sharpe beats buy & hold on {int((bt['SR D*'] > bt['SR B&H']).sum())} and S* on {int((bt['SR D*'] > bt['SR S*']).sum())} "
      f"(descriptive — B&H carries the asset's full beta and no costs; D* is flat most of the day and when untradable).")
print(f"\nFull period from each asset's first session ≥ {EV_START.date()} (2008–2015 is in-sample for D* and S*):")
print(ACC_FULL.round(2).to_string())

# ── one figure per asset: growth of the account (log) and drawdown, IS shaded ──
colors = {"D*": "#1f77b4", "S*": "#ff7f0e", "B&H": "#7f7f7f"}
for a, s in enumerate(TRADE_SYMBOLS):
    acc = ASSET_ACC[s]
    ev = CAL >= acc["first"]
    fig = make_subplots(rows=2, cols=1, shared_xaxes=True, row_heights=[0.7, 0.3], vertical_spacing=0.04)
    for nm in (("B&H",) if s in NEVER else ("D*", "S*", "B&H")):
        g = ACCOUNT_EQUITY * np.cumprod(1 + acc[nm][ev])
        sh = ACC_HOLDOUT.loc[(s, nm), "Sharpe"]
        lab = f"{nm if nm != 'B&H' else s + ' buy & hold'} (holdout SR {sh:.2f})"
        fig.add_trace(go.Scatter(x=CAL[ev], y=g, name=lab, mode="lines", line=dict(color=colors[nm])), 1, 1)
        fig.add_trace(go.Scatter(x=CAL[ev], y=100 * (g / np.maximum.accumulate(g) - 1), mode="lines",
                                 showlegend=False, line=dict(color=colors[nm], width=1)), 2, 1)
    for rr in (1, 2):
        fig.add_vrect(x0=max(acc["first"], CAL[IS_IDX[0]]), x1=CAL[IS_IDX[-1]], fillcolor="grey", opacity=0.12,
                      line_width=0, row=rr, col=1)
        fig.add_vline(x=IEX_SEAM, line_dash="dot", line_color="grey", row=rr, col=1)
    fig.update_yaxes(type="log", title="account $", row=1, col=1)
    fig.update_yaxes(title="drawdown %", row=2, col=1)
    note = " · IEX era not testable (too few eligible sessions)" if s in NOT_TESTABLE_IEX else ""
    if s in NEVER:
        note = f" · never in the tradable universe (cost > {MAX_COST_R} R): ORB flat, buy & hold only"
    elif s not in TRADED:
        note = f" · rarely tradable ({COVER.loc[s, 'tradable % holdout']:.0f} % of holdout sessions)"
    fig.update_layout(height=520, hovermode="x unified",
                      title=f"{s} — D* vs S* vs buy & hold · shaded = 2008–2015 selection period, dotted = IEX seam{note}")
    fig.show()

fig = go.Figure([go.Bar(x=list(TRADE_SYMBOLS), y=[ACC_HOLDOUT.loc[(s, nm), "Sharpe"] for s in TRADE_SYMBOLS],
                        name=nm, marker_color=colors[nm]) for nm in ("D*", "S*", "B&H")])
fig.update_layout(barmode="group", height=380, yaxis_title="holdout Sharpe",
                  title=f"Holdout Sharpe by asset, {OOS_START.year}+ — single-asset accounts vs buy & hold")
fig.show()

Single-asset $ 100,000 accounts at 1% risk per trade (D* = C1 ts_inc_class_h250_k2_pos, S* = per-asset static windows), 1s

Tradable-universe coverage (% of the asset's eligible sessions with median cost ≤ 0.15 R; eligible = Cell 3's opening-bar rule, which the sparse IEX feed often fails):
                                   DIA      GLD      IEF      IWM      QQQ      SHY      SLV      SPY      TLT      USO
median cost (R)                   0.10     0.10     0.34     0.07     0.08     0.67     0.30     0.07     0.18     0.23
tradable % 2008+                 96.79    81.21     0.00   100.00    68.12     0.00    13.44   100.00    20.41    17.85
tradable % holdout               95.12    66.79     0.00   100.00   100.00     0.00     9.04   100.00    35.90     4.56
tradable % 2022-03-07+           46.15    75.24     0.00   100.00   100.00     0.00    63.45   100.00    43.28    16.67
eligible sessions holdout      1577.00  1843.00  1319.00  2357.00  2398.00  1088.00  1671.00  2481.00  2465.

## Cell 12 — Volatility comparison vs VXX (2022-03-07 → 2026-10-15)

In [12]:
# ════════════════════════════════════════════════════════════════════════════
#  Volatility comparison, 2022-03-07 → 2026-10-15, with VXX as the volatility reference.
#   VXX is read here for the first time: never traded, never a model feature, never used in a selection.
#   Question: does the OR window that works — and the window D* and its forests favour — depend on the
#   volatility regime? The regime is ex-ante: z = log VXX close − log median of its last 20 closes,
#   taken from the last VXX session BEFORE the trading day, and cut into terciles by thresholds built
#   from earlier z only (≥ 60 of them). Both forests are shown — the stacked one (μ̂_W = E[u_W | x]) and
#   the multiclass one (P(window is best)); whichever drives D* (Cell 8) is labelled as D*'s model.
#   Tests: Newey–West (Bartlett) HAC on daily cross-asset means, slopes per 1 sd of z, Holm over the
#   family. Everything here is inside the holdout and after selection — it describes D*, it tunes nothing.
#   The ETF data end on CAL[-1]; VXX's later sessions appear only in its own chart.
# ════════════════════════════════════════════════════════════════════════════
VOL_Z_LEN, VOL_MIN_HIST = 20, 60
REG_NAMES = ["low", "mid", "high"]
WLAB = [f"{w}m" for w in OR_WINDOWS]


def hac_ols(Y: np.ndarray, X: np.ndarray, lags: int) -> tuple:
    """OLS of each column of Y on X; joint Bartlett-kernel HAC covariance of all coefficients
    (vec order: regressor-major). Returns (B [k, m], V [k·m, k·m])."""
    Y = np.asarray(Y, float).reshape(len(Y), -1)
    n, m = Y.shape
    B = np.linalg.lstsq(X, Y, rcond=None)[0]
    psi = (X[:, :, None] * (Y - X @ B)[:, None, :]).reshape(n, -1)
    S = psi.T @ psi / n
    for l in range(1, lags + 1):
        G = psi[l:].T @ psi[:-l] / n
        S += (1 - l / (lags + 1)) * (G + G.T)
    Ai = np.kron(np.linalg.inv(X.T @ X / n), np.eye(m))
    return B, Ai @ S @ Ai.T / n


def wald(B: np.ndarray, V: np.ndarray, rows: list) -> tuple:
    m = B.shape[1]
    ix = [i * m + j for i in rows for j in range(m)]
    b = B.reshape(-1)[ix]
    W = float(b @ np.linalg.solve(V[np.ix_(ix, ix)], b))
    return W, len(ix), float(stats.chi2.sf(W, len(ix)))


def regime_wald(Y: np.ndarray, reg: np.ndarray, lags: int) -> tuple:
    """Joint HAC Wald that the mid and high regime means of every column of Y equal the low one."""
    X = np.c_[np.ones(len(reg)), reg == 1, reg == 2].astype(float)
    B, V = hac_ols(Y, X, lags)
    return wald(B, V, [1, 2])


def slope_test(y: np.ndarray, z: np.ndarray, lags: int) -> tuple:
    ok = np.isfinite(y) & np.isfinite(z)
    B, V = hac_ols(y[ok], np.c_[np.ones(ok.sum()), z[ok]], lags)
    t = B[1, 0] / np.sqrt(V[1, 1])
    return float(B[1, 0]), float(t), float(2 * stats.norm.sf(abs(t))), int(ok.sum())


def holm_adjust(p: pd.Series) -> pd.Series:
    o = p.sort_values()
    adj = np.minimum(np.maximum.accumulate([(len(o) - i) * v for i, v in enumerate(o)]), 1.0)
    return pd.Series(adj, index=o.index).reindex(p.index)


def daily_mean(x: np.ndarray, m: np.ndarray) -> np.ndarray:
    """Cross-asset mean of x [A, T, ...] over the cells m [A, T], per session (NaN where none)."""
    mm = m.reshape(A, T, *([1] * (x.ndim - 2)))
    n = m.sum(0).reshape(T, *([1] * (x.ndim - 2)))
    return np.where(n > 0, np.where(mm, x, 0).sum(0) / np.maximum(n, 1), np.nan)


def by_regime(x: np.ndarray, m: np.ndarray, f=np.mean) -> list:
    """f over the cells of x [A, T, ...] in m [A, T], for each regime."""
    return [f(x[m & (REG == r)[None, :]], axis=0) for r in range(3)]


# ── 1. VXX daily closes and the ex-ante regime ──
t0 = time.perf_counter()
assert VOL_SYMBOL not in TRADE_SYMBOLS and VOL_SYMBOL not in panels
assert not any(VOL_SYMBOL.lower() in c.lower() for c in X_RF[D_KEY]["cols_s"] + X_RF[D_KEY]["cols_m"])
vx1 = load_etf(VOL_SYMBOL)
vx1 = vx1.loc[(vx1.index >= VOL_START) & (vx1.index < VOL_END + pd.Timedelta(days=1))]
g = vx1.groupby(vx1.index.normalize())
_last = pd.Series(vx1.index, index=vx1.index).groupby(vx1.index.normalize()).max()
VX = pd.DataFrame({"close": g["Close"].last().astype(float), "bars": g["Close"].size(),
                   "last_min": _last.dt.hour * 60 + _last.dt.minute})
VX["z"] = np.log(VX["close"]) - np.log(VX["close"].rolling(VOL_Z_LEN, min_periods=VOL_Z_LEN).median())
VX["lo"] = VX["z"].expanding(min_periods=VOL_MIN_HIST).quantile(1 / 3).shift(1)
VX["hi"] = VX["z"].expanding(min_periods=VOL_MIN_HIST).quantile(2 / 3).shift(1)
VX["regime"] = np.where(VX["lo"].isna() | VX["z"].isna(), -1,
                        np.where(VX["z"] < VX["lo"], 0, np.where(VX["z"] > VX["hi"], 2, 1)))
big = np.abs(np.log(VX["close"]).diff())
assert big.max() < np.log(2.5), f"VXX close jumps ×{np.exp(big.max()):.1f} on {big.idxmax().date()} — unadjusted split?"

# the regime known at the open of session t is the one set by the last VXX close BEFORE t
vd = VX.index.to_numpy()
j = np.searchsorted(vd, CAL.to_numpy(), side="left") - 1
stale = np.where(j >= 0, (CAL.to_numpy() - vd[np.maximum(j, 0)]) / np.timedelta64(1, "D"), np.inf)
okj = (j >= 0) & (stale <= 5)
Z = np.where(okj, VX["z"].to_numpy()[np.maximum(j, 0)], np.nan)
REG = np.where(okj, VX["regime"].to_numpy()[np.maximum(j, 0)], -1)
VOLW = (CAL >= VOL_START) & (CAL <= VOL_END)
VOLM = VOLW & (REG >= 0)
VIDX = np.flatnonzero(VOLM)
Z_SD = float(np.nanstd(Z[VIDX]))
ZS = Z / Z_SD                                                     # slopes below are per 1 sd of z
NW_V = int(np.floor(4 * (len(VIDX) / 100) ** (2 / 9)))
R_VX_SAME = np.log(VX["close"].reindex(CAL) / VX["close"].shift(1).reindex(CAL)).to_numpy()   # ex-post only

print(f"{VOL_SYMBOL}: {len(VX):,} sessions {VX.index[0].date()} → {VX.index[-1].date()} (reference only, never traded) · "
      f"median {VX['bars'].median():.0f} bars/session (IEX, min {VX['bars'].min()}), "
      f"{(VX['last_min'] < 15 * 60 + 50).mean():.1%} of sessions' last bar is before 15:50 (that close is then stale)")
print(f"Regimes start {CAL[VIDX[0]].date()} ({VOL_Z_LEN} closes for z + {VOL_MIN_HIST} earlier z for thresholds); compared: "
      f"{CAL[VIDX[0]].date()} → {CAL[VIDX[-1]].date()}, {len(VIDX):,} sessions (VXX runs to {VX.index[-1].date()}, the ETF "
      f"data end {CAL[-1].date()}) · sd(z) = {Z_SD:.3f} · HAC lag {NW_V}")
print("Sessions per regime:", {REG_NAMES[r]: int((REG[VIDX] == r).sum()) for r in range(3)})

uw = cbD["u"]
M1W = uniD & VOLW[None, :] & np.isfinite(uw).all(2)               # whole window (the ex-post section uses it)
M1 = M1W & VOLM[None, :]
M2 = uniD & VALID & VOLM[None, :]
cnt = pd.DataFrame({"tradable sessions": M1.sum(1), "D* picks a window": ((pick_D >= 0) & M1).sum(1)}, index=TRADE_SYMBOLS)
cnt["share of D*'s picks %"] = 100 * cnt["D* picks a window"] / max(cnt["D* picks a window"].sum(), 1)
print(f"\nWho is in the comparison (D*'s tradable universe, {CAL[VIDX[0]].date()}+; a pick = D* is not flat, and it "
      f"trades only if that window then breaks out — Cell 11 counts the trades):")
print(cnt.round(1).T.to_string())

# ── 2. which window works, by regime (realised u of D*'s config on every tradable asset-session) ──
U = daily_mean(np.nan_to_num(uw), M1W)                             # [T, W]; per session, so equal to M1 on VOLM days
days = VIDX[np.isfinite(U[VIDX]).all(1)]
days_x = np.flatnonzero(VOLW & np.isfinite(U).all(1))
u0 = np.nan_to_num(uw)
lab = np.where(u0.max(2) > 0, u0.argmax(2), NW)                   # the multiclass forest's label
TAB_U = pd.DataFrame([U[days[REG[days] == r]].mean(0) for r in range(3)], index=REG_NAMES, columns=WLAB)
BEST_SHARE = pd.DataFrame([100 * np.bincount(lab[M1 & (REG == r)[None, :]], minlength=NW + 1) / max((M1 & (REG == r)[None, :]).sum(), 1)
                           for r in range(3)], index=REG_NAMES, columns=WLAB + ["none"])

# ── 3. what D* does, and what its forests believe, by regime ──
MIX = pd.DataFrame([100 * np.bincount(np.where(pick_D < 0, NW, pick_D)[M2 & (REG == r)[None, :]], minlength=NW + 1)
                    / max((M2 & (REG == r)[None, :]).sum(), 1) for r in range(3)], index=REG_NAMES, columns=WLAB + ["flat"])
mins = np.where(pick_D >= 0, np.array(OR_WINDOWS, float)[np.maximum(pick_D, 0)], 0.0)

D_FORM = {"st": "stacked", "mc": "multiclass"}.get(D_NAME.split("_")[1]) if D_NAME.startswith("rf_") else None
ST_ROLE = "D*'s model" if D_FORM == "stacked" else (
    f"for reference — D* ({D_NAME}) uses the multiclass forest" if D_FORM == "multiclass" else
    f"for reference — D* ({D_NAME}) is not a forest selector")
MC_ROLE = "D*'s model" if D_FORM == "multiclass" else "the probability view"
SC = PRED[D_KEY]["stacked"]["score"].astype(float)              # μ̂_W, [A, T, W]
M3 = M1 & np.isfinite(SC).all(2)
GATE_AGREE = (((pick_D >= 0) == (SC.max(2) > 0))[M3].mean(), (pick_D == SC.argmax(2))[M3 & (pick_D >= 0)].mean())
MU_HAT = pd.DataFrame(by_regime(SC, M3), index=REG_NAMES, columns=WLAB)
MU_RANK = 1000 * MU_HAT.sub(MU_HAT.mean(1), axis=0)              # the forest's window preference, level removed
MU_HAT["gate open %"] = [round(100 * x, 1) for x in by_regime((SC.max(2) > 0).astype(float), M3)]
DS = daily_mean(np.nan_to_num(SC), M3)
days3 = days[np.isfinite(DS[days]).all(1)]
MU_SLOPES = pd.DataFrame([dict(zip(("slope", "HAC t", "p", "n"), slope_test(DS[VIDX, w], ZS[VIDX], NW_V))) for w in range(NW)],
                         index=[f"μ̂({c})" for c in WLAB])
MU_SLOPES["u ×1000 per sd z"] = 1000 * MU_SLOPES["slope"]
on = M3 & (pick_D >= 0)
mu_pick = np.take_along_axis(SC, np.maximum(pick_D, 0)[..., None], 2)[..., 0]
u_pick = np.take_along_axis(u0, np.maximum(pick_D, 0)[..., None], 2)[..., 0]
TRADE_PCT = 100 - MIX["flat"]                                    # D*'s own pick rate (not flat) by regime, whatever its model
CALIB = pd.DataFrame({"D* picks": [int((on & (REG == r)[None, :]).sum()) for r in range(3)],
                      "forecast μ̂ of the picked window": by_regime(mu_pick, on),
                      "realised u of the picked window": by_regime(u_pick, on)}, index=REG_NAMES)
CALIB["realised − forecast"] = CALIB.iloc[:, 2] - CALIB.iloc[:, 1]

P = PRED[D_KEY]["multiclass"]["proba"].astype(float)            # [A, T, NW + 1], last = "no window wins"
M4 = M1 & np.isfinite(P).all(2)
PROB = pd.DataFrame([100 * x for x in by_regime(P, M4)], index=REG_NAMES, columns=WLAB + ["none"])
P_ARGMAX_NONE = 100 * (P[M4].argmax(1) == NW).mean()
P_TOPWIN = pd.DataFrame([100 * np.bincount(P[M4 & (REG == r)[None, :], :NW].argmax(1), minlength=NW)
                         / max((M4 & (REG == r)[None, :]).sum(), 1) for r in range(3)], index=REG_NAMES, columns=WLAB)
REAL_GIVEN = pd.DataFrame([100 * np.bincount(lab[M4 & (REG == r)[None, :] & (lab < NW)], minlength=NW + 1)[:NW]
                           / max((M4 & (REG == r)[None, :] & (lab < NW)).sum(), 1) for r in range(3)],
                          index=REG_NAMES, columns=WLAB)
DP = daily_mean(np.nan_to_num(P), M4)
MC_SLOPES = pd.DataFrame([dict(zip(("slope", "HAC t", "p", "n"), slope_test(DP[VIDX, c], ZS[VIDX], NW_V)))
                          for c in range(NW + 1)], index=[f"P({c})" for c in WLAB + ["none"]])
MC_SLOPES["pp per sd z"] = 100 * MC_SLOPES["slope"]

# ── 4. does D*'s edge over S* depend on the regime? ──
dv = d_all[VIDX]
D_REG = pd.DataFrame([{"regime": REG_NAMES[r], "sessions": int((REG[VIDX] == r).sum()),
                       "mean d (ann. %)": 100 * 252 * dv[REG[VIDX] == r].mean(),
                       "SR D*": sharpe(r_D[VIDX][REG[VIDX] == r]), "SR S*": sharpe(r_S[VIDX][REG[VIDX] == r])}
                      for r in range(3)]).set_index("regime")
Bd, Vd = hac_ols(dv, np.c_[np.ones(len(dv)), REG[VIDX] == 1, REG[VIDX] == 2].astype(float), NW_V)
D_HL = (100 * 252 * Bd[2, 0], Bd[2, 0] / np.sqrt(Vd[2, 2]))

# ── 5. the test family: same level / ranking tests for realised u and for the forecast μ̂, then D*'s behaviour ──
#   p at HAC lag NW_V and at a quarter (63): z is persistent (AR(1) ≈ 0.9), so the family uses the larger of the two.
NW_LONG = 63
dem = lambda Y: (Y - Y.mean(1, keepdims=True))[:, :-1]              # window-demeaned, rank NW − 1
FAMILY = [("realised u: ORB level (mean over windows) differs across regimes", "wald", U[days].mean(1), REG[days], None, None),
          ("realised u: window ranking differs across regimes", "wald", dem(U[days]), REG[days], None, None),
          ("forecast μ̂: level differs across regimes", "wald", DS[days3].mean(1), REG[days3], None, None),
          ("forecast μ̂: window ranking differs across regimes", "wald", dem(DS[days3]), REG[days3], None, None),
          ("D*: mean OR minutes chosen (when not flat) vs z", "slope", daily_mean(mins, M2 & (pick_D >= 0)), None, 1.0, "min"),
          ("D*: share of tradable sessions flat vs z", "slope", daily_mean((pick_D < 0).astype(float), M2), None, 100.0, "pp"),
          ("D*'s picks: realised u − forecast μ̂ vs z", "slope", daily_mean(u_pick - mu_pick, on), None, 1000.0, "u ×1000"),
          ("d = k·r_D* − r_S* vs z", "slope", d_all, None, 100 * 252, "%/yr")]
VOL_TESTS = []
for nm, kind, y, rg, scale, unit in FAMILY:
    if kind == "wald":
        (W, df_, p), (_, _, p63) = regime_wald(y, rg, NW_V), regime_wald(y, rg, NW_LONG)
        VOL_TESTS.append({"test": nm, "stat": W, "df": df_, "p": p, f"p lag {NW_LONG}": p63, "n": len(rg)})
    else:
        (b, t, p, n), (_, _, p63, _) = slope_test(y[VIDX], ZS[VIDX], NW_V), slope_test(y[VIDX], ZS[VIDX], NW_LONG)
        VOL_TESTS.append({"test": nm, "stat": t, "df": 1, "p": p, f"p lag {NW_LONG}": p63, "n": n,
                      "effect per sd z": scale * b, "units": unit})
VOL_TESTS = pd.DataFrame(VOL_TESTS).set_index("test")
VOL_TESTS["Holm p"] = holm_adjust(VOL_TESTS[["p", f"p lag {NW_LONG}"]].max(1))
T_VOL = time.perf_counter() - t0

# ── report ──
print(f"\nMean realised u by window (R × position weight per asset-session, D*'s config, ex-ante {VOL_SYMBOL} regime):")
print(TAB_U.round(4).to_string())
print(f"\nShare of asset-sessions on which each window was the ex-post best (%; none = no window ended positive):")
print(BEST_SHARE.round(1).to_string())
print(f"\nD*'s choice on tradable asset-sessions (%):")
print(MIX.round(1).to_string())
print(f"\nStacked forest ({ST_ROLE}) — μ̂_W = E[u_W | x] (mean over tradable asset-sessions) and how often its gate "
      f"(max μ̂ > 0) is open; D*'s picks agree with that gate on {GATE_AGREE[0]:.1%} of tradable cells and, on the "
      f"cells D* picks a window on, with its argmax window on {GATE_AGREE[1]:.1%}:")
print(MU_HAT.round(4).to_string())
print(f"   its window preference with the level removed (μ̂_W − mean over windows, u ×1000):")
print(MU_RANK.round(2).to_string())
print(f"   slopes of the daily mean μ̂_W on z (descriptive; the family below tests level and ranking jointly):")
print(MU_SLOPES[["u ×1000 per sd z", "HAC t", "p"]].round(3).T.to_string())
print(f"\nCalibration on D*'s picks — what the stacked forest expected for the picked window vs what it delivered "
      f"(u = 0 when the window does not break out):")
print(CALIB.round(4).to_string())
print(f"\nMulticlass forest ({MC_ROLE}) — mean P(class) %; P(none) is its argmax on {P_ARGMAX_NONE:.1f} % of cells, so below: which window it "
      f"ranks top among the five, vs which window actually won on sessions some window won:")
print(pd.concat({"mean P": PROB, "top window": P_TOPWIN, "realised winner": REAL_GIVEN}, axis=1).round(1).to_string())
print(f"   slopes of the daily mean P on z (descriptive, outside the test family):")
print(MC_SLOPES[["pp per sd z", "HAC t", "p"]].round(3).T.to_string())
print(f"\nD* vs S* by regime (k = {VOL_K:.3f}):")
print(D_REG.round(3).to_string())
print(f"   high − low: {D_HL[0]:+.2f} %/yr, HAC t {D_HL[1]:+.2f} (descriptive)")
print(f"\nFamily of {len(VOL_TESTS)} regime tests (two-sided; χ² for the Wald rows, t for the slopes per 1 sd of z; "
      f"HAC lags {NW_V} and {NW_LONG}; Holm on the larger p):")
print(VOL_TESTS.round(4).to_string())
sig = VOL_TESTS.index[VOL_TESTS["Holm p"] < 0.05].tolist()
print(f"\n→ {len(sig)} of {len(VOL_TESTS)} survive Holm at 5 %" + (": " + "; ".join(sig) if sig else "."))
_hp = VOL_TESTS["Holm p"].tolist()
_sig = lambda i: "differs" if _hp[i] < 0.05 else "does not differ detectably"
print(f"   Realised: mean u over the windows is {TAB_U.mean(1)['low']:+.4f} / {TAB_U.mean(1)['mid']:+.4f} / "
      f"{TAB_U.mean(1)['high']:+.4f} in low / mid / high {VOL_SYMBOL}; the level {_sig(0)} and the window ranking {_sig(1)}.")
print(f"   Forecast: μ̂ averages {MU_HAT[WLAB].mean(1)['low']:.4f} / {MU_HAT[WLAB].mean(1)['mid']:.4f} / "
      f"{MU_HAT[WLAB].mean(1)['high']:.4f}; its level {_sig(2)} and its ranking {_sig(3)} — but the ranking shifts are "
      f"≤ {MU_RANK.abs().max().max():.1f}×10⁻³ u against a per-cell sd of u of {np.nanstd(u0[M1]):.2f}.")
_cr, _cf, _mu = CALIB["realised u of the picked window"], CALIB["forecast μ̂ of the picked window"], MU_HAT[WLAB].mean(1)
# "picks a window more often in high volatility" only when the flat-share test says so (a falling flat share = more picks)
FLAT_TEST = VOL_TESTS.loc["D*: share of tradable sessions flat vs z"]
_fp, _fe = float(FLAT_TEST["Holm p"]), float(FLAT_TEST["effect per sd z"])
TRADES_MORE_HI = bool(_fp < 0.05 and _fe < 0)
_rate = (f"it is flat {'LESS' if _fe < 0 else 'MORE'} often when volatility is high (flat share {_fe:+.1f} pp per sd z, Holm p = {_fp:.4f})"
         if _fp < 0.05 else f"the difference in how often it is flat is not significant (flat share vs z, Holm p = {_fp:.3f})")
print(f"   So D* picks a window (is not flat) on {TRADE_PCT['low']:.1f} % / {TRADE_PCT['mid']:.1f} % / {TRADE_PCT['high']:.1f} % "
      f"of tradable sessions in low / mid / high {VOL_SYMBOL}: {_rate}.")
print(f"   Its picks realised {_cr['low']:+.4f} / {_cr['mid']:+.4f} / {_cr['high']:+.4f} "
      f"(stacked forecast {_cf['low']:+.4f} / {_cf['mid']:+.4f} / {_cf['high']:+.4f})"
      + ((": it is flat less often where its picks did worse"
          + ("; the stacked forest reads higher volatility as more ORB opportunity and the realised outcomes went the other way."
             if D_FORM == "stacked" and _mu["high"] > _mu["low"] else "."))
         if TRADES_MORE_HI and _cr["high"] < _cr["low"] else "."))

# ── ex-post (labelled, not tradable): hindsight terciles of z, and same-day VXX moves, from VOL_START ──
okz = days_x[np.isfinite(Z[days_x])]
reg_post = np.digitize(Z[okz], np.quantile(Z[okz], [1 / 3, 2 / 3]))
okv = days_x[np.isfinite(R_VX_SAME[days_x])]
reg_same = np.digitize(R_VX_SAME[okv], np.quantile(R_VX_SAME[okv], [1 / 3, 2 / 3]))
EXPOST, W_X = {}, {}
for nm, ix, rg, idx in ((f"z, full-sample terciles ({CAL[okz[0]].date()}+)", okz, reg_post, REG_NAMES),
                        (f"same-day {VOL_SYMBOL} return terciles ({CAL[okv[0]].date()}+)", okv, reg_same,
                         ["falling", "flat", "rising"])):
    tab = pd.DataFrame([U[ix][rg == r].mean(0) for r in range(3)], index=idx, columns=WLAB)
    tab["all windows"] = tab.mean(1)
    EXPOST[nm] = tab
    W_X[nm] = (regime_wald(U[ix].mean(1), rg, NW_V), regime_wald((U[ix] - U[ix].mean(1, keepdims=True))[:, :-1], rg, NW_V))
print(f"\nEX-POST (hindsight, cannot be traded) — mean u by window:")
print(pd.concat(EXPOST).round(4).to_string())
for nm, ((w1, d1, p1), (w2, d2, p2)) in W_X.items():
    print(f"   {nm}: level χ²({d1}) = {w1:.1f}, p = {p1:.4f} · ranking χ²({d2}) = {w2:.1f}, p = {p2:.4f}")
print(f"({T_VOL:.1f}s)")

# ── charts ──
fig = make_subplots(rows=2, cols=1, shared_xaxes=True, row_heights=[0.6, 0.4], vertical_spacing=0.05)
fig.add_trace(go.Scatter(x=VX.index, y=VX["close"], name=f"{VOL_SYMBOL} close", line=dict(color="#d62728")), 1, 1)
fig.add_trace(go.Scatter(x=VX.index, y=VX["z"], name="z = log close − log 20-day median", line=dict(color="#444", width=1)), 2, 1)
for c, nm, col in (("lo", "lower tercile (earlier z only)", "#2ca02c"), ("hi", "upper tercile (earlier z only)", "#ff7f0e")):
    fig.add_trace(go.Scatter(x=VX.index, y=VX[c], name=nm, line=dict(color=col, dash="dash")), 2, 1)
for rr in (1, 2):
    fig.add_vline(x=CAL[-1], line_dash="dot", line_color="grey", row=rr, col=1)
fig.update_yaxes(type="log", title=VOL_SYMBOL, row=1, col=1)
fig.update_yaxes(title="z", row=2, col=1)
fig.update_layout(height=520, hovermode="x unified",
                  title=f"{VOL_SYMBOL} {VX.index[0].date()} → {VX.index[-1].date()} — volatility reference only, never traded "
                        f"(dotted: last strategy session {CAL[-1].date()})")
fig.show()

rcol = {"low": "#2ca02c", "mid": "#7f7f7f", "high": "#d62728"}
fig = make_subplots(rows=1, cols=2, column_widths=[0.65, 0.35],
                    subplot_titles=("mean u by OR window (± 2 HAC s.e.)", "D* − S* (vol-matched), %/yr (± 2 HAC s.e.)"))
for r, nm in enumerate(REG_NAMES):
    dr = days[REG[days] == r]
    se = [2 * np.sqrt(hac_ols(U[dr, w], np.ones((len(dr), 1)), NW_V)[1][0, 0]) for w in range(NW)]
    fig.add_trace(go.Bar(x=WLAB, y=TAB_U.loc[nm], name=f"{nm} vol", marker_color=rcol[nm],
                         error_y=dict(type="data", array=se)), 1, 1)
    vr = dv[REG[VIDX] == r]
    se_d = 2 * 100 * 252 * np.sqrt(hac_ols(vr, np.ones((len(vr), 1)), NW_V)[1][0, 0])
    fig.add_trace(go.Bar(x=[nm], y=[D_REG.loc[nm, "mean d (ann. %)"]], marker_color=rcol[nm], showlegend=False,
                         error_y=dict(type="data", array=[se_d])), 1, 2)
fig.update_layout(barmode="group", height=420, title=f"Realised ORB outcomes by ex-ante {VOL_SYMBOL} regime, "
                                                      f"{CAL[VIDX[0]].date()} → {CAL[VIDX[-1]].date()}")
fig.show()

pal = ["#1f77b4", "#aec7e8", "#ff7f0e", "#2ca02c", "#9467bd", "#c7c7c7"]
fig = make_subplots(rows=2, cols=2, vertical_spacing=0.14,
                    subplot_titles=("D*'s choice (% of tradable asset-sessions)",
                                    "stacked forest μ̂ by window" + (" (D*'s model)" if D_FORM == "stacked" else ""),
                                    "multiclass forest: P(window is best) %, 'none' omitted"
                                    + (" (D*'s model)" if D_FORM == "multiclass" else ""),
                                    "D*'s picks: stacked forecast μ̂ vs realised u"))
for c, col in enumerate(WLAB + ["flat"]):
    fig.add_trace(go.Bar(x=REG_NAMES, y=MIX[col], name=col, marker_color=pal[c], legendgroup=col), 1, 1)
for c, col in enumerate(WLAB):
    fig.add_trace(go.Bar(x=REG_NAMES, y=MU_HAT[col], name=col, marker_color=pal[c], legendgroup=col, showlegend=False), 1, 2)
    fig.add_trace(go.Bar(x=REG_NAMES, y=PROB[col], name=col, marker_color=pal[c], legendgroup=col, showlegend=False), 2, 1)
for nm, col in (("forecast μ̂ of the picked window", "#9467bd"), ("realised u of the picked window", "#17becf")):
    fig.add_trace(go.Bar(x=REG_NAMES, y=CALIB[nm], name=nm.split(" of")[0], marker_color=col), 2, 2)
fig.update_xaxes(title=f"ex-ante {VOL_SYMBOL} regime", row=2)
fig.update_layout(barmode="group", height=720, title=f"Does the switching respond to volatility? ({CAL[VIDX[0]].date()} → {CAL[VIDX[-1]].date()})")
fig.show()

VXX: 1,158 sessions 2022-03-07 → 2026-10-15 (reference only, never traded) · median 241 bars/session (IEX, min 16), 1.6% of sessions' last bar is before 15:50 (that close is then stale)
Regimes start 2022-06-30 (20 closes for z + 60 earlier z for thresholds); compared: 2022-06-30 → 2026-09-21, 1,060 sessions (VXX runs to 2026-10-15, the ETF data end 2026-09-21) · sd(z) = 0.086 · HAC lag 6
Sessions per regime: {'low': 341, 'mid': 355, 'high': 364}

Who is in the comparison (D*'s tradable universe, 2022-06-30+; a pick = D* is not flat, and it trades only if that window then breaks out — Cell 11 counts the trades):
                        DIA    GLD  IEF    IWM    QQQ  SHY    SLV    SPY    TLT  USO
tradable sessions      53.0  296.0  0.0  918.0  944.0  0.0  151.0  978.0  371.0  0.0
D* picks a window      53.0   54.0  0.0  918.0  944.0  0.0   19.0  978.0    3.0  0.0
share of D*'s picks %   1.8    1.8  0.0   30.9   31.8  0.0    0.6   32.9    0.1  0.0

Mean realised u by window (R × position

## Cell 13 — Conclusions

In [13]:
# ════════════════════════════════════════════════════════════════════════════
#  Conclusions — every number and every yes / no below is computed from the cells above.
#   Refuses to run on a QUICK smoke test or if the pre-registration file no longer matches Cell 8.
# ════════════════════════════════════════════════════════════════════════════
assert not QUICK, "QUICK mode is a smoke test — re-run with QUICK = False before reading conclusions"
assert json.loads(prereg_path.read_text())["sha256"] == PREREG_HASH, \
    "pre-registration changed since Cell 8 — the holdout would no longer be ex-ante"

yn = lambda ok: "YES" if ok else "NO"
_arm = lambda prefix: ARMS_TABLE.loc[next(i for i in ARMS_TABLE.index if i.startswith(prefix))]
_vt = lambda prefix: float(VOL_TESTS["Holm p"][[i for i in VOL_TESTS.index if i.startswith(prefix)][0]])
H_P = HOLDOUT[f"NW p one-sided (lag {NW_LAG})"]
N_POS_Q10 = int((SEL["q10 ΔSR vs S*"] > 0).sum())
RF_IS = RF_DIAG[D_KEY][RF_DIAG[D_KEY]["year"].between(WF_FIRST_YEAR, IS_END.year)].groupby("form").mean(numeric_only=True)
WF_P = {s: float(META_TABLE.loc[(s, "2016+ (ex-ante)"), "p one-sided"]) for s in (WF_FIRST_YEAR, WF_FIRST_YEAR + 1)}
STAB_D = STAB["config"].eq(CFG_TAG[D_KEY]) & STAB["selector"].eq(D_NAME)
SENS_P = SENS_TABLE["p one-sided"].iloc[1:]
_SHORT = {"bar-close": "next-open fills", "round-trip": "2× costs", DIV_FIXED: "dividends undone",
          DIV_UNI: "same + universe re-derived"}
bt = ASSET_TESTS.loc[TRADED]
SW_D = float(_arm("switch, no gate")["vs S*: mean d (ann. %)"])
GATE_SR, S_SR = float(_arm("fixed S* + ")["holdout Sharpe"]), float(_arm("fixed S*, no gate")["holdout Sharpe"])
HI_WORST = TAB_U.mean(1).idxmin() == "high"
GATE_HI = TRADES_MORE_HI                                         # Cell 12: flat share falls with z, Holm p < 0.05
_RF_OK = {f: bool(RF_IS.loc[f, "picked = best"] > RF_IS.loc[f, "modal best"]) for f in ("stacked", "multiclass")}
_FLAB = f" — answer for the {D_FORM} forest, D*'s model" if D_FORM else " — D* itself uses neither forest"
_N_RF_OK = sum(_RF_OK.values())
_RF_ANS = (yn(_RF_OK[D_FORM]) if D_FORM else "YES (both)" if _N_RF_OK == 2 else "NO" if _N_RF_OK == 0 else
           f"{next(f for f, ok in _RF_OK.items() if ok).upper()} ONLY")
_P_IS = PRED[D_KEY]["multiclass"]["proba"].astype(float)
_M_IS = (VALID & CB[D_KEY]["TRADABLE"] & ((YEARS >= WF_FIRST_YEAR) & (YEARS <= IS_END.year))[None, :]
         & np.isfinite(_P_IS).all(2))
P_NONE_IS = 100 * (_P_IS[_M_IS].argmax(1) == NW).mean()          # multiclass argmax = "no window wins", 2008–2015
D_DESC = ({"stacked": "the stacked random forest", "multiclass": "the multiclass random forest"}[D_FORM] + f" ({D_NAME})" if D_FORM
          else f"the trailing time-series rule {D_NAME}" if D_NAME.startswith("ts_") else f"the {D_NAME} rule")
_TR, _NT = ACC_HOLDOUT["Total return %"].unstack(), ACC_HOLDOUT["Trades"].unstack()   # asset × strategy, holdout
_dTR = (_TR["D*"] - _TR["S*"])[(_NT["D*"] + _NT["S*"]) > 0]                          # single-asset accounts, D* − S*
_pp = lambda a: f"{a} {_dTR[a]:+.0f} pp ({int(_NT.loc[a, 'D*'])} vs {int(_NT.loc[a, 'S*'])} trades)"
SR_ENS, SR_SCL, SR_SCO = (float(ARMS_TABLE.loc[i, "holdout Sharpe"]) for i in ("all-window ensemble", "S*_class", "S*_common"))
_BEST_ANY = next(i for i in ARMS_TABLE.index if i.startswith("best of any family"))
REPAIRED = SEAM_TABLE.index[SEAM_TABLE["repaired"].astype(bool)].tolist()
dropped = LOOKAHEAD[LOOKAHEAD["group"] == "dropped by old filter"]
kept = LOOKAHEAD[LOOKAHEAD["group"] == "kept by old filter"]

_ST_ROLE = "D*'s model" if D_FORM == "stacked" else "not D*'s model"
VOL_RESP = _vt("D*: share of tradable sessions flat") < 0.05 or _vt("D*: mean OR minutes chosen") < 0.05   # D*'s behaviour only
_UNGATED = D_NAME.endswith("_always")
_GLAB = "D*'s gate" if not _UNGATED else f"the gate of its sibling {GATED_NAME}"
_DIV = DIV_FIXED
_UNI_CHG = DIV_UNI_TABLE[DIV_UNI_TABLE["tradable % holdout (dividends undone)"].round()
                         != DIV_UNI_TABLE["tradable % holdout (split only)"].round()]
_UNI_TXT = ("holdout tradable share " + ", ".join(f"{s} {r['tradable % holdout (split only)']:.0f} → "
                                                 f"{r['tradable % holdout (dividends undone)']:.0f} %" for s, r in _UNI_CHG.iterrows())
            if len(_UNI_CHG) else "no asset's holdout tradable share changes by a whole percent")
_NEVER_BOTH = [s for s in NEVER_DIV if s in NEVER]
_NEVER_TXT = f"; never tradable either way: {', '.join(_NEVER_BOTH)}" if _NEVER_BOTH else ""
_TRD_DIV = [a for a in range(A) if CB_DIV[DIV_KEY]["uni"][UNI][a].any()]
_SW_DIV = " ".join(f"{TRADE_SYMBOLS[a]}{OR_WINDOWS[WA_DIV[a]]}" for a in _TRD_DIV)
_PLB_TXT = ("Placebo B draws each asset's days independently, which spreads the days D* picks on the same sessions across\n"
            "    assets: part of its higher median can be diversification, not better days." if not _UNGATED else
            "D* picks a window on every eligible session, so placebo B's days are D*'s own: it tests the windows only.")
_RD = RESEL["D"] + (" (new = not a Cell 6 finalist)" if DIV_KEY not in CFG_TAG else "")      # the re-made pick
_RESEL_WHY = f"NW p {RESEL['p one-sided']:.3f}" + (f", but mean d changes sign at the {IEX_SEAM.date()} seam"
                                                   if RESEL["p one-sided"] < 0.05 and not RESEL_PASS else "")
_EXEC = ("next-open fills, " if any(i.startswith("bar-close") for i in SENS_P.index) else "") + \
        "2× costs, dividends undone with and without re-deriving the universe"
_t0 = pd.Timestamp(f"{WF_FIRST_YEAR}-01-01")
_i0 = {s: min(int(pd.DatetimeIndex(panels[s]["dates"]).searchsorted(_t0)), len(panels[s]["dates"]) - 1) for s in TRADE_SYMBOLS}
_OV = {s: float(SCALE["split+div"][s][i] / SCALE["split"][s][i]) for s, i in _i0.items()}   # per-share cost overstatement in R
_OV_MAX = max(_OV, key=_OV.get)
SCORE = [
    ("Bootstrap search 2008–2015 — the best variation",
     f"D* = {CFG_TAG[D_KEY]} · {D_NAME}: rank 1 of {len(SEL)} dynamic variants by the 10th-percentile ΔSR vs S*; "
     f"{N_POS_Q10} have that percentile above 0 (best {OBS_SEL:+.3f}) · best variation of any family by IS q10, "
     f"{_BEST_ANY.split('(')[-1].rstrip(')')}: holdout SR {ARMS_TABLE.loc[_BEST_ANY, 'holdout Sharpe']:.2f}", "—"),
    ("Switching beats the best static window in-sample, after data-snooping",
     f"SPA_c p {SPA_S['SPA_c']:.3f} (vs all-window ensemble {SPA_E['SPA_c']:.3f}) · deflated ΔSR {DSR['D* vs S* (N_eff)']:.3f}",
     yn(SPA_S["SPA_c"] < 0.05)),
    (f"Pre-registered holdout 2016 → {CAL[OOS_IDX[-1]].date()}: D* beats S*",
     f"ΔSR {HOLDOUT['ΔSR']:+.3f} (90 % CI {HOLDOUT['ΔSR 90% CI low']:+.2f} … {HOLDOUT['ΔSR 90% CI high']:+.2f}) · "
     f"NW p {H_P:.3f} · detectable only above {MDE['holdout']:.2f}", "PASS" if PASS else "FAIL"),
    ("Holds when the choice is re-made every year (walk-forward)",
     f"2016+ p {WF_P[WF_FIRST_YEAR]:.3f} (training from {WF_FIRST_YEAR}) / {WF_P[WF_FIRST_YEAR + 1]:.3f} (from {WF_FIRST_YEAR + 1})",
     yn(max(WF_P.values()) < 0.05)),
    ("The choice of D* is stable across bootstrap seeds",
     f"{int(STAB_D.sum())} of {len(STAB)} seeds pick D*; {int((STAB_D & STAB['S* = frozen']).sum())} of those also "
     f"reproduce the frozen S*", yn(STAB_D.mean() >= 0.9)),
    (f"Execution changes ({_EXEC}) change the holdout verdict",
     " · ".join(f"{_SHORT.get(i, _SHORT.get(i.split(' ')[0], i))} ΔSR {SENS_TABLE.loc[i, 'holdout ΔSR']:+.2f} p {SENS_TABLE.loc[i, 'p one-sided']:.2f}"
                for i in SENS_P.index), yn((SENS_TABLE["passes"].iloc[1:] != PASS).any())),
    ("Re-run with the same rules on estimated dividend-corrected prices (Stage A, forests, S* and D* re-chosen), the procedure passes the same test",
     f"re-made D {_RD} · S* {_SW_DIV} · ΔSR {RESEL['holdout ΔSR']:+.2f}, {_RESEL_WHY} (registered "
     f"{HOLDOUT['ΔSR']:+.2f}, p {H_P:.3f}) · {'same finalists' if SAME_FINALISTS else 'finalists changed'}"
     f"{'; the registered D* is not among them' if D_RANK_DIV is None else f'; the registered D* ranks {D_RANK_DIV} of {len(RANK_DIV)}'}"
     " · not pre-registered",
     "PASS" if RESEL_PASS else "FAIL"),
    ("D*'s window choice beats a stale one",
     f"lagged-window placebo on D*'s own days p {P_PL['A']:.3f} · switching without the gate {SW_D:+.2f} %/yr vs S* · "
     f"captures {100 * ORACLE['capture share']:+.1f} % of the hindsight-switch gain", yn(P_PL["A"] < 0.05)),
    ("D*'s gate picks better days than random ones within each asset-year",
     f"{_GLAB} with S*'s fixed window: SR {GATE_SR:.2f} vs {S_SR:.2f} on every eligible day (t {_arm('fixed S* + ')['NW t']:.2f}); "
     f"vs random days with the same window, same count per asset-year: p {P_PL['D']:.3f} (descriptive) · registered "
     f"placebo B (random days AND stale windows) p {P_PL['B']:.3f}",
     "n/a — D* is ungated" if _UNGATED else yn(P_PL["D"] < 0.05)),
    (f"The forests pick the winning window more often than the year's modal winner (2008–2015, out of fold){_FLAB}",
     f"on sessions where some window made money, stacked picks the winner {RF_IS.loc['stacked', 'picked = best']:.1%} vs "
     f"{RF_IS.loc['stacked', 'modal best']:.1%} for always naming the year's modal winner (in hindsight) · multiclass "
     f"{RF_IS.loc['multiclass', 'picked = best']:.1%} vs {RF_IS.loc['multiclass', 'modal best']:.1%} · multiclass rates "
     f"'no window' most likely on {P_NONE_IS:.0f} % of in-sample cells",
     _RF_ANS),
    (f"Per asset: D* beats S* after Holm, on the {len(TRADED)} assets it traded ≥ {MIN_TEST_TRADES} times",
     f"{int((bt['Holm p'] < 0.05).sum())} of {len(TRADED)} · smallest: " +
     ", ".join(f"{a} p {bt.loc[a, 'p one-sided']:.3f} (Holm {bt.loc[a, 'Holm p']:.3f})" for a in bt["p one-sided"].nsmallest(2).index),
     yn((bt["Holm p"] < 0.05).any())),
    ("Per asset: D*'s holdout Sharpe beats buy & hold (descriptive)",
     f"{int((bt['SR D*'] > bt['SR B&H']).sum())} of {len(TRADED)} · " +
     ", ".join(f"{a} {bt.loc[a, 'SR D*']:.2f} vs {bt.loc[a, 'SR B&H']:.2f}" for a in TRADED),
     "MOSTLY NO" if (bt["SR D*"] > bt["SR B&H"]).mean() < 0.5 else "MOSTLY YES"),
    (f"{VOL_SYMBOL} {CAL[VIDX[0]].date()} → {CAL[VIDX[-1]].date()}: the window that works depends on volatility",
     f"realised window ranking Holm p {_vt('realised u: window ranking'):.2f} · ORB level Holm p {_vt('realised u: ORB level'):.2f} "
     f"(mean u {TAB_U.mean(1)['low']:+.3f} / {TAB_U.mean(1)['mid']:+.3f} / {TAB_U.mean(1)['high']:+.3f} low / mid / high)",
     yn(_vt("realised u: window ranking") < 0.05)),
    (f"{VOL_SYMBOL}: D*'s trading responds to volatility",
     f"D* picks a window on {TRADE_PCT['low']:.1f} → {TRADE_PCT['high']:.1f} % of tradable sessions low → high (flat-share slope Holm p "
     f"{_vt('D*: share of tradable sessions flat'):.4f}; OR minutes chosen Holm p {_vt('D*: mean OR minutes chosen'):.4f}) · "
     f"evidence only: stacked forecast μ̂ level Holm p {_vt('forecast μ̂: level'):.4f} ({_ST_ROLE}), high-vol picks "
     f"stacked forecast {CALIB.iloc[2, 1]:+.3f}, realised {CALIB.iloc[2, 2]:+.3f}",
     ("YES — the wrong way" if GATE_HI and HI_WORST else "YES") if VOL_RESP else "NO"),
]
print("SCORECARD")
for i, (q, ev, ans) in enumerate(SCORE, 1):
    print(f"{i:>3}. {q}  →  {ans}\n       {ev}")

_QUAL = ([f"the holdout is not blind (caveats)"] if PREREG_HISTORY else []) + \
        ([f"the walk-forward re-test fails when training starts in {max(WF_P, key=WF_P.get)} (p {max(WF_P.values()):.3f})"]
         if max(WF_P.values()) >= 0.05 else []) + \
        ([f"no single asset's gain survives Holm (0 of {len(TRADED)})"] if not (bt["Holm p"] < 0.05).any() else []) + \
        ([f"re-run on estimated dividend-corrected prices, the same rules pick {_RD} and fail ({_RESEL_WHY})"]
         if not RESEL_PASS else [])
_HEAD = (f"Against fixing each asset's best static window (S*) it {'PASSES' if SPA_S['SPA_c'] < 0.05 else 'FAILS'} in-sample after "
         f"data-snooping (SPA p {SPA_S['SPA_c']:.2f})\n  and {'PASSES' if PASS else 'FAILS'} the pre-registered holdout (Sharpe "
         f"{HOLDOUT['SR D*']:.2f} vs {HOLDOUT['SR S*']:.2f}, NW p {H_P:.3f}; this test sees only gaps above ≈ {MDE['holdout']:.2f})"
         + (" — but " + "; ".join(_QUAL) if PASS and _QUAL else "") + "."
         + (f"\n  Re-run on estimated dividend-corrected prices, the same rules pick {_RD} and would pass "
            f"(NW p {RESEL['p one-sided']:.3f}; not pre-registered)." if RESEL_PASS and not PASS else ""))
_A_TXT = (f"{'the window switching' if P_PL['A'] < 0.05 else 'not the window switching'}: placebo A p {P_PL['A']:.3f} (on D*'s own "
          f"days, a {PLACEBO_LAG[0]}–{PLACEBO_LAG[1]}-session-old window did at least as well in {100 * (PL_A >= sharpe(x_D)).mean():.1f} % "
          f"of draws), the ungated switch {SW_D:+.2f} %/yr vs S* (t {_arm('switch, no gate')['NW t']:.2f}), "
          f"{100 * ORACLE['capture share']:+.1f} % of the hindsight-switch gain")
_B_TXT = (f"D* is ungated, so it does not choose days (placebo B p {P_PL['B']:.3f})" if _UNGATED else
          f"{'the choice of days' if P_PL['D'] < 0.05 else 'not the choice of days'} within an asset-year: S*'s fixed window "
          f"on the gate's days vs on random days p {P_PL['D']:.3f} (descriptive); placebo B (random days and stale windows) "
          f"p {P_PL['B']:.3f}")
_C_TXT = ((f"what every placebo keeps is the number of days D* is not flat in each asset-year — D* picks a window on every "
           f"eligible session; placebo D keeps the counts of {_GLAB}. S*'s windows on the days {_GLAB} opens: " if _UNGATED else
           f"what every placebo keeps is how many days the gate opens for each asset each year. "
           f"S*'s windows on D*'s days: ") +
          f"SR {GATE_SR:.2f} vs {S_SR:.2f} (t {_arm('fixed S* + ')['NW t']:.2f}). "
          f"Single-asset accounts, D* − S*: {', '.join(_pp(a) for a in _dTR.nlargest(2).index)}; worst {_pp(_dTR.idxmin())}")
_W_TXT = (f"S* is a weak benchmark out of sample: S*_class {SR_SCL:.2f}, S*_common {SR_SCO:.2f} and the all-window ensemble "
          f"{SR_ENS:.2f} (no window selection at all) vs S* {S_SR:.2f}; D* {'beats' if HOLDOUT['SR D*'] > SR_ENS else 'does not beat'} "
          f"the ensemble ({HOLDOUT['SR D*']:.2f})") if min(SR_SCL, SR_SCO, SR_ENS) > S_SR else \
         (f"S*'s per-asset windows vs simpler statics: S*_class {SR_SCL:.2f}, S*_common {SR_SCO:.2f}, all-window ensemble "
          f"{SR_ENS:.2f}, S* {S_SR:.2f}")
_mu = TAB_U.mean(1)
_REG_ARMS = [f"switch, no gate ({UNGATED_NAME})" if _UNGATED else f"switch + gate ({GATED_NAME})", "fixed S*, no gate (= S*)"]
_REG_ARMS += [i for i in ARMS_TABLE.index if i.startswith("best of any family")
              and BEST_ANY_KEY in ((D_KEY, D_NAME), (D_KEY, "S*_asset"))]
_POST_BEST = ARMS_TABLE.drop(index=_REG_ARMS)["holdout Sharpe"].idxmax()
if VOL_RESP:
    _VOL = (f"D*'s trading responds to volatility (Cell 12){', most in high volatility, where ORB outcomes were worst' if GATE_HI and HI_WORST else ''}")
else:
    _VOL = (f"D*'s trading does not respond to volatility (it picks a window on {TRADE_PCT['low']:.1f} / {TRADE_PCT['high']:.1f} % "
            f"of tradable sessions in low / high {VOL_SYMBOL})" +
            (f", although ORB outcomes were lowest in high volatility (mean u {_mu['high']:+.3f} vs {_mu['low']:+.3f} in low; "
             f"Holm p {_vt('realised u: ORB level'):.2f})" if HI_WORST else ""))
_loses = (f"  · That the ORB loses money: holdout Sharpe D* {HOLDOUT['SR D*']:.2f}, S* {HOLDOUT['SR S*']:.2f} on the tradable universe."
          if min(HOLDOUT["SR D*"], HOLDOUT["SR S*"]) > 0 else
          f"  · That the ORB has an edge at all out of sample: holdout Sharpe D* {HOLDOUT['SR D*']:.2f}, S* {HOLDOUT['SR S*']:.2f}.")
_seam = (f"Cell 2 rescaled the pre-seam prices of {', '.join(f'{s} ×{SEAM_FACTOR[s]:.4f}' for s in REPAIRED)} onto the IEX basis "
         f"(the {IEX_SEAM.date()} gap was below −{100 * SEAM_TOL:.0f} %)" if REPAIRED else "no symbol needed a seam repair")
print(f"""
VERDICT
  The dynamic ORB — {CFG_TAG[D_KEY]} ({D_CFG.label()}), with {D_DESC} picking the OR window each session —
  is the best of {len(SEL)} dynamic variants by the bootstrap rule. {_HEAD}
  Where the holdout difference comes from (descriptive, Cells 9 and 11):
    · {_A_TXT};
    · {_B_TXT};
    · {_C_TXT};
    · {_W_TXT}.
  In the {VOL_SYMBOL} comparison {_VOL}.

WHAT THIS DOES NOT SHOW
{_loses}
  · That no dynamic rule can work: only these features, forests and gates were tried, and a real improvement smaller
    than ≈ {MDE['holdout']:.2f} Sharpe would not show up here.
  · Anything about the other Cell 9 arms — they were looked at after the holdout opened. The best of them,
    "{_POST_BEST}" (SR {ARMS_TABLE.loc[_POST_BEST, 'holdout Sharpe']:.2f}, t {ARMS_TABLE.loc[_POST_BEST, 'NW t']:.2f}), is a hypothesis for the forward log, not a result.

CAVEATS
  · Not a blind holdout. {" ".join(PREREG_HISTORY)}
    This run's rule is frozen (sha256 {PREREG_HASH[:12]}…, universe {UNIV_TAG}), and Cells 10–12 were written knowing Cell 9's answer.
  · Placebos: the registered ones (Cell 9) give each session the window D*'s ungated selector chose {PLACEBO_LAG[0]}–{PLACEBO_LAG[1]}
    sessions earlier. The development registration's permutation method, re-applied to this run's D*, lets a session's own
    outcome into its placebo (leaky; for comparison only: A p {P_PL0['A']:.3f}, B p {P_PL0['B']:.3f}).
    Both registered placebos, and the descriptive days-only placebo D, keep the number of gate-open days per asset-year, so they
    test the windows and the days within an asset-year, not which assets and years are traded.
    {_PLB_TXT}
    Placebo D draws its days jointly per {GATE_SCOPE}-year, the gate's scope{
    "" if GATE_SCOPE == "class" else " (an asset-scope gate can still open correlated assets together, which these draws do not keep)"}.
  · Dividend basis: pitrading bars before {IEX_SEAM.date()} were adjusted to about that date, IEX bars to the end of the file;
    {_seam}. The engine undoes splits only, so the fixed per-share costs are overstated in R — by up to
    ×{_OV[_OV_MAX]:.2f} ({_OV_MAX}) at the start of {WF_FIRST_YEAR}{f" and ×{1 / min(SEAM_FACTOR[s] for s in REPAIRED):.2f} at the seam" if REPAIRED else ""},
    falling to ×1 at the end of the file. It lowers returns and tightens the cost-based tradability rule (MAX_COST_R),
    but unevenly — more for high-yield assets, early years and tight stops — so it also shaped Stage A, S* and D*: it is
    not simply conservative. With the adjustment undone and every pick and k frozen (Cell 10): holdout ΔSR
    {SENS_TABLE.loc[DIV_FIXED, 'holdout ΔSR']:+.2f}, p {SENS_TABLE.loc[DIV_FIXED, 'p one-sided']:.3f}; re-deriving the universe from the undone costs
    ({_UNI_TXT}{_NEVER_TXT}) gives ΔSR {SENS_TABLE.loc[DIV_UNI, 'holdout ΔSR']:+.2f}, p {SENS_TABLE.loc[DIV_UNI, 'p one-sided']:.3f}.
    Re-running the whole selection with the same rules on these prices picks {_RD} vs S* {_SW_DIV}:
    ΔSR {RESEL['holdout ΔSR']:+.2f}, {_RESEL_WHY} — it {'passes' if RESEL_PASS else 'fails'} Cell 9's rule (Cell 10; not pre-registered).
  · C1 is the Static ORB project's winner, chosen on the same 2008–2015 data: absolute in-sample Sharpes are upper bounds.
  · The Static engine's session filter used same-day information (full-day bar coverage). For the Static winner it dropped
    {int(dropped['trades'].sum())} of {int(dropped['trades'].sum() + kept['trades'].sum()):,} equity-ETF trades worth {dropped['sum R'].sum():+.1f} R — a real
    look-ahead, small in that project's results; this project uses ex-ante eligibility only.
  · From {IEX_SEAM.date()} the data are IEX-only: {', '.join(c for c in COVER.index if COVER.loc[c, f'eligible sessions {IEX_SEAM.date()}+'] >= 900)} carry that era;
    {(', '.join(NEVER) + ' are never tradable') if NEVER else 'every asset is tradable at some point'} (median round-trip cost > {MAX_COST_R} R of the stop).
  · The ETF data end {CAL[-1].date()}; {VOL_SYMBOL} runs to {VX.index[-1].date()} and is a reference series only — never traded,
    never a model feature, never used to select anything.
  · Buy & hold uses dividend-adjusted closes and no costs; the ORB accounts earn nothing on idle cash.

NEXT STEP
  The only clean test left is forward: log D* and S* from the first session after {CAL[-1].date()} under the frozen rule, and
  re-test once the log is long enough that the detectable gap falls below the improvement you would act on.
""")

SCORECARD
  1. Bootstrap search 2008–2015 — the best variation  →  —
       D* = C1 · ts_inc_class_h250_k2_pos: rank 1 of 128 dynamic variants by the 10th-percentile ΔSR vs S*; 0 have that percentile above 0 (best -0.279) · best variation of any family by IS q10, C2 S*_class: holdout SR 0.53
  2. Switching beats the best static window in-sample, after data-snooping  →  NO
       SPA_c p 1.000 (vs all-window ensemble 0.922) · deflated ΔSR 0.111
  3. Pre-registered holdout 2016 → 2026-09-21: D* beats S*  →  PASS
       ΔSR +0.216 (90 % CI -0.02 … +0.47) · NW p 0.023 · detectable only above 0.49
  4. Holds when the choice is re-made every year (walk-forward)  →  NO
       2016+ p 0.005 (training from 2008) / 0.257 (from 2009)
  5. The choice of D* is stable across bootstrap seeds  →  YES
       19 of 20 seeds pick D*; 6 of those also reproduce the frozen S*
  6. Execution changes (next-open fills, 2× costs, dividends undone with and without re-deriving the universe) change the holdout ver